# 🌳 Árboles de expresión, datos (CSV), métricas de error y programación genética

**Actividad de hoy (pizarrón):**

$$y = 6.87 + 11\cdot\cos\left(7.23\cdot x^{3}\right), \qquad x \in [-10.0,\ 10.0], \qquad \text{paso} = 0.001$$

1. Construir el **árbol de expresión** de la ecuación (recorridos preorden, inorden y postorden).
2. **Graficar** la ecuación.
3. Generar el **CSV** con las columnas `x, y` (de −10.000 a 10.000 de 0.001 en 0.001 → **20,001 renglones**).
4. Medir el **error** entre los datos y una ecuación con las fórmulas del pizarrón: **MSE, RMSE, NMSE y R²**.
5. **Programación genética** (clase del 20/10/2026): generar, evaluar, mientras → selección, cruza y mutación (hojas, nodos internos, switching y rama nueva), con método de reparación. Además, un **algoritmo evolutivo** que ajusta las constantes de la ecuación.

> **Para mañana:** las ecuaciones de prueba se cambian en **una sola celda** (sección *Actividad*, primer paso). Todo lo demás se recalcula solo. Si el doctor entrega un CSV, también se puede cargar.

---

### Organización (paquete modular `arbol_expresion`)

| Módulo | Responsabilidad |
|---|---|
| `config.py` | Funciones, constantes, operadores, colores y límites |
| `errores.py` | Excepción `ErrorExpresion` y mensajes que señalan dónde está el error |
| `lexico.py` | Normalización y tokenización |
| `sintaxis.py` | Menos unario, multiplicación implícita y validación sintáctica |
| `postfija.py` | Algoritmo Shunting-Yard (infija → postfija) |
| `arbol.py` | Nodo, construcción del árbol, recorridos y estadísticas |
| `evaluador.py` | Evaluación en un punto o en miles de puntos a la vez (numpy) |
| `graficos.py` | Dibujo del árbol y gráfica de la función |
| `analizador.py` | Clase `ArbolExpresion` y función `analizar()` |
| `generador.py` | Biblioteca, valores válidos, altura máxima y generación aleatoria de ecuaciones |
| `datos.py` | Datos (x, y) de una ecuación, CSV (guardar/cargar) y gráfica |
| `metricas.py` | MSE, RMSE, NMSE y R² |
| `ajuste.py` | Evaluar, comparar y buscar ecuaciones que se ajusten a los datos |
| `evolutivo.py` | **Algoritmo evolutivo** genérico: generar, evaluar, mientras (selección, cruza, mutación, reemplazo) |
| `operadores_gp.py` | Cruza monopunto, las 4 mutaciones y el método de reparación |
| `programacion_genetica.py` | **Programación genética**: el individuo es un árbol de expresión |
| `ajuste_constantes.py` | Algoritmo evolutivo de números reales que ajusta los coeficientes |
| `demostraciones.py` | Cada operador paso a paso con dibujo antes → después |
| `interactivo.py` | Modo interactivo |
| `pruebas.py` | Pruebas automáticas |
| `__init__.py` | Interfaz pública del paquete |

**Cómo usarlo en Colab:** `Entorno de ejecución → Ejecutar todas` (`Ctrl + F9`).
Las celdas `%%writefile` crean los archivos del paquete; después se importa y se usa.

# 🧩 Parte 1 · Crear el paquete

## Paso 1 · Crear la carpeta del paquete

In [ ]:
import os
os.makedirs('arbol_expresion', exist_ok=True)
print("📁 Carpeta 'arbol_expresion' lista.")

## Paso 2 · Módulo `config.py`

Operadores con su **precedencia**, **asociatividad** y **aridad**:

| Operador | Significado | Precedencia | Asociatividad | Aridad |
|---|---|---|---|---|
| `+` `-` | suma, resta | 1 | izquierda | 2 |
| `*` `/` | multiplicación, división | 2 | izquierda | 2 |
| `neg` | negativo unario (`-x`) | 3 | derecha | 1 |
| `^` | potencia | 4 | derecha | 2 |

`2^3^2 = 2^(3^2) = 512` y `-x^2 = -(x^2)`. Para agregar una función nueva basta con añadirla a `FUNCIONES`.

In [ ]:
%%writefile arbol_expresion/config.py
"""
config.py — Configuración del lenguaje de expresiones.

Para agregar una función nueva (por ejemplo, 'asin') basta con añadirla
al diccionario FUNCIONES; el resto de los módulos la reconocen solos.
"""
import math

# Funciones de un argumento: nombre -> implementación
FUNCIONES = {
    'sqrt': math.sqrt,   # raíz cuadrada
    'sin':  math.sin,    # seno (radianes)
    'cos':  math.cos,    # coseno (radianes)
    'tan':  math.tan,    # tangente (radianes)
    'log':  math.log10,  # logaritmo base 10
    'ln':   math.log,    # logaritmo natural
    'exp':  math.exp,    # e^x
    'abs':  abs,         # valor absoluto
}

# Constantes: nombre -> valor
CONSTANTES = {'pi': math.pi, 'e': math.e}

# Operadores: símbolo -> (precedencia, asociatividad, aridad)
OPERADORES = {
    '+':   (1, 'izq', 2),
    '-':   (1, 'izq', 2),
    '*':   (2, 'izq', 2),
    '/':   (2, 'izq', 2),
    'neg': (3, 'der', 1),   # menos unario: -x
    '^':   (4, 'der', 2),
}

# Altura máxima que acepta el generador de ecuaciones.
# Un árbol de altura 8 puede tener hasta 2^8 - 1 = 255 nodos; más que eso ya no se puede dibujar legible.
ALTURA_MAXIMA_LIMITE = 8

# Tipos de token que son operandos (hojas del árbol)
TIPOS_OPERANDO = ('NUM', 'VAR', 'CONST')

# Colores del dibujo: tipo de nodo -> (color, etiqueta de la leyenda)
COLORES = {
    'OP':    ('#4C72B0', 'Operador binario'),
    'OP_UN': ('#8172B3', 'Operador unario (neg)'),
    'FUNC':  ('#DD8452', 'Función'),
    'NUM':   ('#55A868', 'Número'),
    'VAR':   ('#C44E52', 'Variable'),
    'CONST': ('#2A9D8F', 'Constante'),
}

## Paso 3 · Módulo `errores.py`

Excepción propia y mensajes con una flecha `^` bajo la posición del error.

In [ ]:
%%writefile arbol_expresion/errores.py
"""
errores.py — Excepción propia y utilidades para mensajes de error.
"""


class ErrorExpresion(Exception):
    """Error de sintaxis o de evaluación en una expresión."""
    pass


def senalar(expr, pos, mensaje):
    """Arma un mensaje de error con una flecha (^) bajo la posición del problema."""
    return f"{mensaje}\n      {expr}\n      {' ' * pos}^"


def simbolo(valor):
    """Representación legible de un operador ('neg' se muestra como '-')."""
    return '-' if valor == 'neg' else valor

## Paso 4 · Módulo `lexico.py`

**Normaliza** (`**` → `^`, `·` y `×` → `*`, `√` → `sqrt`, coma decimal → punto, mayúsculas → minúsculas) y **tokeniza**.
Valida: texto vacío, signo `=`, caracteres no permitidos, números mal formados e identificadores desconocidos.

In [ ]:
%%writefile arbol_expresion/lexico.py
"""
lexico.py — Análisis léxico: normalización y tokenización.

Validaciones de esta etapa:
  - texto vacío o que no es str
  - uso del signo '='
  - caracteres no permitidos
  - números mal formados (3.4.5, '.')
  - identificadores desconocidos (raiz, sen, xy...)
"""
import re
from dataclasses import dataclass

from .config import FUNCIONES, CONSTANTES
from .errores import ErrorExpresion, senalar


@dataclass
class Token:
    tipo: str    # NUM, VAR, CONST, FUNC, OP, PAR_IZQ, PAR_DER
    valor: str
    pos: int     # posición en la cadena, para señalar errores


# Reemplazos para aceptar distintas formas de escribir lo mismo
_REEMPLAZOS = [
    ('**', '^'), ('×', '*'), ('·', '*'), ('÷', '/'), ('−', '-'),
    ('√', ' sqrt '), ('[', '('), (']', ')'), ('{', '('), ('}', ')'),
    (',', '.'),   # coma decimal -> punto decimal
]


def normalizar(expr):
    """Unifica la escritura de la expresión y valida que no esté vacía."""
    if not isinstance(expr, str):
        raise ErrorExpresion("La expresión debe ser un texto (str).")
    expr = expr.strip()
    if not expr:
        raise ErrorExpresion("La expresión está vacía.")
    if '=' in expr:
        raise ErrorExpresion("Escribe solo una expresión, sin el signo '=' "
                             "(por ejemplo: 2*x^2+5*x-3 en lugar de y=2*x^2+5*x-3).")
    for viejo, nuevo in _REEMPLAZOS:
        expr = expr.replace(viejo, nuevo)
    return re.sub(r'\s+', ' ', expr).strip().lower()


def _leer_numero(expr, i):
    texto = re.match(r'\d*\.?\d*', expr[i:]).group()
    fin = i + len(texto)
    if texto == '.':
        raise ErrorExpresion(senalar(expr, i, "Punto decimal sin dígitos."))
    if fin < len(expr) and expr[fin] == '.':
        raise ErrorExpresion(senalar(expr, fin, "Número mal formado (más de un punto decimal)."))
    return Token('NUM', texto, i), fin


def _leer_identificador(expr, i):
    palabra = re.match(r'[a-z]+', expr[i:]).group()
    if palabra in FUNCIONES:
        tipo = 'FUNC'
    elif palabra in CONSTANTES:
        tipo = 'CONST'
    elif len(palabra) == 1:
        tipo = 'VAR'
    else:
        raise ErrorExpresion(senalar(
            expr, i,
            f"Identificador desconocido '{palabra}'.\n"
            f"      Funciones válidas: {', '.join(FUNCIONES)}. Constantes: {', '.join(CONSTANTES)}.\n"
            f"      Las variables son de una letra; para multiplicar variables escribe x*y."))
    return Token(tipo, palabra, i), i + len(palabra)


_SIMBOLOS = {'+': 'OP', '-': 'OP', '*': 'OP', '/': 'OP', '^': 'OP',
             '(': 'PAR_IZQ', ')': 'PAR_DER'}


def tokenizar(expr):
    """Convierte la cadena normalizada en una lista de Token."""
    tokens, i = [], 0
    while i < len(expr):
        c = expr[i]
        if c.isspace():
            i += 1
        elif c.isdigit() or c == '.':
            tok, i = _leer_numero(expr, i)
            tokens.append(tok)
        elif c.isascii() and c.isalpha():
            tok, i = _leer_identificador(expr, i)
            tokens.append(tok)
        elif c in _SIMBOLOS:
            tokens.append(Token(_SIMBOLOS[c], c, i))
            i += 1
        else:
            raise ErrorExpresion(senalar(expr, i, f"Carácter no válido '{c}'."))
    return tokens

## Paso 5 · Módulo `sintaxis.py`

Convierte el `-` unario en `neg`, inserta multiplicaciones implícitas (`2x` → `2*x`) y valida la estructura con una máquina de estados
(operadores seguidos, paréntesis vacíos o desbalanceados, funciones sin paréntesis, etc.).

In [ ]:
%%writefile arbol_expresion/sintaxis.py
"""
sintaxis.py — Preprocesamiento y validación sintáctica.

preprocesar():
  - convierte el '-' unario en el operador 'neg' (y descarta el '+' unario)
  - inserta multiplicaciones implícitas: 2x, 3(x+1), (x+1)(x-1), 2sqrt(x)

validar_sintaxis(): máquina de estados que alterna entre
"espero un operando" y "espero un operador".
"""
from .config import TIPOS_OPERANDO
from .errores import ErrorExpresion, senalar, simbolo
from .lexico import Token


def preprocesar(tokens, expr):
    """Regresa (tokens_procesados, avisos)."""
    resultado, avisos = [], []
    for tok in tokens:
        prev = resultado[-1] if resultado else None

        # 1) Multiplicación implícita
        if prev is not None:
            termina_operando = prev.tipo in TIPOS_OPERANDO + ('PAR_DER',)
            inicia_operando = tok.tipo in TIPOS_OPERANDO + ('FUNC', 'PAR_IZQ')
            if termina_operando and inicia_operando:
                if prev.tipo == 'NUM' and tok.tipo == 'NUM':
                    raise ErrorExpresion(senalar(
                        expr, tok.pos, f"Falta un operador entre {prev.valor} y {tok.valor}."))
                resultado.append(Token('OP', '*', tok.pos))
                avisos.append(f"Multiplicación implícita insertada: {prev.valor} * {tok.valor}")
                prev = resultado[-1]

        # 2) Signo unario: al inicio, después de un operador o después de '('
        if tok.tipo == 'OP' and tok.valor in ('+', '-'):
            if prev is None or prev.tipo in ('OP', 'PAR_IZQ'):
                if tok.valor == '+':
                    continue
                tok = Token('OP', 'neg', tok.pos)

        resultado.append(tok)
    return resultado, avisos


def _error_esperando_operando(tok, anterior, siguiente, expr):
    """Valida un token cuando se esperaba un operando. Regresa True si ya llegó el operando."""
    if tok.tipo in TIPOS_OPERANDO:
        return True
    if tok.tipo == 'PAR_IZQ' or (tok.tipo == 'OP' and tok.valor == 'neg'):
        return False
    if tok.tipo == 'FUNC':
        if siguiente is None or siguiente.tipo != 'PAR_IZQ':
            raise ErrorExpresion(senalar(
                expr, tok.pos,
                f"La función '{tok.valor}' debe ir seguida de '('. Ejemplo: {tok.valor}(x)"))
        return False
    if tok.tipo == 'PAR_DER':
        msg = ("Paréntesis vacíos '()'." if anterior is not None and anterior.tipo == 'PAR_IZQ'
               else "Falta un operando antes de ')'.")
        raise ErrorExpresion(senalar(expr, tok.pos, msg))
    # Operador binario donde se esperaba un operando
    if anterior is None:
        msg = f"La expresión no puede iniciar con el operador '{tok.valor}'."
    elif anterior.tipo == 'OP':
        msg = f"Dos operadores seguidos: '{simbolo(anterior.valor)}' y '{tok.valor}'."
    else:
        msg = f"Falta un operando antes de '{tok.valor}'."
    raise ErrorExpresion(senalar(expr, tok.pos, msg))


def validar_sintaxis(tokens, expr):
    """Lanza ErrorExpresion si la secuencia de tokens no es una expresión válida."""
    if not tokens:
        raise ErrorExpresion("La expresión no contiene elementos válidos.")

    espera_operando = True
    abiertos = []                      # posiciones de '(' sin cerrar

    for i, tok in enumerate(tokens):
        anterior = tokens[i - 1] if i > 0 else None
        siguiente = tokens[i + 1] if i + 1 < len(tokens) else None

        if espera_operando:
            if tok.tipo == 'PAR_IZQ':
                abiertos.append(tok.pos)
            if _error_esperando_operando(tok, anterior, siguiente, expr):
                espera_operando = False
        elif tok.tipo == 'OP':
            espera_operando = True
        elif tok.tipo == 'PAR_DER':
            if not abiertos:
                raise ErrorExpresion(senalar(expr, tok.pos, "Paréntesis ')' sin su '(' correspondiente."))
            abiertos.pop()
        else:
            raise ErrorExpresion(senalar(expr, tok.pos, f"Falta un operador antes de '{tok.valor}'."))

    if espera_operando:
        ultimo = tokens[-1]
        raise ErrorExpresion(senalar(
            expr, ultimo.pos, f"La expresión está incompleta: termina en '{simbolo(ultimo.valor)}'."))
    if abiertos:
        n = len(abiertos)
        raise ErrorExpresion(senalar(
            expr, abiertos[-1], f"Falta{'n' if n > 1 else ''} {n} paréntesis de cierre ')'."))

## Paso 6 · Módulo `postfija.py`

Algoritmo **Shunting-Yard**: convierte la notación infija a postfija y guarda la tabla de pasos.

In [ ]:
%%writefile arbol_expresion/postfija.py
"""
postfija.py — Conversión infija -> postfija con el algoritmo Shunting-Yard (Dijkstra).

Reglas:
  - Operando           -> salida
  - Función o '('      -> pila
  - Operador binario   -> antes de apilarlo salen los de MAYOR precedencia
                          (o IGUAL si es asociativo por la izquierda)
  - Operador unario    -> se apila directamente (es prefijo)
  - ')'                -> desapilar hasta '('; si debajo hay función, también sale
  - Fin                -> vaciar la pila
"""
from .config import OPERADORES, TIPOS_OPERANDO
from .errores import ErrorExpresion


def _texto(lista):
    return ' '.join(t.valor for t in lista) if lista else '∅'


def a_postfija(tokens):
    """Regresa (postfija, pasos). 'pasos' es la tabla del algoritmo para mostrarla."""
    salida, pila, pasos = [], [], []

    def anotar(token, accion):
        pasos.append((token, accion, _texto(pila), _texto(salida)))

    for tok in tokens:
        if tok.tipo in TIPOS_OPERANDO:
            salida.append(tok)
            anotar(tok.valor, "Operando → salida")

        elif tok.tipo in ('FUNC', 'PAR_IZQ'):
            pila.append(tok)
            anotar(tok.valor, "Función → pila" if tok.tipo == 'FUNC' else "'(' → pila")

        elif tok.tipo == 'PAR_DER':
            while pila and pila[-1].tipo != 'PAR_IZQ':
                salida.append(pila.pop())
            if not pila:
                raise ErrorExpresion("Paréntesis desbalanceados.")
            pila.pop()                                    # descarta '('
            accion = "Desapilar hasta '('"
            if pila and pila[-1].tipo == 'FUNC':
                salida.append(pila.pop())
                accion += " y sacar la función"
            anotar(')', accion)

        elif tok.tipo == 'OP':
            prec, asoc, aridad = OPERADORES[tok.valor]
            sacados = 0
            if aridad == 2:
                while pila and pila[-1].tipo == 'OP':
                    prec_tope = OPERADORES[pila[-1].valor][0]
                    if prec_tope > prec or (prec_tope == prec and asoc == 'izq'):
                        salida.append(pila.pop())
                        sacados += 1
                    else:
                        break
            pila.append(tok)
            anotar(tok.valor, "Operador → pila" + (f" (antes salen {sacados})" if sacados else ""))

    while pila:
        tope = pila.pop()
        if tope.tipo == 'PAR_IZQ':
            raise ErrorExpresion("Paréntesis desbalanceados.")
        salida.append(tope)
    anotar('fin', "Vaciar la pila → salida")
    return salida, pasos


def imprimir_pasos(pasos):
    """Imprime la tabla de Shunting-Yard."""
    ancho = max(len("Pila"), *(len(p[2]) for p in pasos))
    encabezado = f"{'#':>3} │ {'Token':<6} │ {'Acción':<40} │ {'Pila':<{ancho}} │ Salida"
    print(encabezado)
    print('─' * (len(encabezado) + 20))
    for k, (tok, accion, pila, salida) in enumerate(pasos, 1):
        print(f"{k:>3} │ {tok:<6} │ {accion:<40} │ {pila:<{ancho}} │ {salida}")

## Paso 7 · Módulo `arbol.py`

| Recorrido | Orden | Notación |
|---|---|---|
| **Preorden** | raíz → izquierdo → derecho | prefija |
| **Inorden** | izquierdo → raíz → derecho | infija |
| **Postorden** | izquierdo → derecho → raíz | postfija |

Las funciones (`cos`, `sqrt`...) y el `neg` tienen un solo hijo, colocado como hijo **derecho**.

In [ ]:
%%writefile arbol_expresion/arbol.py
"""
arbol.py — Estructura del árbol de expresión, construcción y recorridos.

Construcción a partir de la postfija con una pila:
  - Operando         -> hoja, se apila
  - Operador binario -> se desapilan dos nodos (primero el DERECHO, luego el IZQUIERDO)
  - Función / neg    -> se desapila un nodo que queda como hijo DERECHO
Al final la pila debe tener exactamente un nodo: la raíz.

Recorridos:
  Preorden  = raíz, izquierdo, derecho  -> notación prefija (polaca)
  Inorden   = izquierdo, raíz, derecho  -> notación infija (sin paréntesis)
  Postorden = izquierdo, derecho, raíz  -> notación postfija (polaca inversa)
"""
from .config import TIPOS_OPERANDO, OPERADORES
from .errores import ErrorExpresion


class Nodo:
    def __init__(self, valor, tipo, izq=None, der=None):
        self.valor = valor   # texto que se muestra en el nodo
        self.tipo = tipo     # NUM, VAR, CONST, OP (binario), OP_UN (neg), FUNC
        self.izq = izq
        self.der = der

    def es_hoja(self):
        return self.izq is None and self.der is None

    def __repr__(self):
        return f"Nodo({self.valor!r})"


def construir_arbol(postfija):
    """Construye el árbol a partir de una lista de tokens en postfija y regresa la raíz."""
    pila = []
    for tok in postfija:
        if tok.tipo in TIPOS_OPERANDO:
            pila.append(Nodo(tok.valor, tok.tipo))
        elif tok.tipo == 'FUNC' or tok.valor == 'neg':
            if not pila:
                raise ErrorExpresion(f"Falta el operando de '{tok.valor}'.")
            tipo = 'FUNC' if tok.tipo == 'FUNC' else 'OP_UN'
            pila.append(Nodo(tok.valor, tipo, der=pila.pop()))
        else:
            if len(pila) < 2:
                raise ErrorExpresion(f"Faltan operandos para '{tok.valor}'.")
            der, izq = pila.pop(), pila.pop()
            pila.append(Nodo(tok.valor, 'OP', izq, der))
    if len(pila) != 1:
        raise ErrorExpresion("Expresión mal formada: sobran o faltan operandos.")
    return pila[0]


# ---------- Recorridos ----------
def preorden(n):
    return [] if n is None else [n.valor] + preorden(n.izq) + preorden(n.der)


def inorden(n):
    return [] if n is None else inorden(n.izq) + [n.valor] + inorden(n.der)


def postorden(n):
    return [] if n is None else postorden(n.izq) + postorden(n.der) + [n.valor]


def infija_con_parentesis(n):
    """Reconstruye la expresión con todos los paréntesis (conserva el significado)."""
    if n.tipo in TIPOS_OPERANDO:
        return n.valor
    if n.tipo == 'FUNC':
        return f"{n.valor}({infija_con_parentesis(n.der)})"
    if n.tipo == 'OP_UN':
        return f"(-{infija_con_parentesis(n.der)})"
    return f"({infija_con_parentesis(n.izq)} {n.valor} {infija_con_parentesis(n.der)})"


def _precedencia(n):
    """Precedencia del nodo; hojas y funciones nunca necesitan paréntesis (5)."""
    if n.tipo in ('OP', 'OP_UN'):
        return OPERADORES[n.valor][0]
    return 5


def infija_minima(n):
    """Reconstruye la expresión solo con los paréntesis necesarios (la forma más legible)."""
    if n.tipo in TIPOS_OPERANDO:
        return n.valor
    if n.tipo == 'FUNC':
        return f"{n.valor}({infija_minima(n.der)})"
    if n.tipo == 'OP_UN':
        hijo = infija_minima(n.der)
        return f"-{hijo}" if _precedencia(n.der) >= 4 else f"-({hijo})"

    p = _precedencia(n)
    izq, der = infija_minima(n.izq), infija_minima(n.der)
    pi, pd = _precedencia(n.izq), _precedencia(n.der)
    if pi < p or (pi == p and n.valor == '^'):
        izq = f"({izq})"
    if pd < p or (pd == p and n.valor != '^') or der.startswith('-'):
        der = f"({der})"
    sep = ' ' if n.valor in ('+', '-') else ''
    return f"{izq}{sep}{n.valor}{sep}{der}"


# ---------- Estadísticas ----------
def altura(n):
    return 0 if n is None else 1 + max(altura(n.izq), altura(n.der))


def contar_nodos(n):
    return 0 if n is None else 1 + contar_nodos(n.izq) + contar_nodos(n.der)


def contar_hojas(n):
    if n is None:
        return 0
    return 1 if n.es_hoja() else contar_hojas(n.izq) + contar_hojas(n.der)


def variables(n):
    if n is None:
        return set()
    propia = {n.valor} if n.tipo == 'VAR' else set()
    return propia | variables(n.izq) | variables(n.der)

## Paso 8 · Módulo `evaluador.py`

- `evaluar()` calcula **un punto** y explica cualquier error de dominio.
- `evaluar_vectorizado()` calcula **miles de puntos a la vez** con numpy (necesario para 20,001 renglones). Aplica exactamente las mismas reglas de dominio; donde no hay valor, devuelve NaN. Las pruebas verifican que ambas coincidan.

In [ ]:
%%writefile arbol_expresion/evaluador.py
"""
evaluador.py — Evaluación numérica del árbol (recorrido en postorden).

Validaciones de dominio:
  - raíz cuadrada de negativo
  - logaritmo de cero o negativo
  - tangente indefinida
  - división entre cero (incluye 0 elevado a negativo)
  - potencia de base negativa con exponente no entero (resultado complejo)
  - desbordamiento / resultados no finitos
"""
import math

import numpy as np

from .config import FUNCIONES, CONSTANTES
from .errores import ErrorExpresion


def validar_valores(valores):
    """Revisa que 'valores' sea un dict de números finitos. Regresa una copia limpia."""
    if valores is None:
        return {}
    if not isinstance(valores, dict):
        raise ErrorExpresion("'valores' debe ser un diccionario, por ejemplo {'x': 2}.")
    limpios = {}
    for k, v in valores.items():
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            raise ErrorExpresion(f"El valor de '{k}' debe ser numérico (recibido: {v!r}).")
        if not math.isfinite(v):
            raise ErrorExpresion(f"El valor de '{k}' debe ser un número finito.")
        limpios[str(k).lower()] = float(v)
    return limpios


def _aplicar_funcion(nombre, a):
    if nombre == 'sqrt' and a < 0:
        raise ErrorExpresion(f"Error de dominio: sqrt({a:g}) — raíz cuadrada de un número negativo.")
    if nombre in ('log', 'ln') and a <= 0:
        raise ErrorExpresion(f"Error de dominio: {nombre}({a:g}) — el logaritmo requiere un valor mayor que 0.")
    if nombre == 'tan' and abs(math.cos(a)) < 1e-12:
        raise ErrorExpresion(f"Error de dominio: tan({a:g}) no está definida.")
    try:
        return FUNCIONES[nombre](a)
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {nombre}({a:g}).")


def _aplicar_operador(op, a, b):
    if op == '/' and b == 0:
        raise ErrorExpresion(f"División entre cero: {a:g} / 0.")
    if op == '^':
        if a == 0 and b < 0:
            raise ErrorExpresion(f"División entre cero: 0^{b:g}.")
        if a < 0 and not float(b).is_integer():
            raise ErrorExpresion(f"Error de dominio: ({a:g})^{b:g} da un número complejo.")
    operaciones = {
        '+': lambda: a + b,
        '-': lambda: a - b,
        '*': lambda: a * b,
        '/': lambda: a / b,
        '^': lambda: a ** b,
    }
    if op not in operaciones:
        raise ErrorExpresion(f"Operador desconocido '{op}'.")
    try:
        r = operaciones[op]()
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {a:g} {op} {b:g}.")
    if not math.isfinite(r):
        raise ErrorExpresion(f"El resultado de {a:g} {op} {b:g} no es un número finito.")
    return r


def evaluar(n, valores=None):
    """Evalúa el árbol con los valores de las variables. Lanza ErrorExpresion si falla."""
    valores = valores or {}
    if n.tipo == 'NUM':
        return float(n.valor)
    if n.tipo == 'CONST':
        return CONSTANTES[n.valor]
    if n.tipo == 'VAR':
        if n.valor not in valores:
            raise ErrorExpresion(f"Falta el valor de la variable '{n.valor}'.")
        return float(valores[n.valor])
    if n.tipo == 'OP_UN':
        return -evaluar(n.der, valores)
    if n.tipo == 'FUNC':
        return _aplicar_funcion(n.valor, evaluar(n.der, valores))
    return _aplicar_operador(n.valor, evaluar(n.izq, valores), evaluar(n.der, valores))


# ─────────────── Evaluación vectorizada (miles de puntos a la vez) ───────────────
# Sigue EXACTAMENTE las mismas reglas de dominio que evaluar(); donde evaluar() lanzaría
# un error, aquí el resultado es NaN. pruebas.py verifica que ambas coincidan.

def _potencia(a, b):
    a, b = np.broadcast_arrays(a, b)
    invalida = ((a < 0) & (b != np.round(b))) | ((a == 0) & (b < 0))
    return np.where(invalida, np.nan, np.power(np.where(invalida, 1.0, a), b))


def _division(a, b):
    a, b = np.broadcast_arrays(a, b)
    return np.where(b == 0, np.nan, a / np.where(b == 0, 1.0, b))


_FUNCIONES_NP = {
    'sqrt': lambda a: np.sqrt(np.where(a >= 0, a, np.nan)),
    'sin':  np.sin,
    'cos':  np.cos,
    'tan':  lambda a: np.where(np.abs(np.cos(a)) < 1e-12, np.nan, np.tan(a)),
    'log':  lambda a: np.log10(np.where(a > 0, a, np.nan)),
    'ln':   lambda a: np.log(np.where(a > 0, a, np.nan)),
    'exp':  np.exp,
    'abs':  np.abs,
}
_OPERADORES_NP = {
    '+': np.add,
    '-': np.subtract,
    '*': np.multiply,
    '/': _division,
    '^': _potencia,
}


def _evaluar_np(n, valores):
    if n.tipo == 'NUM':
        return np.float64(n.valor)
    if n.tipo == 'CONST':
        return np.float64(CONSTANTES[n.valor])
    if n.tipo == 'VAR':
        if n.valor not in valores:
            raise ErrorExpresion(f"Falta el valor de la variable '{n.valor}'.")
        return valores[n.valor]
    if n.tipo == 'OP_UN':
        return -_evaluar_np(n.der, valores)
    if n.tipo == 'FUNC':
        return _FUNCIONES_NP[n.valor](np.asarray(_evaluar_np(n.der, valores), dtype=float))
    izq = np.asarray(_evaluar_np(n.izq, valores), dtype=float)
    der = np.asarray(_evaluar_np(n.der, valores), dtype=float)
    return _OPERADORES_NP[n.valor](izq, der)


def evaluar_vectorizado(n, valores):
    """Evalúa el árbol en muchos puntos a la vez.

    valores: {'x': arreglo de numpy}. Regresa un arreglo del mismo tamaño; los puntos fuera
    del dominio (raíz de negativo, logaritmo de ≤ 0, división entre cero, potencia compleja,
    desbordamiento...) quedan como NaN.
    """
    if not isinstance(valores, dict) or not valores:
        raise ErrorExpresion("'valores' debe ser un diccionario, por ejemplo {'x': arreglo}.")
    arreglos = {str(k): np.asarray(v, dtype=float) for k, v in valores.items()}
    formas = {a.shape for a in arreglos.values()}
    if len(formas) != 1:
        raise ErrorExpresion("Todos los arreglos de 'valores' deben tener el mismo tamaño.")
    forma = formas.pop()
    with np.errstate(all='ignore'):
        r = np.asarray(_evaluar_np(n, arreglos), dtype=float)
        r = np.array(np.broadcast_to(r, forma), dtype=float)   # una constante -> arreglo completo
    r[~np.isfinite(r)] = np.nan
    return r

## Paso 9 · Módulo `graficos.py`

Dibujo del árbol (colores por tipo de nodo, aristas I/D), comparación de árboles lado a lado con los nodos modificados resaltados en amarillo y la línea del límite de altura, y gráfica de la función.

In [ ]:
%%writefile arbol_expresion/graficos.py
"""
graficos.py — Dibujo del árbol y gráfica de la función con matplotlib.

Distribución del árbol:
  x = posición del nodo en el recorrido inorden (así nunca se enciman)
  y = profundidad del nodo
Los nodos unarios (funciones y neg) se centran encima de su único hijo.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from .config import COLORES
from .arbol import altura, infija_minima
from .errores import ErrorExpresion
from .evaluador import evaluar_vectorizado


def _calcular_posiciones(raiz):
    posiciones, nodos, contador = {}, [], [0]

    def asignar(n, prof):
        if n is None:
            return
        asignar(n.izq, prof + 1)
        posiciones[id(n)] = (contador[0], -prof)
        nodos.append(n)
        contador[0] += 1
        asignar(n.der, prof + 1)

    def centrar_unarios(n):
        if n is None:
            return
        centrar_unarios(n.izq)
        centrar_unarios(n.der)
        if n.tipo in ('FUNC', 'OP_UN'):
            posiciones[id(n)] = (posiciones[id(n.der)][0], posiciones[id(n)][1])

    asignar(raiz, 0)
    centrar_unarios(raiz)
    return posiciones, nodos


RESALTE = '#F2B705'          # borde amarillo de los nodos que cambió un operador genético


def _dibujar_en(ax, raiz, titulo, resaltar=None, leyenda=True, fuente=None, limite=None):
    """Dibuja el árbol en unos ejes de matplotlib.

    resaltar : conjunto de id(nodo) que se marcan con borde amarillo
    limite   : altura máxima; dibuja una línea roja debajo del último nivel permitido
    """
    resaltar = resaltar or set()
    posiciones, nodos = _calcular_posiciones(raiz)
    h = altura(raiz)
    n_nodos = len(nodos)
    if fuente is None:
        fuente = 12 if n_nodos <= 30 else (10 if n_nodos <= 60 else 8)

    # Aristas con la etiqueta I (izquierdo) o D (derecho)
    for n in nodos:
        x0, y0 = posiciones[id(n)]
        for hijo, lado in ((n.izq, 'I'), (n.der, 'D')):
            if hijo is not None:
                x1, y1 = posiciones[id(hijo)]
                ambos = id(n) in resaltar and id(hijo) in resaltar
                ax.plot([x0, x1], [y0, y1], color=RESALTE if ambos else '#555555',
                        lw=3 if ambos else 1.6, zorder=1)
                ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.08, lado, fontsize=8,
                        color='#888888', ha='center', va='center', zorder=2)

    # Nodos coloreados por tipo (con borde amarillo si están resaltados)
    for n in nodos:
        x, y = posiciones[id(n)]
        texto = '−' if n.tipo == 'OP_UN' else n.valor
        forma = 'circle,pad=0.45' if len(texto) <= 2 else 'round,pad=0.45'
        marcado = id(n) in resaltar
        ax.text(x, y, texto, ha='center', va='center', fontsize=fuente, fontweight='bold',
                color='white', zorder=3,
                bbox=dict(boxstyle=forma, fc=COLORES[n.tipo][0],
                          ec=RESALTE if marcado else 'black', lw=3.5 if marcado else 1.2))

    if leyenda:
        presentes = {n.tipo for n in nodos}
        elementos = [Patch(fc=color, ec='black', label=etiqueta)
                     for tipo, (color, etiqueta) in COLORES.items() if tipo in presentes]
        if resaltar:
            elementos.append(Patch(fc='white', ec=RESALTE, lw=3, label='Nodo modificado'))
        ax.legend(handles=elementos, loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)

    abajo = -(h - 1) - 0.6
    if limite is not None:
        y_lim = -(limite - 1) - 0.5
        ax.axhline(y_lim, color='#C44E52', ls='--', lw=1.4, zorder=0)
        ax.text(-0.9, y_lim - 0.12, f"límite: altura {limite}", color='#C44E52', fontsize=8, va='top')
        abajo = min(abajo, y_lim - 0.4)
    ax.set_xlim(-1, max(n_nodos, 2))
    ax.set_ylim(abajo, 0.6)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.axis('off')


def dibujar_arbol(raiz, titulo="Árbol de expresión", guardar_como=None, resaltar=None):
    """Dibuja el árbol. Si se da 'guardar_como', también lo guarda como imagen."""
    _, nodos = _calcular_posiciones(raiz)
    h = altura(raiz)
    ancho = min(max(7, 1.0 * len(nodos)), 30)          # se limita para árboles grandes
    fig, ax = plt.subplots(figsize=(ancho, max(4, 1.3 * h + 1)))
    _dibujar_en(ax, raiz, titulo, resaltar=resaltar)
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=200, bbox_inches='tight')
        print(f"💾 Árbol guardado como '{guardar_como}'")
    plt.show()


def dibujar_comparacion(arboles, titulos, resaltar=None, titulo_general=None, guardar_como=None,
                        limite=None):
    """Dibuja varios árboles lado a lado (por ejemplo antes → después de una mutación).

    arboles  : lista de raíces (Nodo)
    titulos  : un título por árbol
    resaltar : lista con un conjunto de id(nodo) a resaltar por árbol (o None)
    limite   : altura máxima (se dibuja como línea roja en todos los árboles)
    """
    if not arboles or len(arboles) != len(titulos):
        raise ErrorExpresion("Se necesita un título por cada árbol.")
    resaltar = resaltar or [None] * len(arboles)
    anchos = [max(3.0, 0.85 * len(_calcular_posiciones(a)[1])) for a in arboles]
    alto = max(3.2, 1.15 * max([altura(a) for a in arboles] + [limite or 0]) + 1.2)
    fig, ejes = plt.subplots(1, len(arboles), figsize=(min(sum(anchos) + 2.5, 26), alto),
                             gridspec_kw={'width_ratios': anchos}, squeeze=False)
    for k, (ax, raiz, titulo, marcados) in enumerate(zip(ejes[0], arboles, titulos, resaltar)):
        _dibujar_en(ax, raiz, titulo, resaltar=marcados, leyenda=(k == len(arboles) - 1), fuente=11,
                    limite=limite)
    if titulo_general:
        fig.suptitle(titulo_general, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=160, bbox_inches='tight')
        print(f"💾 Imagen guardada como '{guardar_como}'")
    plt.show()


def graficar_funcion(raiz, var, rango=(-10, 10), puntos=2000):
    """Grafica f(var) en 'puntos' valores del rango. Los puntos fuera del dominio se omiten."""
    try:
        a, b = rango
    except (TypeError, ValueError):
        print("⚠️ El rango debe ser una pareja (mínimo, máximo).")
        return
    if not all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in (a, b)) or a >= b:
        print("⚠️ Rango inválido: debe ser (mínimo, máximo) con mínimo < máximo.")
        return
    if isinstance(puntos, bool) or not isinstance(puntos, int) or not 2 <= puntos <= 1_000_000:
        print("⚠️ 'puntos' debe ser un entero entre 2 y 1,000,000.")
        return

    xs = np.linspace(a, b, puntos)
    ys = evaluar_vectorizado(raiz, {var: xs})
    if np.all(np.isnan(ys)):
        print(f"⚠️ La función no está definida en ningún punto de [{a}, {b}].")
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(xs, ys, lw=2, color='#4C72B0', label=f"f({var}) = {infija_minima(raiz)}")
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_xlabel(var)
    ax.set_ylabel(f"f({var})")
    ax.set_title("Gráfica de la función (se omiten los puntos fuera del dominio)")
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()

## Paso 10 · Módulo `analizador.py`

Clase `ArbolExpresion` y función `analizar()`, que imprime el reporte completo.

In [ ]:
%%writefile arbol_expresion/analizador.py
"""
analizador.py — Punto de entrada: une todas las etapas.

  texto --(lexico)--> tokens --(sintaxis)--> tokens válidos
        --(postfija)--> postfija --(arbol)--> árbol
        --(evaluador / graficos)--> resultados

Dos formas de uso:
  1) Orientada a objetos:  a = ArbolExpresion("sqrt(2*(x^2)+5*x-3)")
                           a.preorden(); a.evaluar(x=2); a.dibujar()
  2) Reporte completo:     analizar("sqrt(2*(x^2)+5*x-3)", valores={'x': 2})
"""
from .errores import ErrorExpresion
from .lexico import normalizar, tokenizar
from .sintaxis import preprocesar, validar_sintaxis
from .postfija import a_postfija, imprimir_pasos
from . import arbol as _arbol
from .evaluador import evaluar as _evaluar, validar_valores
from .graficos import dibujar_arbol, graficar_funcion


class ArbolExpresion:
    """Árbol de expresión construido a partir de un texto. Lanza ErrorExpresion si el texto no es válido.

    altura_maxima (opcional): si el árbol resultante es más alto, se rechaza la expresión.
    """

    def __init__(self, expresion, altura_maxima=None):
        if altura_maxima is not None and (isinstance(altura_maxima, bool)
                                          or not isinstance(altura_maxima, int) or altura_maxima < 1):
            raise ErrorExpresion("La altura máxima debe ser un entero mayor o igual a 1.")
        self.original = expresion
        self.normalizada = normalizar(expresion)
        tokens = tokenizar(self.normalizada)
        self.tokens, self.avisos = preprocesar(tokens, self.normalizada)
        validar_sintaxis(self.tokens, self.normalizada)
        self.postfija, self.pasos = a_postfija(self.tokens)
        self.raiz = _arbol.construir_arbol(self.postfija)
        h = _arbol.altura(self.raiz)
        if altura_maxima is not None and h > altura_maxima:
            raise ErrorExpresion(f"El árbol tiene altura {h} y supera la altura máxima permitida ({altura_maxima}).")

    # ----- Recorridos -----
    def preorden(self):
        return _arbol.preorden(self.raiz)

    def inorden(self):
        return _arbol.inorden(self.raiz)

    def postorden(self):
        return _arbol.postorden(self.raiz)

    def infija(self):
        return _arbol.infija_con_parentesis(self.raiz)

    def infija_minima(self):
        return _arbol.infija_minima(self.raiz)

    def altura(self):
        return _arbol.altura(self.raiz)

    # ----- Información -----
    def variables(self):
        return sorted(_arbol.variables(self.raiz))

    def estadisticas(self):
        nodos = _arbol.contar_nodos(self.raiz)
        hojas = _arbol.contar_hojas(self.raiz)
        return {'nodos': nodos, 'hojas': hojas, 'internos': nodos - hojas,
                'altura': _arbol.altura(self.raiz), 'raiz': self.raiz.valor}

    def es_consistente(self):
        """El postorden del árbol debe coincidir con la postfija de Shunting-Yard."""
        return self.postorden() == [t.valor for t in self.postfija]

    # ----- Evaluación y gráficos -----
    def evaluar(self, valores=None, **kwargs):
        """Ejemplos: a.evaluar(x=2)   o   a.evaluar({'x': 2, 'y': 1})"""
        datos = dict(valores or {})
        datos.update(kwargs)
        return _evaluar(self.raiz, validar_valores(datos))

    def mostrar_pasos(self):
        imprimir_pasos(self.pasos)

    def dibujar(self, guardar_como=None):
        dibujar_arbol(self.raiz, titulo=f"Árbol de expresión: {self.normalizada}",
                      guardar_como=guardar_como)

    def graficar(self, rango=(-10, 10), puntos=2000):
        vars_ = self.variables()
        if len(vars_) != 1:
            print(f"⚠️ Solo se grafica si hay exactamente una variable (hay {len(vars_)}).")
            return
        graficar_funcion(self.raiz, vars_[0], rango=rango, puntos=puntos)

    def __repr__(self):
        return f"ArbolExpresion({self.normalizada!r})"


def construir(expresion, altura_maxima=None):
    """Atajo: regresa un ArbolExpresion (lanza ErrorExpresion si no es válida)."""
    return ArbolExpresion(expresion, altura_maxima=altura_maxima)


def analizar(expr, valores=None, mostrar_pasos=True, graficar=True,
             graficar_f=True, rango=(-10, 10), guardar_como=None,
             altura_maxima=None, biblioteca=None, titulo="Expresión recibida", puntos=1000):
    """Imprime el reporte completo. Regresa el ArbolExpresion, o None si la expresión no es válida.

    altura_maxima: rechaza árboles más altos que este valor.
    biblioteca:    objeto Biblioteca; rechaza expresiones que usen elementos que no estén en ella.
    """
    linea = '═' * 70
    print(f"{linea}\n📥 {titulo}: {expr}\n{linea}")

    try:
        a = ArbolExpresion(expr, altura_maxima=altura_maxima)
        if biblioteca is not None:
            fuera = biblioteca.verificar(a.raiz)
            if fuera:
                raise ErrorExpresion("Usa elementos que no están en la biblioteca: " + ', '.join(fuera))
    except ErrorExpresion as err:
        print(f"❌ Expresión NO válida:\n   {err}")
        return None

    print(f"✅ Expresión válida\n   Normalizada: {a.normalizada}")
    for aviso in a.avisos:
        print(f"   ℹ️  {aviso}")

    print("\n🔤 TOKENS\n   " + '  '.join(f"[{t.valor}]" for t in a.tokens))

    if mostrar_pasos:
        print("\n🧮 CONVERSIÓN INFIJA → POSTFIJA (Shunting-Yard)\n")
        a.mostrar_pasos()

    print("\n🌳 RECORRIDOS DEL ÁRBOL")
    print(f"   Preorden  (raíz-izq-der, prefija) : {' '.join(a.preorden())}")
    print(f"   Inorden   (izq-raíz-der, infija)  : {' '.join(a.inorden())}")
    print(f"   Postorden (izq-der-raíz, postfija): {' '.join(a.postorden())}")
    print(f"   Infija con paréntesis             : {a.infija()}")
    print(f"   Infija mínima (legible)           : {a.infija_minima()}")
    print(f"\n🔎 Verificación: postorden == postfija de Shunting-Yard → "
          f"{'✔ coinciden' if a.es_consistente() else '✘ NO coinciden'}")

    e = a.estadisticas()
    vars_expr = a.variables()
    print("\n📊 ESTADÍSTICAS")
    print(f"   Nodos: {e['nodos']} | Hojas: {e['hojas']} | Nodos internos: {e['internos']} | "
          f"Altura: {e['altura']}{f' (máx. {altura_maxima})' if altura_maxima else ''} | Raíz: '{e['raiz']}'")
    print(f"   Variables: {', '.join(vars_expr) if vars_expr else '(ninguna)'}")

    try:
        limpios = validar_valores(valores)
        sobrantes = set(limpios) - set(vars_expr)
        if sobrantes:
            print(f"   ⚠️ Se ignoran valores de variables que no aparecen: {', '.join(sorted(sobrantes))}")
        if not vars_expr or limpios:
            resultado = a.evaluar(limpios)
            asignacion = ', '.join(f"{k} = {limpios[k]:g}" for k in vars_expr)
            print(f"\n🧾 EVALUACIÓN{' con ' + asignacion if asignacion else ''}: {resultado:.10g}")
    except ErrorExpresion as err:
        print(f"\n⚠️ No se pudo evaluar: {err}")

    if graficar:
        a.dibujar(guardar_como=guardar_como)
    if graficar_f and len(vars_expr) == 1:
        a.graficar(rango=rango, puntos=puntos)
    return a

## Paso 11 · Módulo `generador.py`

**Biblioteca** de elementos permitidos (no terminales: operadores y funciones; terminales: variables, constantes y **valores válidos**),
**altura máxima** del árbol y generación aleatoria con los métodos *grow*, *full* y *mixto*.

In [ ]:
%%writefile arbol_expresion/generador.py
"""
generador.py — Generación aleatoria de ecuaciones a partir de una biblioteca.

Conceptos (los mismos que en programación genética):
  - Biblioteca de NO TERMINALES (nodos internos): operadores binarios (+ - * / ^),
    el menos unario (neg) y funciones (sqrt, sin, ...).
  - Biblioteca de TERMINALES (hojas): variables, constantes y la lista de VALORES VÁLIDOS.
  - ALTURA MÁXIMA: limita el tamaño del árbol (la raíz sola tiene altura 1).

Métodos de generación:
  - 'grow'  : en cada nivel se elige al azar entre terminal y no terminal → árboles de forma variada.
  - 'full'  : se usan no terminales hasta el último nivel → árboles "llenos" de altura exacta.
  - 'mixto' : (ramped half-and-half) para cada árbol se sortea una altura entre 2 y la máxima,
              y se alterna entre 'grow' y 'full' → la mayor diversidad.

Cada árbol generado se convierte a texto, se vuelve a analizar con el mismo parser que usan
las expresiones escritas a mano y se verifica que el árbol obtenido sea idéntico al generado.
"""
import math
import random

from .config import OPERADORES, FUNCIONES, CONSTANTES, ALTURA_MAXIMA_LIMITE
from .errores import ErrorExpresion
from .arbol import Nodo, altura, contar_nodos, contar_hojas, infija_minima, variables
from .evaluador import evaluar
from .analizador import ArbolExpresion, analizar

METODOS = ('grow', 'full', 'mixto')


# ───────────────────────── Utilidades de validación ─────────────────────────
def _como_lista(nombre, valor):
    """Acepta lista, tupla o un solo elemento; rechaza tipos que no tengan sentido."""
    if valor is None:
        return []
    if isinstance(valor, (str, int, float)) and not isinstance(valor, bool):
        valor = [valor]
    if not isinstance(valor, (list, tuple, set)):
        raise ErrorExpresion(f"'{nombre}' debe ser una lista, por ejemplo ['+', '*'].")
    lista = list(valor)
    repetidos = sorted({str(v) for v in lista if lista.count(v) > 1})
    if repetidos:
        raise ErrorExpresion(f"'{nombre}' tiene elementos repetidos: {', '.join(repetidos)}.")
    return lista


def _texto_numero(v):
    """Escribe un número sin notación científica (el lexer no la acepta): 3.0 -> '3', 0.5 -> '0.5'."""
    if float(v).is_integer():
        return str(int(v))
    return f"{v:.10f}".rstrip('0').rstrip('.')


# ───────────────────────────────── Biblioteca ─────────────────────────────────
class Biblioteca:
    """Conjunto de elementos permitidos para construir ecuaciones.

    operadores : subconjunto de ['+', '-', '*', '/', '^', 'neg']
    funciones  : subconjunto de las funciones de config.FUNCIONES
    variables  : letras, por ejemplo ['x'] o ['x', 'y']
    valores    : lista de valores válidos (números ≥ 0) que pueden aparecer como hojas
    constantes : subconjunto de ['pi', 'e']
    """

    def __init__(self, operadores=('+', '-', '*', '/'), funciones=('sqrt',),
                 variables=('x',), valores=(1, 2, 3, 5), constantes=()):
        self.operadores = self._validar_operadores(_como_lista('operadores', operadores))
        self.funciones = self._validar_funciones(_como_lista('funciones', funciones))
        self.variables = self._validar_variables(_como_lista('variables', variables))
        self.valores = self._validar_valores(_como_lista('valores', valores))
        self.constantes = self._validar_constantes(_como_lista('constantes', constantes))

        # (valor, tipo de nodo, aridad)
        self.no_terminales = (
            [(op, 'OP', 2) for op in self.operadores if op != 'neg'] +
            [('neg', 'OP_UN', 1)] * ('neg' in self.operadores) +
            [(f, 'FUNC', 1) for f in self.funciones]
        )
        # (texto, tipo de nodo)
        self.terminales = (
            [(v, 'VAR') for v in self.variables] +
            [(_texto_numero(v), 'NUM') for v in self.valores] +
            [(c, 'CONST') for c in self.constantes]
        )
        if not self.terminales:
            raise ErrorExpresion("La biblioteca necesita al menos un terminal (variable, valor o constante).")
        if not self.no_terminales:
            raise ErrorExpresion("La biblioteca necesita al menos un operador o una función para formar ecuaciones.")

    # ----- validaciones -----
    @staticmethod
    def _validar_operadores(ops):
        for op in ops:
            if op not in OPERADORES:
                raise ErrorExpresion(f"Operador no válido '{op}'. Opciones: {', '.join(OPERADORES)}.")
        return ops

    @staticmethod
    def _validar_funciones(funcs):
        for f in funcs:
            if f not in FUNCIONES:
                raise ErrorExpresion(f"Función no válida '{f}'. Opciones: {', '.join(FUNCIONES)}.")
        return funcs

    @staticmethod
    def _validar_variables(vars_):
        for v in vars_:
            if not (isinstance(v, str) and len(v) == 1 and v.isascii() and v.isalpha() and v.islower()):
                raise ErrorExpresion(f"Variable no válida {v!r}: debe ser una sola letra minúscula (a-z).")
            if v in CONSTANTES:
                raise ErrorExpresion(f"'{v}' es una constante; no puede usarse como variable.")
        return vars_

    @staticmethod
    def _validar_valores(vals):
        limpios = []
        for v in vals:
            if isinstance(v, bool) or not isinstance(v, (int, float)):
                raise ErrorExpresion(f"Valor no válido {v!r}: los valores deben ser números.")
            if not math.isfinite(v):
                raise ErrorExpresion(f"Valor no válido {v}: debe ser un número finito.")
            if v < 0:
                raise ErrorExpresion(f"Valor no válido {v}: usa solo valores ≥ 0. "
                                     f"Para obtener negativos agrega el operador 'neg' a la biblioteca.")
            if v > 1e6:
                raise ErrorExpresion(f"Valor no válido {v}: el máximo permitido es 1,000,000.")
            limpios.append(float(v))
        return limpios

    @staticmethod
    def _validar_constantes(consts):
        for c in consts:
            if c not in CONSTANTES:
                raise ErrorExpresion(f"Constante no válida '{c}'. Opciones: {', '.join(CONSTANTES)}.")
        return consts

    # ----- uso -----
    def verificar(self, raiz):
        """Regresa la lista de elementos del árbol que NO pertenecen a la biblioteca (vacía si todo es válido)."""
        fuera = []

        def visitar(n):
            if n is None:
                return
            permitido = {
                'OP':    lambda: n.valor in self.operadores,
                'OP_UN': lambda: 'neg' in self.operadores,
                'FUNC':  lambda: n.valor in self.funciones,
                'VAR':   lambda: n.valor in self.variables,
                'NUM':   lambda: float(n.valor) in self.valores,
                'CONST': lambda: n.valor in self.constantes,
            }[n.tipo]()
            if not permitido:
                etiqueta = '-' if n.tipo == 'OP_UN' else n.valor
                if etiqueta not in fuera:
                    fuera.append(etiqueta)
            visitar(n.izq)
            visitar(n.der)

        visitar(raiz)
        return fuera

    def describir(self):
        print("📚 BIBLIOTECA")
        print(f"   No terminales → operadores: {', '.join(self.operadores) or '(ninguno)'}"
              f" | funciones: {', '.join(self.funciones) or '(ninguna)'}")
        print(f"   Terminales    → variables: {', '.join(self.variables) or '(ninguna)'}"
              f" | valores válidos: {', '.join(_texto_numero(v) for v in self.valores) or '(ninguno)'}"
              f" | constantes: {', '.join(self.constantes) or '(ninguna)'}")

    def __repr__(self):
        return (f"Biblioteca(operadores={self.operadores}, funciones={self.funciones}, "
                f"variables={self.variables}, valores={[_texto_numero(v) for v in self.valores]}, "
                f"constantes={self.constantes})")


# ─────────────────────────── Generador de ecuaciones ───────────────────────────
class GeneradorExpresiones:
    """Genera ecuaciones aleatorias que respetan la biblioteca y la altura máxima.

    altura_maxima       : 1 a ALTURA_MAXIMA_LIMITE (la raíz sola tiene altura 1)
    altura_minima       : descarta árboles más bajos (evita ecuaciones triviales como x + 2)
    metodo             : 'grow', 'full' o 'mixto'
    prob_terminal       : en 'grow', probabilidad de poner una hoja antes del último nivel
    semilla             : entero para obtener siempre las mismas ecuaciones (reproducible)
    exigir_variable     : la ecuación debe contener al menos una variable
    exponente_constante : en '^' el exponente es un valor pequeño de la lista (evita x^(x^x))
    puntos_validos      : dict {variable: [valores]}; la ecuación debe poder evaluarse en todos ellos
    max_intentos        : intentos antes de rendirse si no se cumplen las condiciones
    """

    def __init__(self, biblioteca, altura_maxima=4, altura_minima=1, metodo='grow', prob_terminal=0.3,
                 semilla=None, exigir_variable=True, exponente_constante=True,
                 puntos_validos=None, max_intentos=500):
        # Se valida por atributos (no con isinstance) para que siga funcionando
        # aunque el paquete se vuelva a importar en Colab después de crear la biblioteca.
        if not all(hasattr(biblioteca, a) for a in ('terminales', 'no_terminales', 'valores', 'verificar')):
            raise ErrorExpresion("'biblioteca' debe ser un objeto Biblioteca.")
        if isinstance(altura_maxima, bool) or not isinstance(altura_maxima, int):
            raise ErrorExpresion("La altura máxima debe ser un número entero.")
        if not 1 <= altura_maxima <= ALTURA_MAXIMA_LIMITE:
            raise ErrorExpresion(f"La altura máxima debe estar entre 1 y {ALTURA_MAXIMA_LIMITE}.")
        if isinstance(altura_minima, bool) or not isinstance(altura_minima, int):
            raise ErrorExpresion("La altura mínima debe ser un número entero.")
        if not 1 <= altura_minima <= altura_maxima:
            raise ErrorExpresion(f"La altura mínima debe estar entre 1 y la altura máxima ({altura_maxima}).")
        if metodo not in METODOS:
            raise ErrorExpresion(f"Método no válido '{metodo}'. Opciones: {', '.join(METODOS)}.")
        if isinstance(prob_terminal, bool) or not isinstance(prob_terminal, (int, float)) \
                or not 0 <= prob_terminal <= 1:
            raise ErrorExpresion("'prob_terminal' debe ser un número entre 0 y 1.")
        if semilla is not None and (isinstance(semilla, bool) or not isinstance(semilla, int)):
            raise ErrorExpresion("La semilla debe ser un número entero (o None).")
        if isinstance(max_intentos, bool) or not isinstance(max_intentos, int) or max_intentos < 1:
            raise ErrorExpresion("'max_intentos' debe ser un entero mayor o igual a 1.")
        if exigir_variable and not biblioteca.variables:
            raise ErrorExpresion("Se pidió exigir una variable, pero la biblioteca no tiene variables.")

        self.biblioteca = biblioteca
        self.altura_maxima = altura_maxima
        self.altura_minima = altura_minima
        self.metodo = metodo
        self.prob_terminal = float(prob_terminal)
        self.semilla = semilla
        self.exigir_variable = exigir_variable
        self.exponente_constante = exponente_constante
        self.puntos = self._validar_puntos(puntos_validos)
        self.max_intentos = max_intentos
        self._rng = random.Random(semilla)
        self._contador = 0

    def _validar_puntos(self, puntos):
        """Convierte {'x': [1, 2], 'y': [3, 4]} en [{'x': 1, 'y': 3}, {'x': 2, 'y': 4}]."""
        if puntos is None:
            return []
        if not isinstance(puntos, dict) or not puntos:
            raise ErrorExpresion("'puntos_validos' debe ser un diccionario, por ejemplo {'x': [1, 2, 3]}.")
        longitudes, normalizados = set(), {}
        for var, lista in puntos.items():
            if var not in self.biblioteca.variables:
                raise ErrorExpresion(f"La variable '{var}' de los puntos no está en la biblioteca.")
            if isinstance(lista, (int, float)) and not isinstance(lista, bool):
                lista = [lista]
            if not isinstance(lista, (list, tuple)):
                raise ErrorExpresion(f"Los puntos de '{var}' deben ser una lista de números.")
            lista = list(lista)
            normalizados[var] = lista
            if not lista:
                raise ErrorExpresion(f"La lista de puntos de '{var}' está vacía.")
            for v in lista:
                if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v):
                    raise ErrorExpresion(f"Punto no válido para '{var}': {v!r}.")
            longitudes.add(len(lista))
        if len(longitudes) > 1:
            raise ErrorExpresion("Todas las listas de 'puntos_validos' deben tener la misma longitud.")
        n = longitudes.pop()
        return [{var: float(lista[i]) for var, lista in normalizados.items()} for i in range(n)]

    # ----- construcción aleatoria del árbol -----
    def _terminal(self):
        texto, tipo = self._rng.choice(self.biblioteca.terminales)
        return Nodo(texto, tipo)

    def _exponente(self):
        """Exponente de '^': prefiere enteros 2, 3 o 4 de la lista de valores válidos
        (evita potencias triviales como x^1 o x^0 y potencias que se desbordan)."""
        utiles = [v for v in self.biblioteca.valores if v in (2, 3, 4)]
        pequenos = utiles or [v for v in self.biblioteca.valores if 0 <= v <= 4]
        if pequenos:
            return Nodo(_texto_numero(self._rng.choice(pequenos)), 'NUM')
        return self._terminal()

    def _crecer(self, profundidad, h_max, metodo):
        """Crea un subárbol cuya raíz está en 'profundidad' (la raíz del árbol es profundidad 1)."""
        if profundidad == h_max:                                   # último nivel: solo hojas
            return self._terminal()
        es_raiz = profundidad == 1
        if metodo == 'grow' and not es_raiz and self._rng.random() < self.prob_terminal:
            return self._terminal()

        valor, tipo, aridad = self._rng.choice(self.biblioteca.no_terminales)
        if aridad == 1:
            return Nodo(valor, tipo, der=self._crecer(profundidad + 1, h_max, metodo))
        izq = self._crecer(profundidad + 1, h_max, metodo)
        if valor == '^' and self.exponente_constante:
            der = self._exponente()
        else:
            der = self._crecer(profundidad + 1, h_max, metodo)
        return Nodo(valor, tipo, izq, der)

    def _arbol_aleatorio(self):
        if self.altura_maxima == 1:
            return self._terminal(), 'grow', 1
        if self.metodo == 'mixto':
            h = self._rng.randint(max(2, self.altura_minima), self.altura_maxima)
            metodo = 'full' if self._contador % 2 else 'grow'
        else:
            h, metodo = self.altura_maxima, self.metodo
        return self._crecer(1, h, metodo), metodo, h

    def _cumple_dominio(self, raiz):
        for punto in self.puntos:
            try:
                evaluar(raiz, punto)
            except ErrorExpresion:
                return False
        return True

    # ----- piezas sueltas (las usa la programación genética) -----
    def terminal_aleatorio(self):
        """Una hoja válida al azar (variable, valor válido o constante)."""
        return self._terminal()

    def exponente_aleatorio(self):
        """Un exponente válido para '^' (respeta 'exponente_constante')."""
        return self._exponente() if self.exponente_constante else self._terminal()

    def generar_rama(self, altura_maxima, metodo='grow'):
        """Subárbol aleatorio (Nodo) de altura ≤ altura_maxima. Con altura 1 es solo una hoja."""
        if isinstance(altura_maxima, bool) or not isinstance(altura_maxima, int) or altura_maxima < 1:
            raise ErrorExpresion("La altura de la rama debe ser un entero mayor o igual a 1.")
        if metodo not in ('grow', 'full'):
            raise ErrorExpresion("El método de la rama debe ser 'grow' o 'full'.")
        if altura_maxima == 1:
            return self._terminal()
        return self._crecer(1, altura_maxima, metodo)

    # ----- interfaz pública -----
    def generar(self):
        """Genera una ecuación válida y la regresa como ArbolExpresion."""
        for intento in range(1, self.max_intentos + 1):
            raiz, metodo, h = self._arbol_aleatorio()
            if altura(raiz) < self.altura_minima:
                continue
            if self.exigir_variable and not variables(raiz):
                continue
            if not self._cumple_dominio(raiz):
                continue

            texto = infija_minima(raiz)
            arbol = ArbolExpresion(texto, altura_maxima=self.altura_maxima)
            if _firma(arbol.raiz) != _firma(raiz):           # verificación de ida y vuelta
                raise ErrorExpresion(f"Error interno: el texto '{texto}' no reproduce el árbol generado.")

            self._contador += 1
            arbol.metodo, arbol.intentos, arbol.altura_objetivo = metodo, intento, h
            return arbol
        raise ErrorExpresion(
            f"No se pudo generar una ecuación que cumpla las condiciones en {self.max_intentos} intentos. "
            f"Prueba con otra altura, otra biblioteca o con menos restricciones (puntos_validos).")

    def generar_lote(self, cantidad=10, mostrar=True, evaluar_en=None):
        """Genera varias ecuaciones. 'evaluar_en' = {'x': [0, 1, 2]} agrega columnas con f(x)."""
        if isinstance(cantidad, bool) or not isinstance(cantidad, int) or not 1 <= cantidad <= 500:
            raise ErrorExpresion("La cantidad debe ser un entero entre 1 y 500.")
        puntos = self._validar_puntos(evaluar_en) if evaluar_en else []
        lote = [self.generar() for _ in range(cantidad)]
        if mostrar:
            self._imprimir_lote(lote, puntos)
        return lote

    def _imprimir_lote(self, lote, puntos):
        def f_en(a, p):
            try:
                return f"{a.evaluar(p):.4g}"
            except ErrorExpresion:
                return "indef."

        cab_puntos = [', '.join(f"{k}={v:g}" for k, v in p.items()) for p in puntos]
        ancho_ec = max(len("Ecuación"), *(len(a.infija_minima()) for a in lote))
        enc = f"{'#':>3} │ {'Ecuación':<{ancho_ec}} │ Altura │ Nodos │ Hojas │ Método"
        enc += ''.join(f" │ f({c})" for c in cab_puntos)
        print(enc)
        print('─' * len(enc))
        for i, a in enumerate(lote, 1):
            e = a.estadisticas()
            fila = (f"{i:>3} │ {a.infija_minima():<{ancho_ec}} │ {e['altura']:>6} │ {e['nodos']:>5} │ "
                    f"{e['hojas']:>5} │ {a.metodo:<6}")
            fila += ''.join(f" │ {f_en(a, p):>{len(c) + 3}}" for p, c in zip(puntos, cab_puntos))
            print(fila)

    def analizar_nueva(self, valores=None, **opciones):
        """Genera una ecuación y le aplica el análisis completo (árbol, recorridos, gráficas...)."""
        arbol = self.generar()
        print(f"🎲 Ecuación generada #{self._contador} (método {arbol.metodo}, "
              f"intento {arbol.intentos}, altura máxima {self.altura_maxima})")
        return analizar(arbol.infija_minima(), valores=valores, altura_maxima=self.altura_maxima,
                        biblioteca=self.biblioteca, titulo="Ecuación generada", **opciones)

    def describir(self):
        self.biblioteca.describir()
        print(f"⚙️  GENERADOR → altura: {self.altura_minima} a {self.altura_maxima} | método: {self.metodo} | "
              f"prob. terminal: {self.prob_terminal:g} | semilla: {self.semilla} | "
              f"exigir variable: {'sí' if self.exigir_variable else 'no'}")
        if self.puntos:
            print("   Debe poder evaluarse en: " +
                  '; '.join(', '.join(f"{k}={v:g}" for k, v in p.items()) for p in self.puntos))


def _firma(n):
    """Representación estructural del árbol para comparar dos árboles."""
    if n is None:
        return None
    valor = _texto_numero(float(n.valor)) if n.tipo == 'NUM' else n.valor
    return (valor, n.tipo, _firma(n.izq), _firma(n.der))

## Paso 12 · Módulo `datos.py`

- `generar_datos()` evalúa la ecuación de `inicio` a `fin` con el `paso` indicado. Las x se calculan como `inicio + i·paso` y se redondean al número de decimales del paso, para que no se acumule error (x = −10.000, −9.999, …, 10.000).
- `guardar_csv()` y `cargar_csv()` escriben y leen el CSV. `cargar_csv()` acepta coma, `;` o tabulador, con o sin encabezado, y coma decimal de Excel.
- `graficar_datos()` grafica el rango completo y un acercamiento.

Valida: inicio < fin, paso > 0, que el paso divida exactamente el intervalo, una sola variable, número máximo de puntos y archivos mal formados.

In [ ]:
%%writefile arbol_expresion/datos.py
"""
datos.py — Conjunto de datos (x, y) a partir de una ecuación: generación, CSV y gráfica.

Ejemplo (lo que se pidió en clase):
    y = 6.87 + 11*cos(7.23*x^3),  x de -10.0 a 10.0 con paso 0.001  →  20,001 renglones

    datos = generar_datos("6.87 + 11*cos(7.23*x^3)", inicio=-10.0, fin=10.0, paso=0.001)
    guardar_csv(datos, "datos.csv")
"""
import csv
import math
import os

import numpy as np
import matplotlib.pyplot as plt

from .errores import ErrorExpresion
from .analizador import ArbolExpresion
from .evaluador import evaluar_vectorizado
from .arbol import infija_minima

MAX_PUNTOS = 5_000_000


def _es_numero(v):
    return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) \
        and math.isfinite(float(v))


def _decimales(paso):
    """Decimales necesarios para escribir x sin errores de redondeo: 0.001 → 3, 0.25 → 2, 1 → 0."""
    texto = f"{paso:.10f}".rstrip('0')
    return len(texto.split('.')[1]) if '.' in texto else 0


class ConjuntoDatos:
    """Puntos (x, y) con la ecuación que los generó."""

    def __init__(self, x, y, ecuacion=None, variable='x', paso=None):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        if x.ndim != 1 or y.ndim != 1 or len(x) != len(y):
            raise ErrorExpresion("x y y deben ser listas del mismo tamaño.")
        if len(x) == 0:
            raise ErrorExpresion("El conjunto de datos está vacío.")
        if not (np.all(np.isfinite(x)) and np.all(np.isfinite(y))):
            raise ErrorExpresion("El conjunto de datos contiene valores no finitos (NaN o infinito).")
        self.x, self.y = x, y
        self.ecuacion = ecuacion
        self.variable = variable
        self.paso = paso

    def __len__(self):
        return len(self.x)

    def describir(self, filas=5):
        print("📋 CONJUNTO DE DATOS")
        if self.ecuacion:
            print(f"   Ecuación  : y = {self.ecuacion}")
        print(f"   Renglones : {len(self):,}")
        print(f"   {self.variable} va de {self.x.min():g} a {self.x.max():g}"
              + (f" con paso {self.paso:g}" if self.paso else ""))
        print(f"   y va de {self.y.min():.6f} a {self.y.max():.6f} | promedio ȳ = {self.y.mean():.6f}")
        dx = _decimales(self.paso) if self.paso else 6
        print(f"\n   {self.variable:>12} │ {'y':>12}")
        print(f"   {'─' * 12}─┼─{'─' * 12}")
        mostrar = list(range(min(filas, len(self))))
        if len(self) > 2 * filas:
            mostrar += [None] + list(range(len(self) - filas, len(self)))
        elif len(self) > filas:
            mostrar = list(range(len(self)))
        for i in mostrar:
            if i is None:
                print(f"   {'⋮':>12} │ {'⋮':>12}")
            else:
                print(f"   {self.x[i]:>12.{dx}f} │ {self.y[i]:>12.6f}")

    def __repr__(self):
        return f"ConjuntoDatos({len(self)} puntos, ecuacion={self.ecuacion!r})"


def generar_datos(ecuacion, inicio=-10.0, fin=10.0, paso=0.001, ruido=0.0, semilla=None,
                  omitir_invalidos=True, mostrar=True):
    """Evalúa la ecuación desde 'inicio' hasta 'fin' (ambos incluidos) con el 'paso' indicado.

    ruido            : desviación estándar de un ruido gaussiano opcional sumado a y (0 = sin ruido)
    omitir_invalidos : si la ecuación no está definida en algún punto, se omite ese renglón (True)
                       o se lanza un error (False)
    """
    # ----- validaciones -----
    for nombre, valor in (('inicio', inicio), ('fin', fin), ('paso', paso), ('ruido', ruido)):
        if not _es_numero(valor):
            raise ErrorExpresion(f"'{nombre}' debe ser un número finito (recibido: {valor!r}).")
    if inicio >= fin:
        raise ErrorExpresion(f"'inicio' ({inicio}) debe ser menor que 'fin' ({fin}).")
    if paso <= 0:
        raise ErrorExpresion("'paso' debe ser mayor que 0.")
    if paso > fin - inicio:
        raise ErrorExpresion("'paso' no puede ser mayor que el intervalo (fin - inicio).")
    if ruido < 0:
        raise ErrorExpresion("'ruido' no puede ser negativo.")
    pasos = (fin - inicio) / paso
    n = int(round(pasos)) + 1
    if abs(pasos - round(pasos)) > 1e-6:
        raise ErrorExpresion(f"El intervalo [{inicio}, {fin}] no es múltiplo exacto del paso {paso}.")
    if n > MAX_PUNTOS:
        raise ErrorExpresion(f"Se generarían {n:,} puntos; el máximo es {MAX_PUNTOS:,}. Usa un paso mayor.")

    es_arbol = hasattr(ecuacion, 'raiz') and hasattr(ecuacion, 'infija_minima')
    arbol = ecuacion if es_arbol else ArbolExpresion(ecuacion)
    vars_ = arbol.variables()
    if len(vars_) > 1:
        raise ErrorExpresion(f"La ecuación debe tener una sola variable (tiene: {', '.join(vars_)}).")
    variable = vars_[0] if vars_ else 'x'

    # x = inicio + i*paso (se calcula con índices para no acumular error) y se redondea al paso
    x = np.round(inicio + np.arange(n) * paso, _decimales(paso))
    y = evaluar_vectorizado(arbol.raiz, {variable: x})

    invalidos = np.isnan(y)
    if invalidos.any():
        if not omitir_invalidos:
            primero = x[invalidos][0]
            raise ErrorExpresion(f"La ecuación no está definida en {invalidos.sum():,} puntos "
                                 f"(el primero es {variable} = {primero:g}).")
        if mostrar:
            print(f"⚠️ Se omitieron {invalidos.sum():,} puntos donde la ecuación no está definida.")
        x, y = x[~invalidos], y[~invalidos]
        if len(x) == 0:
            raise ErrorExpresion("La ecuación no está definida en ningún punto del intervalo.")

    if ruido > 0:
        y = y + np.random.default_rng(semilla).normal(0.0, ruido, size=len(y))

    return ConjuntoDatos(x, y, ecuacion=arbol.infija_minima(), variable=variable, paso=float(paso))


def guardar_csv(datos, archivo="datos.csv", decimales_y=6, mostrar=True):
    """Guarda el conjunto en un CSV con encabezado 'x,y'. Regresa la ruta del archivo."""
    if not isinstance(archivo, str) or not archivo.lower().endswith('.csv'):
        raise ErrorExpresion("El nombre del archivo debe terminar en '.csv'.")
    if isinstance(decimales_y, bool) or not isinstance(decimales_y, int) or not 0 <= decimales_y <= 15:
        raise ErrorExpresion("'decimales_y' debe ser un entero entre 0 y 15.")
    dx = _decimales(datos.paso) if datos.paso else 10
    carpeta = os.path.dirname(archivo)
    if carpeta:
        os.makedirs(carpeta, exist_ok=True)
    np.savetxt(archivo, np.column_stack([datos.x, datos.y]), delimiter=',',
               header=f"{datos.variable},y", comments='', fmt=[f'%.{dx}f', f'%.{decimales_y}f'])
    tam = os.path.getsize(archivo)
    if mostrar:
        print(f"💾 CSV guardado: '{archivo}' ({len(datos):,} renglones + encabezado, {tam / 1024:,.1f} KB)")
    return archivo


def _detectar_separador(linea):
    for sep in (';', '\t', ','):
        if sep in linea:
            return sep
    raise ErrorExpresion("No se reconoce el separador del CSV (se acepta coma, punto y coma o tabulador).")


def _a_numero(texto, separador):
    texto = texto.strip()
    if separador != ',':
        texto = texto.replace(',', '.')        # coma decimal (Excel en español)
    return float(texto)


def cargar_csv(archivo, columna_x=0, columna_y=1, mostrar=True):
    """Lee un CSV con columnas x, y (por ejemplo el que entregue el doctor).

    Acepta separador coma, punto y coma o tabulador; con o sin encabezado; y coma decimal
    cuando el separador es ';'. Si el archivo tiene más columnas, elige cuáles usar con
    columna_x y columna_y (la primera columna es 0).
    """
    if not isinstance(archivo, str) or not os.path.isfile(archivo):
        raise ErrorExpresion(f"No se encontró el archivo '{archivo}'.")
    for nombre, valor in (('columna_x', columna_x), ('columna_y', columna_y)):
        if isinstance(valor, bool) or not isinstance(valor, int) or valor < 0:
            raise ErrorExpresion(f"'{nombre}' debe ser un entero ≥ 0.")
    if columna_x == columna_y:
        raise ErrorExpresion("'columna_x' y 'columna_y' deben ser columnas distintas.")

    with open(archivo, newline='', encoding='utf-8-sig') as f:
        lineas = [l for l in f.read().splitlines() if l.strip()]
    if not lineas:
        raise ErrorExpresion("El CSV está vacío.")
    separador = _detectar_separador(lineas[0])
    filas = list(csv.reader(lineas, delimiter=separador))
    necesarias = max(columna_x, columna_y) + 1

    # ¿La primera fila es encabezado o ya son datos?
    primera = filas[0]
    try:
        _a_numero(primera[columna_x], separador)
        _a_numero(primera[columna_y], separador)
        encabezado, datos_filas, inicio_num = None, filas, 1
    except (ValueError, IndexError):
        encabezado, datos_filas, inicio_num = primera, filas[1:], 2
        if len(encabezado) < necesarias:
            raise ErrorExpresion(f"El CSV tiene {len(encabezado)} columna(s): {encabezado}; "
                                 f"se necesitan al menos {necesarias}.")

    xs, ys = [], []
    for num, fila in enumerate(datos_filas, start=inicio_num):
        if len(fila) < necesarias:
            raise ErrorExpresion(f"Renglón {num}: se esperaban al menos {necesarias} columnas y hay {len(fila)}.")
        if encabezado is not None and len(fila) != len(encabezado):
            raise ErrorExpresion(f"Renglón {num}: tiene {len(fila)} columnas y el encabezado {len(encabezado)}.")
        try:
            xs.append(_a_numero(fila[columna_x], separador))
            ys.append(_a_numero(fila[columna_y], separador))
        except ValueError:
            raise ErrorExpresion(f"Renglón {num}: valores no numéricos {fila}.")
    if not xs:
        raise ErrorExpresion("El CSV no tiene datos después del encabezado.")
    if len(xs) < 2:
        raise ErrorExpresion("El CSV debe tener al menos 2 renglones de datos.")

    variable = 'x'
    if encabezado is not None:
        nombre = encabezado[columna_x].strip().lower()
        if len(nombre) == 1 and nombre.isascii() and nombre.isalpha() and nombre != 'e':
            variable = nombre
    pasos = np.diff(xs)
    paso = float(np.round(pasos[0], 10)) if np.allclose(pasos, pasos[0]) and pasos[0] > 0 else None
    if mostrar:
        print(f"📂 CSV cargado: '{archivo}' ({len(xs):,} renglones, separador {separador!r}, "
              f"{'con' if encabezado else 'sin'} encabezado)")
    return ConjuntoDatos(xs, ys, variable=variable, paso=paso)


def graficar_datos(datos, zoom=None, titulo=None, guardar_como=None):
    """Grafica y contra x. Con zoom=(a, b) agrega un segundo panel ampliado en ese intervalo."""
    paneles = 2 if zoom else 1
    fig, ejes = plt.subplots(paneles, 1, figsize=(11, 4.2 * paneles), squeeze=False)
    titulo = titulo or (f"y = {datos.ecuacion}" if datos.ecuacion else "Datos")

    ax = ejes[0][0]
    ax.plot(datos.x, datos.y, lw=0.6, color='#4C72B0')
    ax.set_title(f"{titulo}   ({len(datos):,} puntos, {datos.variable} ∈ [{datos.x.min():g}, {datos.x.max():g}])",
                 fontsize=11)
    if zoom:
        a, b = zoom
        if not (_es_numero(a) and _es_numero(b)) or a >= b:
            raise ErrorExpresion("'zoom' debe ser (a, b) con a < b.")
        ax.axvspan(a, b, color='#DD8452', alpha=0.15, label=f"zoom [{a:g}, {b:g}]")
        ax.legend(loc='upper right', fontsize=9)
        mascara = (datos.x >= a) & (datos.x <= b)
        ax2 = ejes[1][0]
        ax2.plot(datos.x[mascara], datos.y[mascara], lw=1.2, color='#4C72B0')
        ax2.set_title(f"Acercamiento: {datos.variable} ∈ [{a:g}, {b:g}]", fontsize=11)
    for fila in ejes:
        fila[0].set_xlabel(datos.variable)
        fila[0].set_ylabel('y')
        fila[0].grid(alpha=0.3)
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=150, bbox_inches='tight')
        print(f"💾 Gráfica guardada como '{guardar_como}'")
    plt.show()


def descargar(archivo):
    """En Google Colab descarga el archivo a tu computadora; fuera de Colab solo indica dónde está."""
    if not os.path.isfile(archivo):
        raise ErrorExpresion(f"No se encontró el archivo '{archivo}'.")
    try:
        from google.colab import files       # solo existe dentro de Colab
        files.download(archivo)
        print(f"⬇️ Descargando '{archivo}'...")
    except ImportError:
        print(f"ℹ️ No estás en Colab: el archivo está en {os.path.abspath(archivo)}")

## Paso 13 · Módulo `metricas.py`

Fórmulas del pizarrón ($y_i$ = dato real, $\hat{y}_i$ = valor que da la ecuación, $n$ = número de datos):

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2 \qquad\qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

$$\text{NMSE} = \frac{\sum |y_i-\hat{y}_i|^2}{\sum |y_i|^2} \qquad\qquad R^2 = 1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}, \qquad \bar{y}=\frac{1}{n}\sum_{i=1}^{n}y_i$$

| Medida | Ajuste perfecto | Lectura |
|---|---|---|
| MSE | 0 | promedio de los errores al cuadrado |
| RMSE | 0 | error típico, en las mismas unidades que y |
| NMSE | 0 | error relativo al tamaño de los datos (sin unidades) |
| R² | 1 | 0 = igual que predecir siempre ȳ; negativo = peor que eso |

> En la foto del pizarrón la media parece decir $\frac{1}{2}$; la fórmula correcta es $\frac{1}{n}$.

In [ ]:
%%writefile arbol_expresion/metricas.py
"""
metricas.py — Medidas de error entre los datos reales (y) y los de la ecuación (ŷ).

Fórmulas (las del pizarrón):

    MSE  = (1/n) · Σ (yᵢ − ŷᵢ)²                 error cuadrático medio
    RMSE = √[ (1/n) · Σ (yᵢ − ŷᵢ)² ]            raíz del error cuadrático medio (mismas unidades que y)
    NMSE = Σ |yᵢ − ŷᵢ|² / Σ |yᵢ|²               error cuadrático normalizado (sin unidades)
    R²   = 1 − Σ (yᵢ − ŷᵢ)² / Σ (yᵢ − ȳ)²        coeficiente de determinación
    ȳ    = (1/n) · Σ yᵢ                         media de los datos reales

Interpretación:
    MSE, RMSE, NMSE → 0 es un ajuste perfecto; mientras más pequeños, mejor.
    R² → 1 es un ajuste perfecto; 0 equivale a predecir siempre ȳ; puede ser negativo si el modelo es peor que ȳ.
"""
import math

import numpy as np

from .errores import ErrorExpresion


def _preparar(y, y_hat):
    y = np.asarray(y, dtype=float).ravel()
    y_hat = np.asarray(y_hat, dtype=float).ravel()
    if y.size == 0:
        raise ErrorExpresion("No hay datos para calcular el error.")
    if y.size != y_hat.size:
        raise ErrorExpresion(f"y y ŷ deben tener el mismo número de datos ({y.size} ≠ {y_hat.size}).")
    if not np.all(np.isfinite(y)):
        raise ErrorExpresion("Los datos reales (y) contienen valores vacíos o no finitos.")
    return y, y_hat


def media(y):
    y = np.asarray(y, dtype=float).ravel()
    if y.size == 0:
        raise ErrorExpresion("No hay datos para calcular la media.")
    return float(np.sum(y) / y.size)


def mse(y, y_hat):
    y, y_hat = _preparar(y, y_hat)
    return float(np.sum((y - y_hat) ** 2) / y.size)


def rmse(y, y_hat):
    return math.sqrt(mse(y, y_hat))


def nmse(y, y_hat):
    y, y_hat = _preparar(y, y_hat)
    denominador = np.sum(np.abs(y) ** 2)
    if denominador == 0:
        raise ErrorExpresion("NMSE indefinido: todos los valores de y son 0 (Σ|y|² = 0).")
    return float(np.sum(np.abs(y - y_hat) ** 2) / denominador)


def r2(y, y_hat):
    y, y_hat = _preparar(y, y_hat)
    y_barra = media(y)
    ss_tot = np.sum((y - y_barra) ** 2)
    if ss_tot == 0:
        raise ErrorExpresion("R² indefinido: todos los valores de y son iguales (Σ(y − ȳ)² = 0).")
    return float(1 - np.sum((y - y_hat) ** 2) / ss_tot)


def calcular_metricas(y, y_hat):
    """Regresa un dict con n, puntos inválidos, ȳ, MSE, RMSE, NMSE y R².

    Si la ecuación no está definida en algún punto (ŷ = NaN), el ajuste no es válido en todo
    el dominio: se reporta cuántos puntos fallan y los errores valen infinito (y R² = −∞).
    """
    y, y_hat = _preparar(y, y_hat)
    invalidos = int(np.sum(~np.isfinite(y_hat)))
    resultado = {'n': int(y.size), 'invalidos': invalidos, 'media_y': media(y)}
    if invalidos:
        resultado.update(MSE=math.inf, RMSE=math.inf, NMSE=math.inf, R2=-math.inf)
        return resultado

    resultado['MSE'] = mse(y, y_hat)
    resultado['RMSE'] = math.sqrt(resultado['MSE'])
    for nombre, funcion in (('NMSE', nmse), ('R2', r2)):
        try:
            resultado[nombre] = funcion(y, y_hat)
        except ErrorExpresion:
            resultado[nombre] = math.nan
    return resultado


def imprimir_metricas(m, titulo="MÉTRICAS DE ERROR"):
    """Imprime el resultado de calcular_metricas() con la fórmula y la interpretación de cada medida."""
    print(f"📏 {titulo}")
    if 'n' in m:
        print(f"   n = {m['n']:,} puntos" + (f" | ȳ = {m['media_y']:.6f}" if 'media_y' in m else ""))
    if m.get('invalidos'):
        print(f"   ⚠️ La ecuación no está definida en {m['invalidos']:,} puntos → errores infinitos.")
        return
    filas = [
        ('MSE',  "(1/n)·Σ(yᵢ − ŷᵢ)²",           "0 = perfecto"),
        ('RMSE', "√[(1/n)·Σ(yᵢ − ŷᵢ)²]",        "0 = perfecto (mismas unidades que y)"),
        ('NMSE', "Σ|yᵢ − ŷᵢ|² / Σ|yᵢ|²",         "0 = perfecto (sin unidades)"),
        ('R2',   "1 − Σ(yᵢ − ŷᵢ)² / Σ(yᵢ − ȳ)²", "1 = perfecto; < 0 es peor que predecir ȳ"),
    ]
    for clave, formula, lectura in filas:
        valor = m.get(clave, math.nan)
        texto = "indefinido" if isinstance(valor, float) and math.isnan(valor) else f"{valor:.10g}"
        nombre = 'R²' if clave == 'R2' else clave
        print(f"   {nombre:<5}= {texto:<18} {formula:<30} {lectura}")


METRICAS_APTITUD = ('MSE', 'RMSE', 'NMSE')


def error_rapido(y, y_hat, metrica='RMSE'):
    """Error para usar como APTITUD en los algoritmos evolutivos (menor = mejor).

    Es la misma fórmula que mse/rmse/nmse pero sin validaciones lentas. Regresa infinito si
    la ecuación no está definida en algún punto (NaN) o si el resultado se desborda.
    """
    if np.isnan(y_hat).any():
        return math.inf
    with np.errstate(all='ignore'):
        sse = float(np.sum((y - y_hat) ** 2))
    if not math.isfinite(sse):
        return math.inf
    if metrica == 'MSE':
        return sse / y.size
    if metrica == 'RMSE':
        return math.sqrt(sse / y.size)
    if metrica == 'NMSE':
        denominador = float(np.sum(y ** 2))
        return sse / denominador if denominador > 0 else math.inf
    raise ErrorExpresion(f"Métrica de aptitud no válida '{metrica}'. Opciones: {', '.join(METRICAS_APTITUD)}.")

## Paso 14 · Módulo `ajuste.py`

- `evaluar_modelo()` calcula las métricas de una ecuación contra los datos.
- `comparar_modelos()` ordena varias ecuaciones de mejor a peor.
- `busqueda_aleatoria()` genera ecuaciones con la biblioteca y se queda con las de menor error (regresión simbólica básica).
- `graficar_ajuste()` dibuja los datos contra la ecuación.

Si una ecuación no está definida en algún punto de los datos, se considera inválida y su error es infinito.

In [ ]:
%%writefile arbol_expresion/ajuste.py
"""
ajuste.py — Evaluar qué tan bien una ecuación (modelo) reproduce un conjunto de datos.

Esta es la base de la regresión simbólica: se proponen ecuaciones candidatas (escritas a mano
o generadas con GeneradorExpresiones), se calculan sus métricas contra los datos y se elige la mejor.
"""
import numpy as np
import matplotlib.pyplot as plt

from .errores import ErrorExpresion
from .analizador import ArbolExpresion
from .evaluador import evaluar_vectorizado
from .metricas import calcular_metricas, imprimir_metricas


def _como_arbol(modelo):
    """Acepta un texto o un ArbolExpresion (validado por atributos, no con isinstance)."""
    if hasattr(modelo, 'raiz') and hasattr(modelo, 'infija_minima'):
        return modelo
    return ArbolExpresion(modelo)


def predecir(modelo, datos):
    """Regresa ŷ: la ecuación evaluada en todas las x del conjunto (NaN donde no está definida)."""
    arbol = _como_arbol(modelo)
    vars_ = arbol.variables()
    if len(vars_) > 1:
        raise ErrorExpresion(f"El modelo debe tener una sola variable (tiene: {', '.join(vars_)}).")
    if vars_ and vars_[0] != datos.variable:
        raise ErrorExpresion(f"El modelo usa la variable '{vars_[0]}', pero los datos usan '{datos.variable}'.")
    return evaluar_vectorizado(arbol.raiz, {datos.variable: datos.x})


def evaluar_modelo(modelo, datos, mostrar=True):
    """Calcula MSE, RMSE, NMSE y R² del modelo contra los datos.
    Si el modelo no está definido en algún punto, sus métricas son infinitas (modelo inválido)."""
    arbol = _como_arbol(modelo)
    y_hat = predecir(arbol, datos)
    invalidos = int(np.isnan(y_hat).sum())
    if invalidos:
        m = {'MSE': float('inf'), 'RMSE': float('inf'), 'NMSE': float('inf'), 'R2': float('-inf')}
    else:
        m = calcular_metricas(datos.y, y_hat)
    m['invalidos'] = invalidos
    m['modelo'] = arbol.infija_minima()
    if mostrar:
        print(f"🧪 Modelo: ŷ = {m['modelo']}")
        if invalidos:
            print(f"   ⚠️ No está definido en {invalidos:,} de {len(datos):,} puntos → métricas infinitas.")
        else:
            imprimir_metricas(m, titulo="MÉTRICAS contra los datos")
    return m


def _numero(v):
    """Formato de la tabla: los residuos de redondeo (|v| < 1e-12) se muestran como 0."""
    return f"{0.0 if abs(v) < 1e-12 else v:.6g}"


def _tabla(resultados, titulo):
    ancho = max(len("Modelo ŷ"), *(len(r['modelo']) for r in resultados))
    enc = (f"{'#':>3} │ {'Modelo ŷ':<{ancho}} │ {'MSE':>12} │ {'RMSE':>12} │ "
           f"{'NMSE':>12} │ {'R²':>12}")
    print(titulo)
    print(enc)
    print('─' * len(enc))
    for i, r in enumerate(resultados, 1):
        if r['invalidos']:
            aviso = f"no está definido en {r['invalidos']:,} puntos"
            print(f"{i:>3} │ {r['modelo']:<{ancho}} │ {aviso}")
        else:
            print(f"{i:>3} │ {r['modelo']:<{ancho}} │ {_numero(r['MSE']):>12} │ {_numero(r['RMSE']):>12} │ "
                  f"{_numero(r['NMSE']):>12} │ {_numero(r['R2']):>12}")


def comparar_modelos(modelos, datos, ordenar_por='RMSE', mostrar=True):
    """Evalúa varios modelos y los muestra ordenados del mejor al peor."""
    if ordenar_por not in ('MSE', 'RMSE', 'NMSE', 'R2'):
        raise ErrorExpresion("'ordenar_por' debe ser 'MSE', 'RMSE', 'NMSE' o 'R2'.")
    if not isinstance(modelos, (list, tuple)) or not modelos:
        raise ErrorExpresion("'modelos' debe ser una lista con al menos una ecuación.")
    resultados = []
    for modelo in modelos:
        try:
            resultados.append(evaluar_modelo(modelo, datos, mostrar=False))
        except ErrorExpresion as err:
            if mostrar:
                print(f"❌ Modelo descartado {modelo!r}: {str(err).splitlines()[0]}")
    if not resultados:
        raise ErrorExpresion("Ningún modelo es válido.")
    signo = -1 if ordenar_por == 'R2' else 1
    resultados.sort(key=lambda r: signo * r[ordenar_por])
    if mostrar:
        _tabla(resultados, f"🏁 COMPARACIÓN DE MODELOS (ordenados por {ordenar_por}, mejor primero)")
    return resultados


def busqueda_aleatoria(generador, datos, candidatos=300, top=5, ordenar_por='RMSE'):
    """Genera 'candidatos' ecuaciones con el generador y regresa las 'top' mejores según la métrica.
    Es la forma más simple de regresión simbólica (la programación genética la mejora con evolución)."""
    if not hasattr(generador, 'generar'):
        raise ErrorExpresion("'generador' debe ser un GeneradorExpresiones.")
    for nombre, valor, maximo in (('candidatos', candidatos, 100_000), ('top', top, 100)):
        if isinstance(valor, bool) or not isinstance(valor, int) or not 1 <= valor <= maximo:
            raise ErrorExpresion(f"'{nombre}' debe ser un entero entre 1 y {maximo}.")
    if ordenar_por not in ('MSE', 'RMSE', 'NMSE', 'R2'):
        raise ErrorExpresion("'ordenar_por' debe ser 'MSE', 'RMSE', 'NMSE' o 'R2'.")

    vistos, resultados, invalidos = set(), [], 0
    for _ in range(candidatos):
        arbol = generador.generar()
        texto = arbol.infija_minima()
        if texto in vistos:
            continue
        vistos.add(texto)
        r = evaluar_modelo(arbol, datos, mostrar=False)
        if r['invalidos']:
            invalidos += 1
            continue
        r['arbol'] = arbol
        resultados.append(r)
    if not resultados:
        raise ErrorExpresion("Ninguna ecuación generada está definida en todos los puntos de los datos.")
    signo = -1 if ordenar_por == 'R2' else 1
    resultados.sort(key=lambda r: signo * r[ordenar_por])
    print(f"🎲 Búsqueda aleatoria: {candidatos} intentos → {len(vistos)} ecuaciones distintas, "
          f"{invalidos} no definidas en todo el intervalo, {len(resultados)} evaluadas.\n")
    _tabla(resultados[:top], f"🏆 LAS {min(top, len(resultados))} MEJORES (ordenadas por {ordenar_por})")
    return resultados[:top]


def graficar_ajuste(datos, modelo, zoom=None, guardar_como=None):
    """Compara los datos reales con la predicción del modelo (y el error en el acercamiento)."""
    arbol = _como_arbol(modelo)
    y_hat = predecir(arbol, datos)
    paneles = 2 if zoom else 1
    fig, ejes = plt.subplots(paneles, 1, figsize=(11, 4.2 * paneles), squeeze=False)

    def dibujar(ax, mascara, titulo):
        ax.plot(datos.x[mascara], datos.y[mascara], lw=1.6, color='#4C72B0', label='y real (datos)')
        ax.plot(datos.x[mascara], y_hat[mascara], lw=1.0, color='#DD8452', ls='--',
                label=f"ŷ = {arbol.infija_minima()}")
        ax.set_title(titulo, fontsize=11)
        ax.set_xlabel(datos.variable)
        ax.set_ylabel('y')
        ax.grid(alpha=0.3)
        ax.legend(loc='upper right', fontsize=9)

    todo = np.ones(len(datos), dtype=bool)
    dibujar(ejes[0][0], todo, f"Datos vs. modelo ({len(datos):,} puntos)")
    if zoom:
        a, b = zoom
        if a >= b:
            raise ErrorExpresion("'zoom' debe ser (a, b) con a < b.")
        dibujar(ejes[1][0], (datos.x >= a) & (datos.x <= b),
                f"Acercamiento: {datos.variable} ∈ [{a:g}, {b:g}]")
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=150, bbox_inches='tight')
        print(f"💾 Gráfica guardada como '{guardar_como}'")
    plt.show()

## Paso 15 · Módulo `evolutivo.py`

Esqueleto general de un **algoritmo evolutivo**; sirve para cualquier representación de individuo:

```
1. GENERAR la población inicial
2. EVALUAR cada individuo (aptitud = error; menor es mejor)
3. MIENTRAS no se cumpla el criterio de paro:
      → SELECCIÓN de padres (torneo o ruleta)
      → CRUZA                (probabilidad prob_cruza)
      → MUTACIÓN             (probabilidad prob_mutacion)
      → REPARACIÓN
      → REEMPLAZO (con elitismo: los mejores pasan sin cambios)
      → EVALUAR
```

Criterios de paro: número máximo de generaciones, error objetivo alcanzado o `paciencia` generaciones sin mejorar.

In [ ]:
%%writefile arbol_expresion/evolutivo.py
"""
evolutivo.py — Esqueleto general de un ALGORITMO EVOLUTIVO.

    1. GENERAR   la población inicial (individuos aleatorios)
    2. EVALUAR   la aptitud de cada individuo (aquí: el error contra los datos; menor = mejor)
    3. MIENTRAS  no se cumpla el criterio de paro:
         → SELECCIÓN  de padres (torneo o ruleta)
         → CRUZA      de los padres para formar hijos           (con probabilidad prob_cruza)
         → MUTACIÓN   de los hijos                              (con probabilidad prob_mutacion)
         → REPARACIÓN de los hijos que rompan alguna regla
         → REEMPLAZO  la nueva generación sustituye a la anterior; los mejores (élite) pasan directo
         → EVALUAR    la nueva población

Criterios de paro: número máximo de generaciones, alcanzar un error objetivo, o pasar
'paciencia' generaciones sin mejorar.

Esta clase no sabe qué es un individuo. Las subclases definen la representación:
  - ProgramacionGenetica : el individuo es un ÁRBOL DE EXPRESIÓN (busca la forma de la ecuación)
  - AjusteConstantes     : el individuo es un VECTOR DE NÚMEROS REALES (ajusta los coeficientes)
"""
import copy
import math
import random
import statistics
import time

import matplotlib.pyplot as plt

from .errores import ErrorExpresion

SELECCIONES = ('torneo', 'ruleta')


def _entero(nombre, valor, minimo, maximo):
    if isinstance(valor, bool) or not isinstance(valor, int) or not minimo <= valor <= maximo:
        raise ErrorExpresion(f"'{nombre}' debe ser un entero entre {minimo:,} y {maximo:,} (recibido: {valor!r}).")


def _probabilidad(nombre, valor):
    if isinstance(valor, bool) or not isinstance(valor, (int, float)) or not 0 <= valor <= 1:
        raise ErrorExpresion(f"'{nombre}' debe ser un número entre 0 y 1 (recibido: {valor!r}).")


def _formato(v):
    if v == math.inf:
        return "inválido"
    return f"{0.0 if abs(v) < 1e-12 else v:.6g}"


class ResultadoEvolutivo:
    """Lo que regresa ejecutar(): el mejor individuo, su aptitud y el historial de cada generación."""

    def __init__(self, algoritmo, mejor, aptitud, historial, motivo, segundos):
        self.algoritmo = algoritmo
        self.mejor = mejor
        self.aptitud = aptitud
        self.historial = historial
        self.motivo = motivo
        self.segundos = segundos
        self.generaciones = historial[-1]['generacion']
        self.estadisticas = dict(algoritmo.estadisticas)

    def graficar_convergencia(self, guardar_como=None):
        """Mejor aptitud y mediana de la población por generación (escala logarítmica si conviene).

        Se usa la mediana y no el promedio: unos cuantos individuos con errores enormes
        (por ejemplo 10^60) volverían inútil el promedio.
        """
        gens = [h['generacion'] for h in self.historial]
        mejores = [h['mejor'] for h in self.historial]
        medianas = [h['mediana'] for h in self.historial]
        fig, ax = plt.subplots(figsize=(9, 4.2))
        ax.plot(gens, medianas, lw=1.5, color='#DD8452', label='Mediana de la población (válidos)')
        ax.plot(gens, mejores, lw=2.2, color='#4C72B0', marker='o', ms=3, label='Mejor individuo')
        valores = [v for v in mejores + medianas if math.isfinite(v)]
        positivos = [v for v in valores if v > 0]
        if positivos and max(positivos) / min(positivos) > 50:
            if len(positivos) == len(valores):
                ax.set_yscale('log')
            else:                                   # el mejor llegó a 0: escala log con tramo lineal cerca de 0
                ax.set_yscale('symlog', linthresh=min(positivos))
        ax.set_xlabel('Generación')
        ax.set_ylabel(f"Aptitud ({self.algoritmo.nombre_aptitud}, menor = mejor)")
        ax.set_title(f"Convergencia — {self.algoritmo.NOMBRE}")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        if guardar_como:
            plt.savefig(guardar_como, dpi=150, bbox_inches='tight')
            print(f"💾 Gráfica guardada como '{guardar_como}'")
        plt.show()

    def __repr__(self):
        return (f"ResultadoEvolutivo(aptitud={_formato(self.aptitud)}, generaciones={self.generaciones}, "
                f"motivo={self.motivo!r})")


class AlgoritmoEvolutivo:
    """Ciclo evolutivo genérico. Las subclases implementan los métodos marcados como 'a implementar'.

    tam_poblacion : número de individuos (en clase: 100)
    generaciones  : número máximo de generaciones (criterio de paro principal)
    prob_cruza    : probabilidad de cruzar a dos padres (si no, los hijos son copias)
    prob_mutacion : probabilidad de mutar a cada hijo
    elitismo      : cuántos de los mejores pasan sin cambios a la siguiente generación
    seleccion     : 'torneo' (gana el mejor de 'tam_torneo' elegidos al azar) o 'ruleta'
                    (probabilidad proporcional a 1 / (1 + error))
    objetivo      : se detiene si la mejor aptitud llega a este valor o menos (None = no aplica)
    paciencia     : se detiene si pasan estas generaciones sin mejora (None = no aplica)
    semilla       : entero para repetir exactamente la misma ejecución (None = distinta cada vez)
    mostrar_cada  : imprime una fila de la tabla cada cuántas generaciones (0 = no imprimir)
    """
    NOMBRE = "Algoritmo evolutivo"
    nombre_aptitud = "aptitud"

    def __init__(self, tam_poblacion=100, generaciones=50, prob_cruza=0.9, prob_mutacion=0.3,
                 elitismo=2, seleccion='torneo', tam_torneo=3, objetivo=None, paciencia=None,
                 semilla=None, mostrar_cada=5):
        _entero('tam_poblacion', tam_poblacion, 2, 100_000)
        _entero('generaciones', generaciones, 0, 100_000)
        _probabilidad('prob_cruza', prob_cruza)
        _probabilidad('prob_mutacion', prob_mutacion)
        _entero('elitismo', elitismo, 0, tam_poblacion - 1)
        if seleccion not in SELECCIONES:
            raise ErrorExpresion(f"Selección no válida '{seleccion}'. Opciones: {', '.join(SELECCIONES)}.")
        _entero('tam_torneo', tam_torneo, 2, tam_poblacion)
        if objetivo is not None and (isinstance(objetivo, bool) or not isinstance(objetivo, (int, float))
                                     or not objetivo >= 0):
            raise ErrorExpresion("'objetivo' debe ser un número ≥ 0 (o None).")
        if paciencia is not None:
            _entero('paciencia', paciencia, 1, 100_000)
        if semilla is not None:
            _entero('semilla', semilla, -2**63, 2**63)
        _entero('mostrar_cada', mostrar_cada, 0, 100_000)

        self.tam_poblacion = tam_poblacion
        self.generaciones = generaciones
        self.prob_cruza = float(prob_cruza)
        self.prob_mutacion = float(prob_mutacion)
        self.elitismo = elitismo
        self.seleccion = seleccion
        self.tam_torneo = tam_torneo
        self.objetivo = objetivo
        self.paciencia = paciencia
        self.semilla = semilla
        self.mostrar_cada = mostrar_cada
        self.rng = random.Random(semilla)
        self._cache = {}
        self.estadisticas = {}

    # ─────────────── a implementar en cada subclase ───────────────
    def individuo_aleatorio(self):
        raise NotImplementedError

    def aptitud(self, individuo):
        """Error del individuo (menor = mejor; math.inf = inválido)."""
        raise NotImplementedError

    def cruzar(self, padre1, padre2):
        raise NotImplementedError

    def mutar(self, individuo):
        raise NotImplementedError

    def reparar(self, individuo):
        return individuo

    def clave(self, individuo):
        """Identificador del individuo (para no evaluar dos veces lo mismo y medir diversidad)."""
        return repr(individuo)

    def tamano(self, individuo):
        """Desempate: con la misma aptitud gana el más pequeño (evita árboles innecesariamente grandes)."""
        return 0

    def describir(self, individuo):
        return repr(individuo)

    def copiar(self, individuo):
        return copy.deepcopy(individuo)

    # ─────────────── ciclo evolutivo ───────────────
    def contar(self, evento):
        self.estadisticas[evento] = self.estadisticas.get(evento, 0) + 1

    def evaluar(self, individuo):
        k = self.clave(individuo)
        if k not in self._cache:
            self._cache[k] = self.aptitud(individuo)
            self.contar('evaluaciones')
        return self._cache[k]

    def _orden(self, aptitudes, tamanos):
        return sorted(range(len(aptitudes)), key=lambda i: (aptitudes[i], tamanos[i]))

    def seleccionar(self, aptitudes, tamanos):
        """Regresa el índice del individuo elegido como padre."""
        n = len(aptitudes)
        if self.seleccion == 'torneo':
            participantes = self.rng.sample(range(n), self.tam_torneo)
            return min(participantes, key=lambda i: (aptitudes[i], tamanos[i]))
        pesos = [0.0 if not math.isfinite(a) else 1.0 / (1.0 + a) for a in aptitudes]
        if sum(pesos) == 0:
            return self.rng.randrange(n)
        return self.rng.choices(range(n), weights=pesos)[0]

    def _registro(self, generacion, poblacion, aptitudes, inicio):
        validas = [a for a in aptitudes if math.isfinite(a)]
        i_mejor = self._orden(aptitudes, [self.tamano(p) for p in poblacion])[0]
        return {
            'generacion': generacion,
            'mejor': aptitudes[i_mejor],
            'promedio': statistics.fmean(validas) if validas else math.inf,
            'mediana': statistics.median(validas) if validas else math.inf,
            'invalidos': len(aptitudes) - len(validas),
            'distintos': len({self.clave(p) for p in poblacion}),
            'descripcion': self.describir(poblacion[i_mejor]),
            'segundos': time.perf_counter() - inicio,
        }

    def _imprimir_encabezado(self):
        print(f"{'Gen':>4} │ {'Mejor':>11} │ {'Mediana':>11} │ {'Inválidos':>9} │ {'Distintos':>9} │ Mejor individuo")
        print('─' * 110)

    def _imprimir_fila(self, r):
        texto = r['descripcion'] if len(r['descripcion']) <= 60 else r['descripcion'][:57] + '...'
        print(f"{r['generacion']:>4} │ {_formato(r['mejor']):>11} │ {_formato(r['mediana']):>11} │ "
              f"{r['invalidos']:>9} │ {r['distintos']:>9} │ {texto}")

    def poblacion_inicial(self):
        return [self.individuo_aleatorio() for _ in range(self.tam_poblacion)]

    def ejecutar(self, mostrar=True):
        """Corre el algoritmo completo y regresa un ResultadoEvolutivo."""
        inicio = time.perf_counter()
        self._cache = {}
        self.estadisticas = {}
        imprimir = mostrar and self.mostrar_cada > 0

        # 1. GENERAR la población inicial
        poblacion = self.poblacion_inicial()
        # 2. EVALUAR
        aptitudes = [self.evaluar(p) for p in poblacion]
        historial = [self._registro(0, poblacion, aptitudes, inicio)]
        if imprimir:
            print(f"🧬 {self.NOMBRE}: población {self.tam_poblacion}, hasta {self.generaciones} generaciones, "
                  f"selección por {self.seleccion}, cruza {self.prob_cruza:.0%}, mutación {self.prob_mutacion:.0%}, "
                  f"élite {self.elitismo}\n")
            self._imprimir_encabezado()
            self._imprimir_fila(historial[-1])

        mejor_global, sin_mejora, motivo = historial[-1]['mejor'], 0, None
        generacion = 0
        # 3. MIENTRAS no se cumpla el criterio de paro
        while True:
            if self.objetivo is not None and mejor_global <= self.objetivo:
                motivo = f"se alcanzó el error objetivo (≤ {self.objetivo:g})"
                break
            if generacion >= self.generaciones:
                motivo = f"se completaron las {self.generaciones} generaciones"
                break
            if self.paciencia is not None and sin_mejora >= self.paciencia:
                motivo = f"{self.paciencia} generaciones sin mejorar"
                break

            tamanos = [self.tamano(p) for p in poblacion]
            orden = self._orden(aptitudes, tamanos)
            # Elitismo: los mejores pasan sin cambios
            nueva = [self.copiar(poblacion[i]) for i in orden[:self.elitismo]]
            while len(nueva) < self.tam_poblacion:
                # → SELECCIÓN
                padre1 = poblacion[self.seleccionar(aptitudes, tamanos)]
                padre2 = poblacion[self.seleccionar(aptitudes, tamanos)]
                # → CRUZA
                if self.rng.random() < self.prob_cruza:
                    hijos = self.cruzar(padre1, padre2)
                    self.contar('cruzas')
                else:
                    hijos = (self.copiar(padre1), self.copiar(padre2))
                    self.contar('copias')
                for hijo in hijos:
                    # → MUTACIÓN (un solo tipo por mutación)
                    if self.rng.random() < self.prob_mutacion:
                        hijo = self.mutar(hijo)
                    # → REPARACIÓN
                    hijo = self.reparar(hijo)
                    if len(nueva) < self.tam_poblacion:
                        nueva.append(hijo)
            # → REEMPLAZO y EVALUACIÓN
            poblacion = nueva
            aptitudes = [self.evaluar(p) for p in poblacion]
            generacion += 1

            registro = self._registro(generacion, poblacion, aptitudes, inicio)
            historial.append(registro)
            if registro['mejor'] < mejor_global - 1e-15:
                mejor_global, sin_mejora = registro['mejor'], 0
            else:
                sin_mejora += 1
            if imprimir and (generacion % self.mostrar_cada == 0 or generacion == self.generaciones):
                self._imprimir_fila(registro)

        i_mejor = self._orden(aptitudes, [self.tamano(p) for p in poblacion])[0]
        resultado = ResultadoEvolutivo(self, poblacion[i_mejor], aptitudes[i_mejor], historial,
                                       motivo, time.perf_counter() - inicio)
        if imprimir:
            if historial[-1]['generacion'] % self.mostrar_cada != 0 and \
                    historial[-1]['generacion'] != self.generaciones:
                self._imprimir_fila(historial[-1])
            print(f"\n🏁 Terminó porque {motivo} (generación {resultado.generaciones}, "
                  f"{resultado.segundos:.1f} s, {self.estadisticas.get('evaluaciones', 0):,} evaluaciones).")
        return resultado

## Paso 16 · Módulo `operadores_gp.py`

Operadores de la programación genética, tal como en el cuaderno:

| Apunte | Implementación |
|---|---|
| **Cruza monopunto**: "agarras cualquier parte del árbol (de cualquier altura) aleatoriamente e intercambias las conexiones" | se elige un nodo al azar en cada padre y se intercambian sus subárboles |
| **1. Mutación de hojas**: "aleatoriamente seleccionar un nodo hoja (y cambiarlo) por un valor válido" (3.0 → 7.2, π, x) | una hoja al azar cambia por otro terminal de la biblioteca |
| **2. Nodos internos**: "sustituir el valor de un nodo interno por otro operador válido" | mismo número de hijos: `+ − * / ^` entre sí; `sqrt cos sin…` entre sí |
| **3. Switching**: "intercambiar valores de los lados, izquierda ↔ derecha" | se intercambian los hijos izquierdo y derecho de un nodo |
| **4. Rama nueva**: "agarro y genero algo nuevo cuidando la altura del árbol" | un subárbol se reemplaza por una rama aleatoria que no rebasa la altura máxima |
| "Se utilizan aleatorio los métodos" / "no mezclar los métodos de mutación" | cada mutación sortea **un solo** tipo |
| **Método de reparación**: "cuando excede el límite del árbol que hemos establecido" | lo que queda en el último nivel con hijos se poda y se cambia por una hoja válida |

Todos los operadores trabajan sobre **copias**; `validar_estructura()` revisa las reglas de los árboles de expresión.

In [ ]:
%%writefile arbol_expresion/operadores_gp.py
"""
operadores_gp.py — Operadores genéticos sobre árboles de expresión (programación genética).

Todos los operadores trabajan sobre COPIAS: nunca modifican el árbol original.
Cada uno regresa (árbol_nuevo, info), donde 'info' describe qué cambió y qué nodos
resaltar al dibujarlo.

CRUZA MONOPUNTO (apunte: "agarras cualquier parte del árbol, de cualquier altura,
aleatoriamente, e intercambias las conexiones"):
    se elige un nodo al azar en cada padre y se intercambian los subárboles que cuelgan de ellos.

MUTACIÓN (4 tipos). Apunte: "se utilizan aleatorio los métodos" y "no mezclar los métodos
de mutación porque cambia el sentido de la ecuación" → en cada mutación se aplica UN solo tipo,
elegido al azar.
    1. 'hoja'      : una hoja al azar se cambia por otro valor válido (3.0 → 7.2, π, x).
    2. 'interno'   : el operador de un nodo interno se cambia por otro válido con el mismo
                     número de hijos (+ ↔ − ↔ * ↔ /,  cos ↔ sin ↔ sqrt...).
    3. 'switching' : se intercambian los hijos izquierdo y derecho de un nodo (izq ↔ der).
    4. 'rama'      : un subárbol se reemplaza por una rama nueva generada al azar,
                     "cuidando la altura del árbol" (nunca rebasa la altura máxima).

REPARACIÓN (apunte: "método de reparación = cuando excede el límite del árbol que hemos
establecido"; "puede haber método de poda o de reparación"):
    los nodos que quedan en el último nivel permitido y todavía tienen hijos se reemplazan
    por una hoja válida, de modo que el árbol vuelve a respetar la altura máxima.

REGLAS DE LOS ÁRBOLES DE EXPRESIÓN (se verifican con validar_estructura):
    - las hojas son terminales (número, variable o constante) y no tienen hijos;
    - los operadores binarios tienen exactamente 2 hijos;
    - las funciones y el '−' unario tienen exactamente 1 hijo (el derecho);
    - si el generador usa exponente constante, el exponente de '^' debe ser un número;
    - la altura no rebasa la altura máxima y todos los elementos están en la biblioteca.
"""
from .config import TIPOS_OPERANDO, OPERADORES
from .errores import ErrorExpresion
from .arbol import Nodo, altura
from .generador import _texto_numero

TIPOS_MUTACION = ('hoja', 'interno', 'switching', 'rama')
NOMBRES_MUTACION = {
    'hoja': 'Mutación de hojas',
    'interno': 'Mutación de nodos internos',
    'switching': 'Switching (intercambio izquierda ↔ derecha)',
    'rama': 'Generación de una rama nueva',
}
CONMUTATIVOS = ('+', '*')


# ───────────────────────────── Utilidades de árboles ─────────────────────────────
def copiar(n):
    """Copia profunda de un árbol."""
    if n is None:
        return None
    return Nodo(n.valor, n.tipo, copiar(n.izq), copiar(n.der))


def listar_nodos(raiz):
    """Lista en preorden de tuplas (nodo, padre, lado, profundidad). La raíz tiene profundidad 1."""
    resultado = []

    def visitar(n, padre, lado, prof):
        if n is None:
            return
        resultado.append((n, padre, lado, prof))
        visitar(n.izq, n, 'izq', prof + 1)
        visitar(n.der, n, 'der', prof + 1)

    visitar(raiz, None, None, 1)
    return resultado


def reemplazar(raiz, padre, lado, nuevo):
    """Pone 'nuevo' en el lugar del hijo 'lado' de 'padre'. Si padre es None, 'nuevo' es la raíz."""
    if padre is None:
        return nuevo
    setattr(padre, lado, nuevo)
    return raiz


def es_exponente(padre, lado):
    return padre is not None and padre.valor == '^' and lado == 'der'


def clave(n):
    """Texto en notación prefija que identifica la estructura del árbol (para caché y diversidad)."""
    if n.izq is None and n.der is None:
        return n.valor
    if n.izq is None:                                  # función o '−' unario
        return f"({n.valor} {clave(n.der)})"
    return f"({n.valor} {clave(n.izq)} {clave(n.der)})"


def _texto_nodo(n):
    return '−' if n.tipo == 'OP_UN' else n.valor


def _ids(n):
    return {id(m) for m, *_ in listar_nodos(n)}


def validar_estructura(raiz, biblioteca=None, altura_maxima=None, exponente_constante=False):
    """Regresa la lista de reglas que el árbol NO cumple (vacía si es un árbol de expresión válido)."""
    problemas = []
    for n, padre, lado, _ in listar_nodos(raiz):
        if n.tipo in TIPOS_OPERANDO:
            if n.izq is not None or n.der is not None:
                problemas.append(f"la hoja '{n.valor}' tiene hijos")
        elif n.tipo == 'OP':
            if n.valor not in OPERADORES or OPERADORES[n.valor][2] != 2:
                problemas.append(f"'{n.valor}' no es un operador binario")
            if n.izq is None or n.der is None:
                problemas.append(f"el operador '{n.valor}' no tiene 2 hijos")
        elif n.tipo in ('FUNC', 'OP_UN'):
            if n.izq is not None or n.der is None:
                problemas.append(f"'{_texto_nodo(n)}' debe tener exactamente 1 hijo (el derecho)")
        else:
            problemas.append(f"tipo de nodo desconocido '{n.tipo}'")
        if exponente_constante and es_exponente(padre, lado) and n.tipo != 'NUM':
            problemas.append(f"el exponente de '^' debe ser un número (es '{_texto_nodo(n)}')")
    if altura_maxima is not None and altura(raiz) > altura_maxima:
        problemas.append(f"la altura {altura(raiz)} rebasa la altura máxima {altura_maxima}")
    if biblioteca is not None:
        fuera = biblioteca.verificar(raiz)
        if fuera:
            problemas.append("usa elementos fuera de la biblioteca: " + ', '.join(fuera))
    return problemas


# ───────────────────────────── Reparación (control de altura) ─────────────────────────────
def reparar(raiz, generador, altura_maxima):
    """Método de reparación / poda.

    - Todo nodo en el último nivel permitido que todavía tenga hijos se reemplaza por una hoja válida.
    - Si el generador usa exponente constante, un exponente de '^' que no sea número se reemplaza
      por un exponente válido.
    Regresa (árbol_reparado, info).
    """
    nuevo = copiar(raiz)
    podados, exponentes = [], []
    # Recorrido con pila (raíz → hojas) para poder reemplazar nodos mientras se avanza
    pila = [(nuevo, None, None, 1)]
    raiz_nueva = nuevo
    resaltar = set()
    while pila:
        n, padre, lado, prof = pila.pop()
        if generador.exponente_constante and es_exponente(padre, lado) and n.tipo != 'NUM':
            hoja = generador.exponente_aleatorio()
            exponentes.append(f"exponente '{_texto_nodo(n)}…' → '{hoja.valor}'")
            raiz_nueva = reemplazar(raiz_nueva, padre, lado, hoja)
            resaltar.add(id(hoja))
            continue
        if prof >= altura_maxima and not (n.izq is None and n.der is None):
            hoja = (generador.exponente_aleatorio() if es_exponente(padre, lado)
                    else generador.terminal_aleatorio())
            podados.append(f"'{_texto_nodo(n)}' (nivel {prof}) → hoja '{hoja.valor}'")
            raiz_nueva = reemplazar(raiz_nueva, padre, lado, hoja)
            resaltar.add(id(hoja))
            continue
        if n.der is not None:
            pila.append((n.der, n, 'der', prof + 1))
        if n.izq is not None:
            pila.append((n.izq, n, 'izq', prof + 1))
    cambios = podados + exponentes
    info = {
        'operador': 'reparacion',
        'aplicado': bool(cambios),
        'descripcion': ('Reparación: ' + '; '.join(cambios)) if cambios else 'No necesitó reparación',
        'resaltar': resaltar,
    }
    return raiz_nueva, info


# ───────────────────────────── Cruza monopunto ─────────────────────────────
def cruza_monopunto(padre1, padre2, rng, punto1=None, punto2=None):
    """Elige un nodo al azar en cada padre (cualquier altura) e intercambia sus subárboles.

    punto1 / punto2 (opcionales): índice del nodo en preorden, para demostraciones.
    Regresa (hijo1, hijo2, info). Los hijos pueden rebasar la altura: después se reparan.
    """
    h1, h2 = copiar(padre1), copiar(padre2)
    nodos1, nodos2 = listar_nodos(h1), listar_nodos(h2)
    i = rng.randrange(len(nodos1)) if punto1 is None else punto1
    j = rng.randrange(len(nodos2)) if punto2 is None else punto2
    if not (0 <= i < len(nodos1) and 0 <= j < len(nodos2)):
        raise ErrorExpresion("Punto de cruza fuera del árbol.")
    n1, p1, l1, prof1 = nodos1[i]
    n2, p2, l2, prof2 = nodos2[j]
    h1 = reemplazar(h1, p1, l1, n2)
    h2 = reemplazar(h2, p2, l2, n1)
    info = {
        'operador': 'cruza',
        'aplicado': True,
        'descripcion': (f"Cruza monopunto: el subárbol de '{_texto_nodo(n1)}' (nivel {prof1} del padre 1) "
                        f"se intercambió con el de '{_texto_nodo(n2)}' (nivel {prof2} del padre 2)"),
        'resaltar_padre1': _ids(n1),
        'resaltar_padre2': _ids(n2),
        'resaltar_hijo1': _ids(n2),
        'resaltar_hijo2': _ids(n1),
        'puntos': (i, j),
    }
    return h1, h2, info


# ───────────────────────────── Mutaciones ─────────────────────────────
def _candidatos(raiz, tipo, generador):
    """Nodos (índice en preorden) donde se puede aplicar cada tipo de mutación."""
    bib = generador.biblioteca
    binarios = [op for op in bib.operadores if op != 'neg']
    unarios = list(bib.funciones) + (['neg'] if 'neg' in bib.operadores else [])
    nodos = listar_nodos(raiz)
    resultado = []
    for k, (n, padre, lado, prof) in enumerate(nodos):
        if tipo == 'hoja':
            if n.tipo in TIPOS_OPERANDO:
                if es_exponente(padre, lado) and generador.exponente_constante:
                    alternativas = {e for e in _exponentes_validos(generador)} - {n.valor}
                else:
                    alternativas = {t for t, _ in bib.terminales} - {n.valor}
                if alternativas:
                    resultado.append(k)
        elif tipo == 'interno':
            if n.tipo == 'OP' and _binarios_validos(n, generador):
                resultado.append(k)
            elif n.tipo in ('FUNC', 'OP_UN') and len(set(unarios) - {n.valor}) > 0:
                resultado.append(k)
        elif tipo == 'switching':
            if n.tipo == 'OP' and not (n.valor == '^' and generador.exponente_constante):
                resultado.append(k)
        elif tipo == 'rama':
            resultado.append(k)
    if tipo == 'switching':
        # Se prefieren los operadores no conmutativos (− y /): en + y * el intercambio no cambia el valor
        utiles = [k for k in resultado if nodos[k][0].valor not in CONMUTATIVOS]
        resultado = utiles or resultado
    return resultado


def _binarios_validos(n, generador):
    """Operadores binarios por los que se puede cambiar el nodo n. Con exponente constante,
    '^' solo es opción si el hijo derecho ya es un número (si no, el exponente sería una expresión)."""
    return [o for o in generador.biblioteca.operadores
            if o not in ('neg', n.valor)
            and not (o == '^' and generador.exponente_constante and n.der.tipo != 'NUM')]


def _exponentes_validos(generador):
    valores = generador.biblioteca.valores
    utiles = [v for v in valores if v in (2, 3, 4)] or [v for v in valores if 0 <= v <= 4]
    return [_texto_numero(v) for v in utiles] or [t for t, _ in generador.biblioteca.terminales]


def tipos_aplicables(raiz, generador):
    return [t for t in TIPOS_MUTACION if _candidatos(raiz, t, generador)]


def mutar(raiz, tipo, generador, rng, altura_maxima, nodo=None, valor=None):
    """Aplica UN tipo de mutación ('hoja', 'interno', 'switching' o 'rama').

    nodo  (opcional): índice en preorden del nodo a mutar (para demostraciones).
    valor (opcional): nuevo valor para 'hoja' o 'interno' (para demostraciones).
    Regresa (árbol_mutado, info). Si el tipo no se puede aplicar, info['aplicado'] es False.
    """
    if tipo not in TIPOS_MUTACION:
        raise ErrorExpresion(f"Tipo de mutación no válido '{tipo}'. Opciones: {', '.join(TIPOS_MUTACION)}.")
    nuevo = copiar(raiz)
    candidatos = _candidatos(nuevo, tipo, generador)
    if not candidatos:
        return nuevo, {'operador': tipo, 'aplicado': False, 'resaltar': set(),
                       'descripcion': f"{NOMBRES_MUTACION[tipo]}: no hay nodos donde aplicarla"}
    if nodo is not None and nodo not in candidatos:
        raise ErrorExpresion(f"La {NOMBRES_MUTACION[tipo].lower()} no se puede aplicar en el nodo {nodo}.")
    k = rng.choice(candidatos) if nodo is None else nodo
    n, padre, lado, prof = listar_nodos(nuevo)[k]
    bib = generador.biblioteca

    if tipo == 'hoja':
        if es_exponente(padre, lado) and generador.exponente_constante:
            opciones = [(e, 'NUM') for e in _exponentes_validos(generador) if e != n.valor]
        else:
            opciones = [(t, tp) for t, tp in bib.terminales if t != n.valor]
        texto, tp = rng.choice(opciones) if valor is None else next(
            ((t, tp) for t, tp in opciones if t == str(valor)), (None, None))
        if texto is None:
            raise ErrorExpresion(f"'{valor}' no es un valor válido de la biblioteca para esa hoja.")
        anterior = n.valor
        n.valor, n.tipo = texto, tp
        desc = f"la hoja '{anterior}' (nivel {prof}) cambió por el valor válido '{texto}'"
        resaltar = {id(n)}

    elif tipo == 'interno':
        if n.tipo == 'OP':
            opciones = _binarios_validos(n, generador)
        else:
            unarios = list(bib.funciones) + (['neg'] if 'neg' in bib.operadores else [])
            opciones = [f for f in unarios if f != n.valor]
        nuevo_valor = rng.choice(opciones) if valor is None else str(valor)
        if nuevo_valor not in opciones:
            raise ErrorExpresion(f"'{valor}' no es un operador válido para ese nodo.")
        anterior = _texto_nodo(n)
        n.valor = nuevo_valor
        if n.tipo in ('FUNC', 'OP_UN'):
            n.tipo = 'OP_UN' if nuevo_valor == 'neg' else 'FUNC'
        desc = f"el nodo interno '{anterior}' (nivel {prof}) cambió por el operador '{_texto_nodo(n)}'"
        resaltar = {id(n)}

    elif tipo == 'switching':
        n.izq, n.der = n.der, n.izq
        nota = " (es conmutativo: el valor no cambia)" if n.valor in CONMUTATIVOS else ""
        desc = f"se intercambiaron los hijos izquierdo ↔ derecho del nodo '{n.valor}' (nivel {prof}){nota}"
        resaltar = _ids(n)

    else:  # rama nueva
        h_rama = max(1, altura_maxima - prof + 1)     # cuidando la altura del árbol
        if es_exponente(padre, lado) and generador.exponente_constante:
            rama = generador.exponente_aleatorio()
        else:
            rama = generador.generar_rama(h_rama, metodo=rng.choice(('grow', 'full')))
        nuevo = reemplazar(nuevo, padre, lado, rama)
        desc = (f"el subárbol de '{_texto_nodo(n)}' (nivel {prof}) se reemplazó por una rama nueva "
                f"de altura {altura(rama)} (máximo permitido en ese nivel: {h_rama})")
        resaltar = _ids(rama)

    return nuevo, {'operador': tipo, 'aplicado': True, 'nodo': k, 'resaltar': resaltar,
                   'descripcion': f"{NOMBRES_MUTACION[tipo]}: {desc}"}


def mutar_aleatorio(raiz, generador, rng, altura_maxima, pesos=None):
    """Elige al azar UN tipo de mutación aplicable (según 'pesos') y lo aplica. No mezcla tipos."""
    pesos = pesos or {t: 1.0 for t in TIPOS_MUTACION}
    aplicables = [t for t in tipos_aplicables(raiz, generador) if pesos.get(t, 0) > 0]
    if not aplicables:
        return copiar(raiz), {'operador': None, 'aplicado': False, 'resaltar': set(),
                              'descripcion': 'Ninguna mutación es aplicable a este árbol'}
    tipo = rng.choices(aplicables, weights=[pesos[t] for t in aplicables])[0]
    return mutar(raiz, tipo, generador, rng, altura_maxima)

## Paso 17 · Módulo `programacion_genetica.py`

**Programación genética**: algoritmo evolutivo cuyo individuo es un **árbol de expresión**.
Encuentra la ecuación que mejor reproduce los datos, usando solo la biblioteca y sin rebasar la altura máxima.
La población inicial solo acepta ecuaciones definidas en todos los datos.

In [ ]:
%%writefile arbol_expresion/programacion_genetica.py
"""
programacion_genetica.py — Programación genética: el individuo es un ÁRBOL DE EXPRESIÓN.

Busca la ECUACIÓN (su forma y sus valores) que mejor reproduce un conjunto de datos (x, y),
usando solo los elementos de una Biblioteca y sin rebasar una altura máxima.

    1. GENERAR   100 ecuaciones aleatorias (método mixto: grow + full); solo se aceptan las que
                 están definidas en todos los datos
    2. EVALUAR   aptitud = error (RMSE, MSE o NMSE) de la ecuación contra los datos
    3. MIENTRAS  no se cumpla el criterio de paro:
         → selección   : torneo o ruleta (+ elitismo)
         → cruza       : monopunto (intercambio de subárboles)
         → mutación    : UN tipo al azar: hojas, nodos internos, switching o rama nueva
         → reparación  : poda lo que rebase la altura máxima ('reparar') o se queda con el padre ('rechazar')
"""
import math

import numpy as np

from .errores import ErrorExpresion
from .config import ALTURA_MAXIMA_LIMITE
from .arbol import contar_nodos, infija_minima, altura
from .evaluador import evaluar_vectorizado
from .metricas import error_rapido, calcular_metricas, imprimir_metricas, METRICAS_APTITUD
from .analizador import ArbolExpresion
from .generador import GeneradorExpresiones
from .evolutivo import AlgoritmoEvolutivo, _entero
from . import operadores_gp as op

CONTROLES_ALTURA = ('reparar', 'rechazar')


def _validar_datos(datos):
    if not all(hasattr(datos, a) for a in ('x', 'y', 'variable')):
        raise ErrorExpresion("'datos' debe ser un ConjuntoDatos (usa generar_datos() o cargar_csv()).")
    if len(datos.x) < 2:
        raise ErrorExpresion("Se necesitan al menos 2 datos.")


def _submuestra(datos, max_puntos):
    """Toma puntos repartidos uniformemente para acelerar la evaluación (None = todos)."""
    x, y = np.asarray(datos.x, float), np.asarray(datos.y, float)
    if max_puntos is None or len(x) <= max_puntos:
        return x, y
    indices = np.unique(np.linspace(0, len(x) - 1, max_puntos).round().astype(int))
    return x[indices], y[indices]


class ProgramacionGenetica(AlgoritmoEvolutivo):
    """Programación genética sobre árboles de expresión.

    datos                 : ConjuntoDatos con los puntos (x, y) que se quieren reproducir
    biblioteca            : Biblioteca con los operadores, funciones, variables y valores válidos
    altura_maxima         : límite del tamaño de los árboles (1 a 8)
    metrica               : aptitud a minimizar: 'RMSE', 'MSE' o 'NMSE' (R² se reporta al final)
    pesos_mutacion        : probabilidad relativa de cada tipo de mutación, por ejemplo
                            {'hoja': 1, 'interno': 1, 'switching': 1, 'rama': 1}
    control_altura        : 'reparar' (poda lo que exceda) o 'rechazar' (se queda el padre)
    max_puntos            : usa como máximo estos datos para evaluar (acelera); None = todos.
                            Las métricas finales siempre se calculan con TODOS los datos.
    altura_minima_inicial : altura mínima de las ecuaciones de la población inicial
    exponente_constante   : en '^' el exponente siempre es un número pequeño de la biblioteca
    (y todos los parámetros de AlgoritmoEvolutivo: tam_poblacion, generaciones, prob_cruza,
     prob_mutacion, elitismo, seleccion, tam_torneo, objetivo, paciencia, semilla, mostrar_cada)
    """
    NOMBRE = "Programación genética"

    def __init__(self, datos, biblioteca, altura_maxima=6, metrica='RMSE', pesos_mutacion=None,
                 control_altura='reparar', max_puntos=None, altura_minima_inicial=2,
                 exponente_constante=True, **parametros):
        super().__init__(**parametros)
        _validar_datos(datos)
        if not all(hasattr(biblioteca, a) for a in ('terminales', 'no_terminales', 'variables', 'verificar')):
            raise ErrorExpresion("'biblioteca' debe ser un objeto Biblioteca.")
        if datos.variable not in biblioteca.variables:
            raise ErrorExpresion(f"Los datos usan la variable '{datos.variable}', pero la biblioteca tiene: "
                                 f"{', '.join(biblioteca.variables) or '(ninguna)'}.")
        _entero('altura_maxima', altura_maxima, 1, ALTURA_MAXIMA_LIMITE)
        _entero('altura_minima_inicial', altura_minima_inicial, 1, altura_maxima)
        if metrica not in METRICAS_APTITUD:
            raise ErrorExpresion(f"Métrica no válida '{metrica}'. Opciones: {', '.join(METRICAS_APTITUD)}.")
        if control_altura not in CONTROLES_ALTURA:
            raise ErrorExpresion(f"'control_altura' debe ser {' o '.join(map(repr, CONTROLES_ALTURA))}.")
        self.pesos_mutacion = self._validar_pesos(pesos_mutacion)
        if max_puntos is not None:
            _entero('max_puntos', max_puntos, 2, 10_000_000)

        self.datos = datos
        self.biblioteca = biblioteca
        self.altura_maxima = altura_maxima
        self.metrica = metrica
        self.nombre_aptitud = metrica
        self.control_altura = control_altura
        self.max_puntos = max_puntos
        self.variable = datos.variable
        self.x, self.y = _submuestra(datos, max_puntos)
        self.generador = GeneradorExpresiones(
            biblioteca, altura_maxima=altura_maxima, altura_minima=altura_minima_inicial,
            metodo='mixto', semilla=None if self.semilla is None else self.semilla + 1,
            exigir_variable=True,
            exponente_constante=exponente_constante)

    @staticmethod
    def _validar_pesos(pesos):
        if pesos is None:
            return {t: 1.0 for t in op.TIPOS_MUTACION}
        if not isinstance(pesos, dict):
            raise ErrorExpresion("'pesos_mutacion' debe ser un diccionario, por ejemplo {'hoja': 1, 'rama': 2}.")
        desconocidos = set(pesos) - set(op.TIPOS_MUTACION)
        if desconocidos:
            raise ErrorExpresion(f"Tipos de mutación desconocidos: {', '.join(sorted(desconocidos))}. "
                                 f"Opciones: {', '.join(op.TIPOS_MUTACION)}.")
        limpios = {}
        for t in op.TIPOS_MUTACION:
            v = pesos.get(t, 0)
            if isinstance(v, bool) or not isinstance(v, (int, float)) or v < 0 or not math.isfinite(v):
                raise ErrorExpresion(f"El peso de '{t}' debe ser un número ≥ 0.")
            limpios[t] = float(v)
        if sum(limpios.values()) == 0:
            raise ErrorExpresion("Al menos un tipo de mutación debe tener peso mayor que 0.")
        return limpios

    # ─────────────── representación: árbol de expresión ───────────────
    def individuo_aleatorio(self):
        """1. GENERAR: ecuación aleatoria definida en todos los datos de entrenamiento."""
        ultimo = None
        for _ in range(200):
            raiz = self.generador.generar().raiz
            ultimo = raiz
            if math.isfinite(self.evaluar(raiz)):
                return raiz
            self.contar('iniciales_descartados')
        return ultimo

    def aptitud(self, raiz):
        """2. EVALUAR: error de la ecuación contra los datos (inf si no está definida en algún punto)."""
        y_hat = evaluar_vectorizado(raiz, {self.variable: self.x})
        return error_rapido(self.y, y_hat, self.metrica)

    def cruzar(self, padre1, padre2):
        """→ CRUZA monopunto + control de altura inmediato (la mutación siempre recibe árboles válidos)."""
        h1, h2, _ = op.cruza_monopunto(padre1, padre2, self.rng)
        hijos = []
        for hijo, padre in ((h1, padre1), (h2, padre2)):
            if altura(hijo) > self.altura_maxima:
                self.contar('hijos_altos')
                if self.control_altura == 'rechazar':
                    hijo = op.copiar(padre)          # el hijo no vale: se conserva el padre
                    self.contar('rechazados')
            hijos.append(self.reparar(hijo))         # poda / reglas del exponente
        return hijos

    def mutar(self, raiz):
        """→ MUTACIÓN: un solo tipo, elegido al azar según los pesos."""
        nuevo, info = op.mutar_aleatorio(raiz, self.generador, self.rng, self.altura_maxima, self.pesos_mutacion)
        if info['aplicado']:
            self.contar(f"mutacion_{info['operador']}")
        return nuevo

    def reparar(self, raiz):
        """→ REPARACIÓN: garantiza que el hijo cumpla las reglas de los árboles de expresión."""
        if altura(raiz) <= self.altura_maxima and not op.validar_estructura(
                raiz, exponente_constante=self.generador.exponente_constante):
            return raiz
        reparado, info = op.reparar(raiz, self.generador, self.altura_maxima)
        if info['aplicado']:
            self.contar('reparaciones')
        return reparado

    def clave(self, raiz):
        return op.clave(raiz)

    def tamano(self, raiz):
        return contar_nodos(raiz)

    def describir(self, raiz):
        return infija_minima(raiz)

    def copiar(self, raiz):
        return op.copiar(raiz)

    # ─────────────── ejecución y resultados ───────────────
    def describir_configuracion(self):
        self.biblioteca.describir()
        pesos = ', '.join(f"{t} {p:g}" for t, p in self.pesos_mutacion.items())
        print(f"⚙️  {self.NOMBRE} → población {self.tam_poblacion} | generaciones ≤ {self.generaciones} | "
              f"altura máxima {self.altura_maxima} | aptitud {self.metrica}")
        print(f"   selección {self.seleccion}" + (f" (k = {self.tam_torneo})" if self.seleccion == 'torneo' else "")
              + f" | cruza {self.prob_cruza:.0%} | mutación {self.prob_mutacion:.0%} (pesos: {pesos})")
        print(f"   élite {self.elitismo} | control de altura: {self.control_altura} | semilla {self.semilla}"
              + (f" | objetivo ≤ {self.objetivo:g}" if self.objetivo is not None else "")
              + (f" | paciencia {self.paciencia}" if self.paciencia is not None else ""))
        print(f"   datos: {len(self.datos):,} puntos" +
              (f" (se evalúa con {len(self.x):,} repartidos uniformemente)" if len(self.x) < len(self.datos) else ""))

    def ejecutar(self, mostrar=True):
        if mostrar:
            self.describir_configuracion()
            print()
        resultado = super().ejecutar(mostrar=mostrar)

        # La mejor ecuación se vuelve a leer como texto: así pasa por TODAS las validaciones del parser
        texto = infija_minima(resultado.mejor)
        resultado.ecuacion = texto
        resultado.arbol = ArbolExpresion(texto, altura_maxima=self.altura_maxima)
        y_hat = evaluar_vectorizado(resultado.arbol.raiz, {self.variable: self.datos.x})
        resultado.metricas = calcular_metricas(self.datos.y, y_hat)
        if mostrar:
            print(f"\n🏆 Mejor ecuación encontrada:  ŷ = {texto}")
            e = resultado.arbol.estadisticas()
            print(f"   altura {e['altura']} (máx. {self.altura_maxima}) | {e['nodos']} nodos | "
                  f"elementos fuera de la biblioteca: {self.biblioteca.verificar(resultado.arbol.raiz) or 'ninguno'}")
            self.imprimir_operadores()
            print()
            imprimir_metricas(resultado.metricas, titulo="MÉTRICAS con todos los datos")
        return resultado

    def imprimir_operadores(self):
        e = self.estadisticas
        cortos = {'hoja': 'hojas', 'interno': 'nodos internos', 'switching': 'switching', 'rama': 'rama nueva'}
        partes = [f"cruzas {e.get('cruzas', 0):,}"]
        partes += [f"{cortos[t]} {e.get('mutacion_' + t, 0):,}" for t in op.TIPOS_MUTACION]
        partes.append(f"reparaciones {e.get('reparaciones', 0):,}")
        if self.control_altura == 'rechazar':
            partes.append(f"rechazados {e.get('rechazados', 0):,}")
        print("   Operadores aplicados → " + " | ".join(partes))

## Paso 18 · Módulo `ajuste_constantes.py`

**Algoritmo evolutivo de números reales**: el individuo es el vector de constantes de una ecuación fija.
Cruza **BLX-α** (cada hijo toma un valor entre los de sus padres) y mutación **gaussiana** (ruido normal pequeño).
Complementa a la programación genética: la PG encuentra la **forma** y este ajusta los **coeficientes**.

In [ ]:
%%writefile arbol_expresion/ajuste_constantes.py
"""
ajuste_constantes.py — Algoritmo evolutivo con NÚMEROS REALES: ajusta los coeficientes de una ecuación.

La programación genética encuentra la FORMA de la ecuación, pero solo puede usar los valores
de la biblioteca. Este algoritmo toma una ecuación ya fija (por ejemplo la mejor que encontró
la programación genética) y busca los valores de sus constantes que minimizan el error.

    Individuo : vector con las constantes de la ecuación, por ejemplo [6.9, 10.8, 7.2]
    1. GENERAR   vectores aleatorios alrededor de los valores iniciales (el original también entra)
    2. EVALUAR   error de la ecuación con esas constantes contra los datos
    3. MIENTRAS  no se cumpla el criterio de paro:
         → selección : torneo o ruleta (+ elitismo)
         → cruza     : BLX-α (cada hijo toma, gen por gen, un valor al azar en el intervalo
                       entre los dos padres, ampliado un α por cada lado)
         → mutación  : gaussiana (a algunas constantes se les suma un ruido normal pequeño)

Por defecto no se modifican los exponentes de '^' (cambiar x^3 por x^2.97 casi nunca tiene sentido
y con x negativa da números complejos).
"""
import math

import numpy as np

from .errores import ErrorExpresion
from .arbol import Nodo, infija_minima
from .evaluador import evaluar_vectorizado
from .metricas import error_rapido, calcular_metricas, imprimir_metricas, METRICAS_APTITUD
from .analizador import ArbolExpresion
from .evolutivo import AlgoritmoEvolutivo, _entero
from .programacion_genetica import _validar_datos, _submuestra
from . import operadores_gp as op


def _numero_positivo(nombre, valor):
    if isinstance(valor, bool) or not isinstance(valor, (int, float)) or not math.isfinite(valor) or valor <= 0:
        raise ErrorExpresion(f"'{nombre}' debe ser un número mayor que 0 (recibido: {valor!r}).")


class AjusteConstantes(AlgoritmoEvolutivo):
    """Algoritmo evolutivo de valores reales para los coeficientes de una ecuación fija.

    modelo              : texto o ArbolExpresion (por ejemplo "7 + 10*cos(7.2*x^3)")
    datos               : ConjuntoDatos
    rango               : la población inicial se reparte en ±rango·|valor| alrededor de cada constante
    sigma               : tamaño relativo del ruido de la mutación gaussiana
    alfa                : α de la cruza BLX-α
    incluir_exponentes  : también ajusta los exponentes de '^'
    decimales           : redondeo de las constantes en la ecuación final
    metrica, max_puntos y los parámetros de AlgoritmoEvolutivo (tam_poblacion, generaciones, ...)
    """
    NOMBRE = "Algoritmo evolutivo (ajuste de constantes)"

    def __init__(self, modelo, datos, rango=0.1, sigma=0.01, alfa=0.3, incluir_exponentes=False,
                 decimales=6, metrica='RMSE', max_puntos=None, **parametros):
        parametros.setdefault('prob_mutacion', 0.5)
        super().__init__(**parametros)
        _validar_datos(datos)
        for nombre, valor in (('rango', rango), ('sigma', sigma)):
            _numero_positivo(nombre, valor)
        if isinstance(alfa, bool) or not isinstance(alfa, (int, float)) or not 0 <= alfa <= 1:
            raise ErrorExpresion("'alfa' debe ser un número entre 0 y 1.")
        _entero('decimales', decimales, 0, 12)
        if metrica not in METRICAS_APTITUD:
            raise ErrorExpresion(f"Métrica no válida '{metrica}'. Opciones: {', '.join(METRICAS_APTITUD)}.")
        if max_puntos is not None:
            _entero('max_puntos', max_puntos, 2, 10_000_000)

        arbol = modelo if hasattr(modelo, 'raiz') else ArbolExpresion(modelo)
        vars_ = arbol.variables()
        if vars_ and vars_ != [datos.variable]:
            raise ErrorExpresion(f"El modelo usa {', '.join(vars_)} y los datos usan '{datos.variable}'.")

        self.plantilla = op.copiar(arbol.raiz)
        self.posiciones = [n for n, padre, lado, _ in op.listar_nodos(self.plantilla)
                           if n.tipo == 'NUM' and (incluir_exponentes or not op.es_exponente(padre, lado))]
        if not self.posiciones:
            raise ErrorExpresion("La ecuación no tiene constantes numéricas que ajustar.")
        self.iniciales = np.array([float(n.valor) for n in self.posiciones])
        self.escala = np.where(self.iniciales != 0, np.abs(self.iniciales), 1.0)
        self.rango, self.sigma, self.alfa = float(rango), float(sigma), float(alfa)
        self.decimales = decimales
        self.metrica = metrica
        self.nombre_aptitud = metrica
        self.datos = datos
        self.variable = datos.variable
        self.x, self.y = _submuestra(datos, max_puntos)
        self._creados = 0

    # ─────────────── representación: vector de reales ───────────────
    def _aplicar(self, valores):
        for nodo, v in zip(self.posiciones, valores):
            nodo.valor = repr(float(v))
        return self.plantilla

    def individuo_aleatorio(self):
        self._creados += 1
        if self._creados == 1:
            return self.iniciales.copy()          # el modelo original siempre participa
        ruido = np.array([self.rng.uniform(-1, 1) for _ in self.iniciales])
        return self.iniciales + ruido * self.rango * self.escala

    def poblacion_inicial(self):
        self._creados = 0
        return super().poblacion_inicial()

    def aptitud(self, valores):
        y_hat = evaluar_vectorizado(self._aplicar(valores), {self.variable: self.x})
        return error_rapido(self.y, y_hat, self.metrica)

    def cruzar(self, a, b):
        hijos = []
        for _ in range(2):
            hijo = np.empty_like(a)
            for i, (u, v) in enumerate(zip(a, b)):
                bajo, alto = min(u, v), max(u, v)
                d = alto - bajo
                hijo[i] = self.rng.uniform(bajo - self.alfa * d, alto + self.alfa * d)
            hijos.append(hijo)
        return hijos

    def mutar(self, valores):
        nuevo = valores.copy()
        prob = 1.0 / len(nuevo)
        cambiados = [i for i in range(len(nuevo)) if self.rng.random() < prob] or \
                    [self.rng.randrange(len(nuevo))]
        for i in cambiados:
            nuevo[i] += self.rng.gauss(0.0, self.sigma * self.escala[i])
        self.contar('mutaciones_gaussianas')
        return nuevo

    def clave(self, valores):
        return tuple(np.round(valores, 12))

    def describir(self, valores):
        return infija_minima(self._construir(valores))

    def copiar(self, valores):
        return valores.copy()

    # ─────────────── resultado ───────────────
    def _construir(self, valores):
        """Árbol con las constantes redondeadas; un valor negativo se escribe como −(valor)."""
        reemplazo = {id(n): round(float(v), self.decimales) for n, v in zip(self.posiciones, valores)}

        def copiar(n):
            if n is None:
                return None
            if id(n) in reemplazo:
                v = reemplazo[id(n)]
                # sin notación científica (el lector de expresiones no la acepta)
                texto = f"{abs(v):.{self.decimales}f}".rstrip('0').rstrip('.') or '0'
                hoja = Nodo(texto, 'NUM')
                return Nodo('neg', 'OP_UN', der=hoja) if v < 0 else hoja
            return Nodo(n.valor, n.tipo, copiar(n.izq), copiar(n.der))

        return copiar(self.plantilla)

    def ejecutar(self, mostrar=True):
        if mostrar:
            print(f"🔧 Ecuación a ajustar: ŷ = {infija_minima(self._construir(self.iniciales))}")
            print(f"   Constantes: {', '.join(f'{v:g}' for v in self.iniciales)} "
                  f"(población inicial en ±{self.rango:.0%} de cada una)\n")
        resultado = super().ejecutar(mostrar=mostrar)
        texto = infija_minima(self._construir(resultado.mejor))
        resultado.ecuacion = texto
        resultado.arbol = ArbolExpresion(texto)
        resultado.constantes = [round(float(v), self.decimales) for v in resultado.mejor]
        y_hat = evaluar_vectorizado(resultado.arbol.raiz, {self.variable: self.datos.x})
        resultado.metricas = calcular_metricas(self.datos.y, y_hat)
        if mostrar:
            print(f"\n🏆 Ecuación ajustada:  ŷ = {texto}")
            print("   Constantes: " + ', '.join(f"{a:g} → {b:g}" for a, b in zip(self.iniciales, resultado.constantes)))
            print()
            imprimir_metricas(resultado.metricas, titulo="MÉTRICAS con todos los datos")
        return resultado

## Paso 19 · Módulo `demostraciones.py`

Aplica cada operador a un árbol, imprime qué cambió y dibuja antes → después.

In [ ]:
%%writefile arbol_expresion/demostraciones.py
"""
demostraciones.py — Muestra cada operador genético paso a paso: imprime qué cambió y dibuja
el árbol ANTES y DESPUÉS (con los nodos modificados resaltados en amarillo).

Sirven para explicar en clase cada operador por separado; el algoritmo completo los usa
de la misma forma (programacion_genetica.py).
"""
import random

from .errores import ErrorExpresion
from .analizador import ArbolExpresion
from .arbol import infija_minima, altura
from .generador import GeneradorExpresiones
from .graficos import dibujar_comparacion
from . import operadores_gp as op


def _raiz(expresion):
    """Acepta texto, ArbolExpresion o directamente un Nodo (por ejemplo un hijo de la cruza)."""
    if hasattr(expresion, 'raiz'):
        return expresion.raiz
    if all(hasattr(expresion, a) for a in ('valor', 'tipo', 'izq', 'der')):
        return expresion
    return ArbolExpresion(expresion).raiz


def _generador(biblioteca, altura_maxima, semilla):
    return GeneradorExpresiones(biblioteca, altura_maxima=max(altura_maxima, 1), semilla=semilla,
                                exigir_variable=False)


def numerar_nodos(expresion):
    """Imprime los nodos en preorden con su número (para elegir el nodo de una demostración)."""
    raiz = _raiz(expresion)
    print(f"Nodos de {infija_minima(raiz)} (preorden):")
    for k, (n, padre, lado, prof) in enumerate(op.listar_nodos(raiz)):
        tipo = 'hoja' if n.izq is None and n.der is None else 'interno'
        print(f"   {k:>2}: '{'−' if n.tipo == 'OP_UN' else n.valor}'  nivel {prof}, {tipo}"
              + (f", hijo {lado} de '{padre.valor}'" if padre is not None else ", raíz"))


def demostrar_mutacion(expresion, tipo, biblioteca, altura_maxima=4, nodo=None, valores=None,
                       semilla=None, guardar_como=None):
    """Aplica un tipo de mutación y dibuja antes → después.

    nodo    : número del nodo en preorden (usa numerar_nodos para verlos); None = al azar
    valores : solo para 'hoja' o 'interno': lista de valores nuevos; se dibuja un resultado por valor
              (por ejemplo ['7.2', 'pi', 'x'] como en el apunte)
    """
    raiz = _raiz(expresion)
    if altura(raiz) > altura_maxima:
        raise ErrorExpresion(f"El árbol tiene altura {altura(raiz)}, mayor que la altura máxima {altura_maxima}.")
    gen = _generador(biblioteca, altura_maxima, semilla)
    rng = random.Random(semilla)
    resultados = []
    for v in (valores or [None]):
        mutado, info = op.mutar(raiz, tipo, gen, rng, altura_maxima, nodo=nodo, valor=v)
        resultados.append((mutado, info))

    print(f"🧬 {op.NOMBRES_MUTACION[tipo]}")
    print(f"   Antes  : {infija_minima(raiz)}")
    for mutado, info in resultados:
        print(f"   Después: {infija_minima(mutado):<28} ← {info['descripcion'].split(': ', 1)[-1]}")
        problemas = op.validar_estructura(mutado, biblioteca, altura_maxima)
        print(f"            ¿árbol válido? {'sí' if not problemas else 'NO: ' + '; '.join(problemas)}")
    marcados_antes = set()
    if resultados[0][1].get('aplicado') and 'nodo' in resultados[0][1]:
        k = resultados[0][1]['nodo']
        n = op.listar_nodos(raiz)[k][0]
        marcados_antes = {id(n)} if tipo in ('hoja', 'interno') else {id(m) for m, *_ in op.listar_nodos(n)}
    dibujar_comparacion(
        [raiz] + [m for m, _ in resultados],
        ['Antes'] + [f"Después: {infija_minima(m)}" for m, _ in resultados],
        resaltar=[marcados_antes] + [i['resaltar'] for _, i in resultados],
        titulo_general=op.NOMBRES_MUTACION[tipo], guardar_como=guardar_como, limite=altura_maxima)
    return [m for m, _ in resultados]


def demostrar_cruza(padre1, padre2, punto1=None, punto2=None, semilla=None, guardar_como=None):
    """Cruza monopunto: intercambia los subárboles de los nodos elegidos y dibuja padres → hijos."""
    r1, r2 = _raiz(padre1), _raiz(padre2)
    h1, h2, info = op.cruza_monopunto(r1, r2, random.Random(semilla), punto1, punto2)
    i, j = info['puntos']
    n1, n2 = op.listar_nodos(r1)[i][0], op.listar_nodos(r2)[j][0]
    print("✂️  Cruza monopunto")
    print(f"   Padre 1: {infija_minima(r1):<26} punto de cruza: nodo {i} ('{n1.valor}')")
    print(f"   Padre 2: {infija_minima(r2):<26} punto de cruza: nodo {j} ('{n2.valor}')")
    print(f"   Hijo 1 : {infija_minima(h1)}   (altura {altura(h1)})")
    print(f"   Hijo 2 : {infija_minima(h2)}   (altura {altura(h2)})")
    dibujar_comparacion(
        [r1, r2, h1, h2], ['Padre 1', 'Padre 2', 'Hijo 1', 'Hijo 2'],
        resaltar=[{id(m) for m, *_ in op.listar_nodos(n1)}, {id(m) for m, *_ in op.listar_nodos(n2)},
                  info['resaltar_hijo1'], info['resaltar_hijo2']],
        titulo_general="Cruza monopunto (intercambio de subárboles)", guardar_como=guardar_como)
    return h1, h2


def demostrar_reparacion(expresion, biblioteca, altura_maxima, semilla=None, guardar_como=None):
    """Método de reparación: poda lo que rebasa la altura máxima y lo reemplaza por hojas válidas."""
    raiz = _raiz(expresion)
    gen = _generador(biblioteca, altura_maxima, semilla)
    reparado, info = op.reparar(raiz, gen, altura_maxima)
    print("🛠️  Método de reparación")
    print(f"   Antes  : {infija_minima(raiz)}   (altura {altura(raiz)}, máximo {altura_maxima})")
    print(f"   Después: {infija_minima(reparado)}   (altura {altura(reparado)})")
    print(f"   {info['descripcion']}")
    excedidos = {id(n) for n, _, _, prof in op.listar_nodos(raiz) if prof > altura_maxima}
    dibujar_comparacion([raiz, reparado], ['Antes (rebasa el límite)', 'Después de reparar'],
                        resaltar=[excedidos, info['resaltar']], limite=altura_maxima,
                        titulo_general="Método de reparación (poda)", guardar_como=guardar_como)
    return reparado

## Paso 20 · Módulo `interactivo.py`

Modo interactivo: escribe expresiones y valores con el teclado.

In [ ]:
%%writefile arbol_expresion/interactivo.py
"""
interactivo.py — Modo interactivo por consola (funciona en Colab con input()).
"""
from .errores import ErrorExpresion
from .analizador import ArbolExpresion, analizar


def _pedir_valores(variables, leer):
    """Pide el valor de cada variable. Regresa None si el usuario deja uno vacío."""
    valores = {}
    for v in variables:
        while True:
            texto = leer(f"   Valor de {v} (Enter para no evaluar): ").strip().replace(',', '.')
            if texto == '':
                return None
            try:
                valores[v] = float(texto)
                break
            except ValueError:
                print("   ⚠️ Escribe un número válido, por ejemplo 2 o -3.5")
    return valores


def modo_interactivo(mostrar_pasos=True, leer=None):
    """leer: función para leer texto (por defecto input). Permite probar el modo sin teclado."""
    leer = leer or input
    print("Escribe una expresión (o 'salir' para terminar).")
    print("Ejemplos: sqrt(2*(x^2)+5*x-3) | (a+b)*(a-b) | 3x^2 - 2x + 1 | 6.87 + 11*cos(7.23*x^3)\n")
    while True:
        expr = leer("➤ Expresión: ").strip()
        if expr.lower() in ('salir', 'exit', 'q'):
            print("👋 Fin del modo interactivo.")
            break
        try:
            a = ArbolExpresion(expr)
        except ErrorExpresion as err:
            print(f"❌ {err}\n")
            continue
        valores = _pedir_valores(a.variables(), leer)
        analizar(expr, valores=valores, mostrar_pasos=mostrar_pasos)
        print()

## Paso 21 · Módulo `pruebas.py`

Pruebas de sintaxis, precedencia, dominio, altura máxima, biblioteca, generador, evaluación vectorizada,
métricas (comparadas con valores calculados a mano), datos, CSV, ajuste, operadores genéticos
(sobre 300 árboles aleatorios), programación genética y ajuste de constantes.

In [ ]:
%%writefile arbol_expresion/pruebas.py
"""
pruebas.py — Pruebas automáticas de validación, recorridos y evaluación.
"""
import math

from .errores import ErrorExpresion
from .analizador import ArbolExpresion

# (expresión, ¿debe ser válida?)
CASOS_SINTAXIS = [
    ("sqrt(2*(x^2)+5*x-3)", True),
    ("-x^2 + 3*(x - 1)/2", True),
    ("2^3^2", True),
    ("sin(pi/2) + ln(e)", True),
    ("(x+1)(x-1)", True),
    ("√(x)", True),
    ("", False),
    ("2*(x+3", False),
    ("2*x+3)", False),
    ("2*/x", False),
    ("*x+1", False),
    ("x+", False),
    ("()", False),
    ("sqrt x", False),
    ("raiz(x)", False),
    ("2 3 + x", False),
    ("3.4.5 + x", False),
    ("x & 2", False),
    ("y = 2x + 1", False),
]

# (expresión, valores, resultado esperado)
CASOS_EVALUACION = [
    ("sqrt(2*(x^2)+5*x-3)", {'x': 2}, math.sqrt(15)),
    ("2x^2 + 5x - 3", {'x': 2}, 15.0),
    ("2^3^2", {}, 512.0),          # asociatividad derecha
    ("-x^2", {'x': 3}, -9.0),      # el menos unario va después de la potencia
    ("10 - 4 - 3", {}, 3.0),       # asociatividad izquierda
    ("sin(pi/2) + ln(e)", {}, 2.0),
]

# (expresión, valores) que deben producir error de dominio
CASOS_DOMINIO = [
    ("sqrt(2*(x^2)+5*x-3)", {'x': 0}),
    ("1/(x-2)", {'x': 2}),
    ("ln(x)", {'x': -1}),
    ("(-8)^(1/3)", {}),
    ("exp(x)", {'x': 1000}),
]


def ejecutar_pruebas(detalle=True):
    total = aciertos = 0

    print("1) Validación sintáctica")
    for expr, esperado in CASOS_SINTAXIS:
        try:
            ArbolExpresion(expr)
            valida, msg = True, ""
        except ErrorExpresion as err:
            valida, msg = False, str(err).split('\n')[0]
        ok = valida == esperado
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            estado = "VÁLIDA  " if valida else "INVÁLIDA"
            print(f"   {'✔' if ok else '✘'} {estado} │ {expr!r:<24} │ {msg}")

    print("\n2) Evaluación y precedencia")
    for expr, vals, esperado in CASOS_EVALUACION:
        a = ArbolExpresion(expr)
        r = a.evaluar(vals)
        ok = math.isclose(r, esperado, rel_tol=1e-9) and a.es_consistente()
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con {vals} = {r:.6g} (esperado {esperado:.6g})")

    print("\n3) Errores de dominio")
    for expr, vals in CASOS_DOMINIO:
        try:
            ArbolExpresion(expr).evaluar(vals)
            ok, msg = False, "no se detectó el error"
        except ErrorExpresion as err:
            ok, msg = True, str(err)
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con {vals}: {msg}")

    print("\n4) Altura máxima en expresiones escritas a mano")
    for expr, h_max, esperado in [("sqrt(2*(x^2)+5*x-3)", 6, True),
                                  ("sqrt(2*(x^2)+5*x-3)", 4, False),
                                  ("x", 1, True)]:
        try:
            ArbolExpresion(expr, altura_maxima=h_max)
            valida, msg = True, "aceptada"
        except ErrorExpresion as err:
            valida, msg = False, str(err)
        ok = valida == esperado
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {expr} con altura máxima {h_max}: {msg}")

    print("\n5) Biblioteca y generador: configuraciones inválidas")
    from .generador import Biblioteca, GeneradorExpresiones
    malas = [
        ("operador desconocido", lambda: Biblioteca(operadores=['+', '%'])),
        ("función desconocida", lambda: Biblioteca(funciones=['raiz'])),
        ("variable de 2 letras", lambda: Biblioteca(variables=['xy'])),
        ("variable 'e'", lambda: Biblioteca(variables=['e'])),
        ("valor negativo", lambda: Biblioteca(valores=[-2])),
        ("valor no numérico", lambda: Biblioteca(valores=['3'])),
        ("valores repetidos", lambda: Biblioteca(valores=[1, 1])),
        ("sin terminales", lambda: Biblioteca(variables=[], valores=[])),
        ("sin operadores ni funciones", lambda: Biblioteca(operadores=[], funciones=[])),
        ("altura 0", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=0)),
        ("altura mayor al límite", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=50)),
        ("altura no entera", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=3.5)),
        ("altura mínima > máxima", lambda: GeneradorExpresiones(Biblioteca(), altura_maxima=3, altura_minima=5)),
        ("método desconocido", lambda: GeneradorExpresiones(Biblioteca(), metodo='otro')),
        ("probabilidad fuera de [0,1]", lambda: GeneradorExpresiones(Biblioteca(), prob_terminal=2)),
        ("puntos con variable ajena", lambda: GeneradorExpresiones(Biblioteca(), puntos_validos={'y': [1]})),
    ]
    for nombre, crear in malas:
        try:
            crear()
            ok, msg = False, "no se detectó"
        except ErrorExpresion as err:
            ok, msg = True, str(err)
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} {nombre}: {msg}")

    print("\n6) Generación: 300 ecuaciones por método respetan altura (3 a 5), biblioteca y dominio")
    bib = Biblioteca(operadores=['+', '-', '*', '/', '^', 'neg'], funciones=['sqrt', 'sin'],
                     variables=['x'], valores=[1, 2, 3, 0.5], constantes=['pi'])
    for metodo in ('grow', 'full', 'mixto'):
        gen = GeneradorExpresiones(bib, altura_maxima=5, altura_minima=3, metodo=metodo, semilla=7,
                                   puntos_validos={'x': [1, 2, 3]})
        problemas = 0
        for _ in range(300):
            a = gen.generar()
            if (not 3 <= a.altura() <= 5 or bib.verificar(a.raiz) or not a.variables()
                    or not a.es_consistente()):
                problemas += 1
            for x in (1, 2, 3):
                a.evaluar(x=x)            # no debe lanzar error
        ok = problemas == 0
        total, aciertos = total + 1, aciertos + ok
        if detalle:
            print(f"   {'✔' if ok else '✘'} método {metodo}: {problemas} ecuaciones con problemas")

    gen_a = GeneradorExpresiones(bib, altura_maxima=4, semilla=123)
    gen_b = GeneradorExpresiones(bib, altura_maxima=4, semilla=123)
    ok = [gen_a.generar().infija_minima() for _ in range(5)] == \
         [gen_b.generar().infija_minima() for _ in range(5)]
    total, aciertos = total + 1, aciertos + ok
    if detalle:
        print(f"   {'✔' if ok else '✘'} la misma semilla produce las mismas ecuaciones")

    def registrar(ok, texto):
        nonlocal total, aciertos
        total, aciertos = total + 1, aciertos + bool(ok)
        if detalle:
            print(f"   {'✔' if ok else '✘'} {texto}")

    def falla_con_error(funcion):
        try:
            funcion()
            return False, "no se detectó"
        except ErrorExpresion as err:
            return True, str(err).splitlines()[0]

    print("\n7) Evaluación vectorizada = evaluación punto por punto (incluye errores de dominio)")
    import numpy as np
    from .evaluador import evaluar, evaluar_vectorizado
    xs = np.array([-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0])
    gen = GeneradorExpresiones(bib, altura_maxima=5, metodo='mixto', semilla=99, exigir_variable=True)
    discrepancias = 0
    for _ in range(300):
        a = gen.generar()
        vector = evaluar_vectorizado(a.raiz, {'x': xs})
        for x, v in zip(xs, vector):
            try:
                escalar = evaluar(a.raiz, {'x': float(x)})
            except ErrorExpresion:
                escalar = math.nan
            if math.isnan(escalar) != math.isnan(v) or \
                    (not math.isnan(v) and not math.isclose(escalar, v, rel_tol=1e-9, abs_tol=1e-12)):
                discrepancias += 1
    registrar(discrepancias == 0, f"300 ecuaciones × 8 puntos: {discrepancias} discrepancias")

    print("\n8) Métricas (valores calculados a mano)")
    from .metricas import mse, rmse, nmse, r2, calcular_metricas
    y, y_hat = [1, 2, 3], [1, 2, 4]
    registrar(math.isclose(mse(y, y_hat), 1 / 3), "MSE([1,2,3],[1,2,4]) = 1/3")
    registrar(math.isclose(rmse(y, y_hat), math.sqrt(1 / 3)), "RMSE = √(1/3)")
    registrar(math.isclose(nmse(y, y_hat), 1 / 14), "NMSE = 1/14  (Σ|y−ŷ|² = 1, Σ|y|² = 14)")
    registrar(math.isclose(r2(y, y_hat), 0.5), "R² = 1 − 1/2 = 0.5  (Σ(y−ȳ)² = 2)")
    registrar(math.isclose(r2(y, [2, 2, 2]), 0.0), "R² = 0 cuando el modelo siempre predice ȳ")
    m = calcular_metricas(y, y)
    registrar(m['MSE'] == 0 and m['NMSE'] == 0 and m['R2'] == 1, "modelo perfecto: MSE = NMSE = 0 y R² = 1")
    m = calcular_metricas(y, [1, math.nan, 3])
    registrar(m['invalidos'] == 1 and m['RMSE'] == math.inf, "ŷ no definido en un punto → error infinito")
    for nombre, f in [("tamaños distintos", lambda: mse([1, 2], [1])),
                      ("sin datos", lambda: mse([], [])),
                      ("R² con y constante", lambda: r2([5, 5, 5], [5, 5, 4])),
                      ("NMSE con y = 0", lambda: nmse([0, 0], [1, 1]))]:
        ok, msg = falla_con_error(f)
        registrar(ok, f"{nombre}: {msg}")

    print("\n9) Datos y CSV: y = 6.87 + 11*cos(7.23*x^3) en [-10, 10] con paso 0.001")
    import os
    import tempfile
    from .datos import generar_datos, guardar_csv, cargar_csv
    d = generar_datos("6.87 + 11*cos(7.23*x^3)", -10.0, 10.0, 0.001)
    registrar(len(d) == 20001, f"{len(d):,} renglones (esperados 20,001)")
    registrar(d.x[0] == -10.0 and d.x[-1] == 10.0 and d.x[10000] == 0.0, "x empieza en -10, pasa por 0 y termina en 10")
    registrar(np.allclose(np.diff(d.x), 0.001), "paso constante de 0.001")
    registrar(math.isclose(d.y[10000], 17.87), "y(0) = 6.87 + 11·cos(0) = 17.87")
    registrar(math.isclose(d.y[0], 6.87 + 11 * math.cos(7.23 * (-10.0) ** 3)), "y(−10) coincide con math.cos")
    with tempfile.TemporaryDirectory() as carpeta:
        ruta = os.path.join(carpeta, "prueba.csv")
        guardar_csv(d, ruta, decimales_y=6, mostrar=False)
        with open(ruta, encoding='utf-8') as f:
            lineas = f.read().splitlines()
        registrar(lineas[0] == "x,y" and len(lineas) == 20002, "CSV con encabezado 'x,y' y 20,001 renglones")
        registrar(lineas[1].startswith("-10.000,") and lineas[-1].startswith("10.000,"),
                  f"primer y último renglón: {lineas[1]} … {lineas[-1]}")
        leido = cargar_csv(ruta, mostrar=False)
        registrar(np.array_equal(leido.x, d.x) and np.max(np.abs(leido.y - d.y)) <= 5e-7,
                  "el CSV se vuelve a leer igual (y redondeado a 6 decimales)")
        malo = os.path.join(carpeta, "malo.csv")
        for contenido, nombre in [("x,y\n1,2\n2,abc\n", "valor no numérico"),
                                  ("x,y,z\n1,2,3\n4,5\n", "un renglón incompleto"),
                                  ("x,y\n", "solo encabezado")]:
            with open(malo, 'w', encoding='utf-8') as f:
                f.write(contenido)
            ok, msg = falla_con_error(lambda: cargar_csv(malo, mostrar=False))
            registrar(ok, f"CSV con {nombre}: {msg}")
        otro = os.path.join(carpeta, "excel.csv")
        with open(otro, 'w', encoding='utf-8') as f:
            f.write("x;y\n0,5;1,25\n1,0;2,5\n")
        e = cargar_csv(otro, mostrar=False)
        registrar(list(e.x) == [0.5, 1.0] and list(e.y) == [1.25, 2.5], "CSV de Excel con ';' y coma decimal")
        with open(otro, 'w', encoding='utf-8') as f:
            f.write("0.0,2.71\n0.001,7.27\n")
        e = cargar_csv(otro, mostrar=False)
        registrar(list(e.x) == [0.0, 0.001] and list(e.y) == [2.71, 7.27], "CSV sin encabezado")
        ok, msg = falla_con_error(lambda: cargar_csv(os.path.join(carpeta, "no_existe.csv")))
        registrar(ok, f"CSV inexistente: {msg}")
    for nombre, f in [("inicio ≥ fin", lambda: generar_datos("x", 10, -10, 0.001)),
                      ("paso 0", lambda: generar_datos("x", -10, 10, 0)),
                      ("paso que no divide el intervalo", lambda: generar_datos("x", 0, 1, 0.3)),
                      ("dos variables", lambda: generar_datos("x*y", -1, 1, 0.1)),
                      ("demasiados puntos", lambda: generar_datos("x", -1e6, 1e6, 0.0001))]:
        ok, msg = falla_con_error(f)
        registrar(ok, f"{nombre}: {msg}")
    con_huecos = generar_datos("sqrt(x)", -1, 1, 0.5, mostrar=False)
    registrar(list(con_huecos.x) == [0.0, 0.5, 1.0], "sqrt(x) en [-1, 1]: se omiten los puntos no definidos")

    print("\n10) Ajuste de modelos contra los datos")
    from .ajuste import evaluar_modelo, comparar_modelos
    m = evaluar_modelo("6.87 + 11*cos(7.23*x^3)", d, mostrar=False)
    registrar(m['RMSE'] == 0 and m['R2'] == 1, "la ecuación original tiene RMSE = 0 y R² = 1")
    m_media = evaluar_modelo(f"{float(np.mean(d.y))!r}", d, mostrar=False)
    registrar(abs(m_media['R2']) < 1e-9, "un modelo constante igual a ȳ tiene R² = 0")
    orden = comparar_modelos(["6.87", "6.87 + 11*cos(7.23*x^3)", "6.87 + 11*cos(7.2*x^3)"], d, mostrar=False)
    registrar(orden[0]['modelo'] == "6.87 + 11*cos(7.23*x^3)", "comparar_modelos pone primero a la ecuación exacta")
    m = evaluar_modelo("1/x", d, mostrar=False)
    registrar(m['invalidos'] == 1 and m['RMSE'] == math.inf, "1/x no está definido en x = 0 → modelo inválido")

    print("\n11) Operadores genéticos sobre 300 árboles aleatorios")
    import random as _random
    from . import operadores_gp as op
    from .arbol import contar_nodos, altura as _altura, infija_minima
    from .generador import _firma
    bib_gp = Biblioteca(operadores=['+', '-', '*', '/', '^', 'neg'], funciones=['sqrt', 'cos', 'sin'],
                        variables=['x'], valores=[1, 2, 3, 0.5], constantes=['pi'])
    H = 5
    gen_gp = GeneradorExpresiones(bib_gp, altura_maxima=H, metodo='mixto', semilla=11, exigir_variable=False)
    rng = _random.Random(5)
    arboles = [gen_gp.generar().raiz for _ in range(300)]
    for tipo in op.TIPOS_MUTACION:
        invalidos = cambiaron_original = aplicadas = 0
        for raiz in arboles:
            antes = op.clave(raiz)
            mutado, info = op.mutar(raiz, tipo, gen_gp, rng, H)
            aplicadas += info['aplicado']
            if op.clave(raiz) != antes:
                cambiaron_original += 1
            if op.validar_estructura(mutado, bib_gp, H, exponente_constante=True):
                invalidos += 1
        registrar(invalidos == 0 and cambiaron_original == 0 and aplicadas > 0,
                  f"{op.NOMBRES_MUTACION[tipo]}: aplicada {aplicadas} veces, {invalidos} árboles inválidos, "
                  f"{cambiaron_original} originales modificados")

    # Cada tipo hace SOLO su cambio
    ok_hoja = ok_interno = ok_switch = True
    for raiz in arboles[:100]:
        m, info = op.mutar(raiz, 'hoja', gen_gp, rng, H)
        a, b = op.listar_nodos(raiz), op.listar_nodos(m)
        if info['aplicado']:
            dif = [k for k, (u, v) in enumerate(zip(a, b)) if u[0].valor != v[0].valor]
            ok_hoja &= len(a) == len(b) and len(dif) == 1 and a[dif[0]][0].izq is None and a[dif[0]][0].der is None
        m, info = op.mutar(raiz, 'interno', gen_gp, rng, H)
        a, b = op.listar_nodos(raiz), op.listar_nodos(m)
        if info['aplicado']:
            dif = [k for k, (u, v) in enumerate(zip(a, b)) if u[0].valor != v[0].valor]
            ok_interno &= len(a) == len(b) and len(dif) == 1 and \
                ((a[dif[0]][0].izq is None) == (b[dif[0]][0].izq is None))
        m, info = op.mutar(raiz, 'switching', gen_gp, rng, H)
        if info['aplicado']:
            n_antes = op.listar_nodos(raiz)[info['nodo']][0]
            n_desp = op.listar_nodos(m)[info['nodo']][0]
            ok_switch &= op.clave(n_antes.izq) == op.clave(n_desp.der) and op.clave(n_antes.der) == op.clave(n_desp.izq)
    registrar(ok_hoja, "mutación de hojas: cambia exactamente UNA hoja y nada más")
    registrar(ok_interno, "mutación de nodos internos: cambia UN operador y conserva el número de hijos")
    registrar(ok_switch, "switching: el hijo izquierdo pasa a la derecha y viceversa")

    conserva = validos = True
    for k in range(0, 298, 2):
        p1, p2 = arboles[k], arboles[k + 1]
        h1, h2, _ = op.cruza_monopunto(p1, p2, rng)
        conserva &= contar_nodos(h1) + contar_nodos(h2) == contar_nodos(p1) + contar_nodos(p2)
        for h in (h1, h2):
            r, _ = op.reparar(h, gen_gp, H)
            validos &= not op.validar_estructura(r, bib_gp, H, exponente_constante=True)
    registrar(conserva, "cruza monopunto: los hijos tienen en total los mismos nodos que los padres")
    registrar(validos, "cruza + reparación: todos los hijos respetan la altura máxima y las reglas")

    ida_vuelta = 0
    for raiz in arboles:
        m, _ = op.mutar_aleatorio(raiz, gen_gp, rng, H)
        if _firma(ArbolExpresion(infija_minima(m)).raiz) != _firma(m):
            ida_vuelta += 1
    registrar(ida_vuelta == 0, f"texto ↔ árbol: los 300 árboles mutados se escriben y se vuelven a leer igual "
                               f"({ida_vuelta} diferencias)")

    print("\n12) Programación genética")
    from .programacion_genetica import ProgramacionGenetica
    from .ajuste_constantes import AjusteConstantes
    d_pol = generar_datos("2*x^2 + 5*x - 3", -5, 5, 0.05, mostrar=False)
    bib_pol = Biblioteca(operadores=['+', '-', '*'], funciones=[], variables=['x'], valores=[1, 2, 3, 5])
    gp = ProgramacionGenetica(d_pol, bib_pol, altura_maxima=5, generaciones=60, semilla=1,
                              objetivo=1e-9, mostrar_cada=0)
    r = gp.ejecutar(mostrar=False)
    registrar(r.metricas['RMSE'] < 1e-9 and math.isclose(r.metricas['R2'], 1.0),
              f"encuentra 2x² + 5x − 3 exacta: ŷ = {r.ecuacion} (generación {r.generaciones})")
    mejores = [h['mejor'] for h in r.historial]
    registrar(all(b <= a + 1e-12 for a, b in zip(mejores, mejores[1:])),
              "con elitismo el mejor error nunca empeora de una generación a otra")
    registrar(r.arbol.altura() <= 5 and not bib_pol.verificar(r.arbol.raiz),
              "la mejor ecuación respeta la altura máxima y la biblioteca")
    r2b = ProgramacionGenetica(d_pol, bib_pol, altura_maxima=5, generaciones=60, semilla=1,
                               objetivo=1e-9, mostrar_cada=0).ejecutar(mostrar=False)
    registrar(r2b.ecuacion == r.ecuacion and r2b.generaciones == r.generaciones,
              "la misma semilla produce exactamente la misma ejecución")
    for control, sel in (('rechazar', 'ruleta'), ('reparar', 'ruleta')):
        gp2 = ProgramacionGenetica(d_pol, bib_pol, altura_maxima=4, generaciones=8, semilla=3,
                                   control_altura=control, seleccion=sel, mostrar_cada=0)
        r3 = gp2.ejecutar(mostrar=False)
        registrar(r3.arbol.altura() <= 4 and math.isfinite(r3.aptitud),
                  f"control de altura '{control}' con selección '{sel}': altura {r3.arbol.altura()} ≤ 4")
    for nombre, f in [
            ("población de 1", lambda: ProgramacionGenetica(d_pol, bib_pol, tam_poblacion=1)),
            ("probabilidad 1.5", lambda: ProgramacionGenetica(d_pol, bib_pol, prob_mutacion=1.5)),
            ("selección desconocida", lambda: ProgramacionGenetica(d_pol, bib_pol, seleccion='azar')),
            ("élite = población", lambda: ProgramacionGenetica(d_pol, bib_pol, tam_poblacion=10, elitismo=10)),
            ("peso negativo", lambda: ProgramacionGenetica(d_pol, bib_pol, pesos_mutacion={'hoja': -1})),
            ("mutación desconocida", lambda: ProgramacionGenetica(d_pol, bib_pol, pesos_mutacion={'otra': 1})),
            ("todos los pesos en 0", lambda: ProgramacionGenetica(d_pol, bib_pol, pesos_mutacion={'hoja': 0})),
            ("control de altura inválido", lambda: ProgramacionGenetica(d_pol, bib_pol, control_altura='x')),
            ("métrica inválida", lambda: ProgramacionGenetica(d_pol, bib_pol, metrica='R2')),
            ("altura 9", lambda: ProgramacionGenetica(d_pol, bib_pol, altura_maxima=9)),
            ("variable de los datos fuera de la biblioteca",
             lambda: ProgramacionGenetica(d_pol, Biblioteca(variables=['t']))),
            ("datos que no son ConjuntoDatos", lambda: ProgramacionGenetica([1, 2, 3], bib_pol))]:
        ok, msg = falla_con_error(f)
        registrar(ok, f"{nombre}: {msg}")

    print("\n13) Algoritmo evolutivo de constantes (números reales)")
    aj = AjusteConstantes("1.5*x^2 + 4*x - 2", d_pol, tam_poblacion=50, generaciones=200, rango=0.5,
                          semilla=2, objetivo=1e-5, mostrar_cada=0)
    r = aj.ejecutar(mostrar=False)
    registrar(r.metricas['RMSE'] < 1e-3 and all(abs(a - b) < 1e-2 for a, b in zip(r.constantes, [2, 5, 3])),
              f"de 1.5x² + 4x − 2 recupera las constantes: {r.constantes} → ŷ = {r.ecuacion}")
    registrar(aj.posiciones and len(aj.posiciones) == 3, "no modifica el exponente de x^2 (3 constantes, no 4)")
    ok, msg = falla_con_error(lambda: AjusteConstantes("x*x", d_pol))
    registrar(ok, f"ecuación sin constantes: {msg}")
    ok, msg = falla_con_error(lambda: AjusteConstantes("2*x", d_pol, rango=0))
    registrar(ok, f"rango 0: {msg}")

    print(f"\nResultado: {aciertos}/{total} pruebas correctas.")
    return aciertos, total

## Paso 22 · Módulo `__init__.py`

Define qué se importa con `import arbol_expresion`.

In [ ]:
%%writefile arbol_expresion/__init__.py
"""
arbol_expresion — Paquete para construir, recorrer, evaluar, graficar y generar árboles de expresión,
y para medir qué tan bien una ecuación reproduce un conjunto de datos.

Módulos:
  config       Funciones, constantes, operadores, colores y límites
  errores      Excepción ErrorExpresion y mensajes con señalamiento
  lexico       Normalización y tokenización
  sintaxis     Menos unario, multiplicación implícita y validación
  postfija     Algoritmo Shunting-Yard (infija -> postfija)
  arbol        Nodo, construcción, recorridos y estadísticas
  evaluador    Evaluación numérica (un punto o miles a la vez) con validación de dominio
  graficos     Dibujo del árbol y gráfica de la función
  analizador   Clase ArbolExpresion y función analizar()
  generador    Biblioteca, valores válidos, altura máxima y generación aleatoria de ecuaciones
  datos        Datos (x, y) de una ecuación, CSV y gráfica
  metricas     MSE, RMSE, NMSE y R²
  ajuste       Evaluar, comparar y buscar ecuaciones que se ajusten a los datos
  evolutivo    Algoritmo evolutivo genérico: generar, evaluar, mientras (selección, cruza, mutación)
  operadores_gp       Cruza monopunto, 4 mutaciones (hojas, internos, switching, rama nueva) y reparación
  programacion_genetica  Programación genética: el individuo es un árbol de expresión
  ajuste_constantes   Algoritmo evolutivo de números reales para ajustar coeficientes
  demostraciones      Cada operador genético paso a paso, con dibujo antes → después
  interactivo  Modo interactivo con input()
  pruebas      Pruebas automáticas
"""
from .errores import ErrorExpresion
from .analizador import ArbolExpresion, analizar, construir
from .generador import Biblioteca, GeneradorExpresiones
from .datos import ConjuntoDatos, generar_datos, guardar_csv, cargar_csv, graficar_datos, descargar
from .metricas import mse, rmse, nmse, r2, calcular_metricas, imprimir_metricas
from .ajuste import predecir, evaluar_modelo, comparar_modelos, busqueda_aleatoria, graficar_ajuste
from .evolutivo import AlgoritmoEvolutivo, ResultadoEvolutivo
from .programacion_genetica import ProgramacionGenetica
from .ajuste_constantes import AjusteConstantes
from .demostraciones import numerar_nodos, demostrar_mutacion, demostrar_cruza, demostrar_reparacion
from .graficos import dibujar_comparacion
from .interactivo import modo_interactivo
from .pruebas import ejecutar_pruebas

__all__ = [
    'ErrorExpresion', 'ArbolExpresion', 'analizar', 'construir',
    'Biblioteca', 'GeneradorExpresiones',
    'ConjuntoDatos', 'generar_datos', 'guardar_csv', 'cargar_csv', 'graficar_datos', 'descargar',
    'mse', 'rmse', 'nmse', 'r2', 'calcular_metricas', 'imprimir_metricas',
    'predecir', 'evaluar_modelo', 'comparar_modelos', 'busqueda_aleatoria', 'graficar_ajuste',
    'AlgoritmoEvolutivo', 'ResultadoEvolutivo', 'ProgramacionGenetica', 'AjusteConstantes',
    'numerar_nodos', 'demostrar_mutacion', 'demostrar_cruza', 'demostrar_reparacion', 'dibujar_comparacion',
    'modo_interactivo', 'ejecutar_pruebas',
]
__version__ = '5.0'

## Paso 23 · Importar el paquete

Esta celda borra cualquier versión anterior del paquete que esté en memoria y lo vuelve a importar.
Así se evitan errores al volver a ejecutar celdas.

**Si modificas un módulo:** vuelve a ejecutar su celda `%%writefile`, luego esta celda y después las celdas que quieras usar.

In [ ]:
import sys
for nombre in [m for m in sys.modules if m == 'arbol_expresion' or m.startswith('arbol_expresion.')]:
    del sys.modules[nombre]

import numpy as np
import arbol_expresion as ae
from arbol_expresion import (ArbolExpresion, ErrorExpresion, analizar,
                             Biblioteca, GeneradorExpresiones,
                             ProgramacionGenetica, AjusteConstantes)
from arbol_expresion import operadores_gp as op

print(f"✅ Paquete arbol_expresion v{ae.__version__} importado.")
print("   Módulos:", ', '.join(sorted(f[:-3] for f in os.listdir('arbol_expresion') if f.endswith('.py'))))

# 📈 Parte 2 · Actividad: ecuación → árbol → gráfica → CSV → error

$$y = 6.87 + 11\cdot\cos\left(7.23\cdot x^{3}\right), \qquad x \in [-10.0,\ 10.0], \qquad \text{paso} = 0.001$$

## Paso 24 · Configuración (👉 la única celda que cambias mañana)

Escribe la ecuación con `x` como variable: `*` para multiplicar, `^` (o `**`) para potencias, y funciones como `cos(...)`, `sin(...)`, `sqrt(...)`, `exp(...)`, `ln(...)`.

In [ ]:
ECUACION    = "6.87 + 11*cos(7.23*x^3)"   # y = 6.87 + 11·cos(7.23·x³)
X_MIN       = -10.0                        # inicio del intervalo
X_MAX       = 10.0                         # fin del intervalo
PASO        = 0.001                        # separación entre valores de x
ARCHIVO_CSV = "datos_ecuacion.csv"         # nombre del archivo que se genera
ZOOM        = (-1.5, 1.5)                  # intervalo del acercamiento en las gráficas

try:
    ecuacion = ArbolExpresion(ECUACION)
    print(f"✅ Ecuación válida: y = {ecuacion.infija_minima()}")
    print(f"   Intervalo [{X_MIN}, {X_MAX}] con paso {PASO} → {round((X_MAX - X_MIN) / PASO) + 1:,} puntos")
except ErrorExpresion as err:
    print(f"❌ Revisa la ecuación:\n   {err}")

## Paso 25 · Árbol de expresión de la ecuación

Tabla de Shunting-Yard, recorridos preorden, inorden y postorden, dibujo del árbol y evaluación en x = 1.

In [ ]:
arbol_ecuacion = analizar(ECUACION, valores={'x': 1}, graficar_f=False,
                          guardar_como="arbol_ecuacion.png")

## Paso 26 · Generar los datos y el CSV

Se evalúa la ecuación en cada x desde `X_MIN` hasta `X_MAX` con el paso indicado.
El CSV tiene encabezado `x,y`; x se escribe con los decimales del paso y y con 6 decimales.

> Los números del ejemplo del pizarrón (2.71, 7.27, 5.73) solo ilustran el formato. Con esta ecuación, en x = 0 se obtiene y = 6.87 + 11·cos(0) = **17.87**.

In [ ]:
datos = ae.generar_datos(ECUACION, inicio=X_MIN, fin=X_MAX, paso=PASO)
datos.describir()
print()
ae.guardar_csv(datos, ARCHIVO_CSV)

### Descargar el CSV

En Colab, esta celda descarga el archivo a tu computadora. También lo encuentras en el panel 📁 de la izquierda.

In [ ]:
ae.descargar(ARCHIVO_CSV)

## Paso 27 · Graficar la ecuación

**Arriba:** el intervalo completo con los 20,001 puntos del CSV. **Abajo:** un acercamiento.

**¿Por qué la gráfica completa se ve como una banda sólida?** Dentro del coseno está x³, así que la función oscila cada vez más rápido al alejarse de 0. Cerca de x = ±10 completa una oscilación cada ~0.003 unidades, es decir, solo hay unas 3 muestras por oscilación con paso 0.001. Los datos son correctos; simplemente las oscilaciones son demasiado juntas para distinguirlas a esa escala. En el acercamiento, cerca de x = 0, se ve la forma real de la curva.

In [ ]:
ae.graficar_datos(datos, zoom=ZOOM, guardar_como="grafica_ecuacion.png")

## Paso 28 · Error de la ecuación contra los datos (MSE, RMSE, NMSE, R²)

**Comprobación:** al comparar la ecuación con los datos que ella misma generó, el error debe ser exactamente 0 y R² = 1.

In [ ]:
metricas_exacta = ae.evaluar_modelo(ECUACION, datos)

### Comparar varias ecuaciones contra los mismos datos

Así se mide qué tan lejos queda una ecuación aproximada. Aquí se prueba con coeficientes ligeramente distintos y con modelos simples.
La tabla se ordena de mejor a peor según el RMSE.

In [ ]:
y_media = float(np.mean(datos.y))
candidatas = [
    ECUACION,
    "6.87 + 11*cos(7.2*x^3)",       # 7.2 en lugar de 7.23
    "6.87 + 10*cos(7.23*x^3)",      # 10 en lugar de 11
    "7 + 11*cos(7.23*x^3)",         # 7 en lugar de 6.87
    f"{y_media:.6f}",               # constante = ȳ  → R² ≈ 0
    "6.87",                         # solo la constante
]
resultados = ae.comparar_modelos(candidatas, datos)

**¿Por qué `7.2` en lugar de `7.23` da el peor resultado, incluso peor que una constante?**
Dentro del coseno el coeficiente multiplica a $x^3$, que llega a 1000. Una diferencia de 0.03 desfasa el coseno hasta $0.03 \cdot 1000 = 30$ radianes en los extremos (casi 5 vueltas completas). La curva aproximada queda "fuera de fase": sube cuando los datos bajan, y por eso $R^2 < 0$.
En cambio, un error en la constante (`7` en lugar de `6.87`) solo desplaza la curva hacia arriba 0.13, y el RMSE es exactamente 0.13.
**Conclusión:** en esta ecuación, el coeficiente de $x^3$ es el que más influye en el error.

### Gráfica: datos contra una ecuación aproximada

En el acercamiento se ve cómo la ecuación aproximada (línea punteada) se separa de los datos.

In [ ]:
ae.graficar_ajuste(datos, "6.87 + 10*cos(7.23*x^3)", zoom=ZOOM)
ae.evaluar_modelo("6.87 + 10*cos(7.23*x^3)", datos)

## Paso 29 · (Opcional) Cargar un CSV, por ejemplo el que entregue el doctor

Con `SUBIR_CSV = True`, Colab te pedirá elegir un archivo de tu computadora. Con `False` se carga el CSV que se generó arriba.
El archivo puede tener encabezado o no, y estar separado por coma, `;` o tabulador.

In [ ]:
SUBIR_CSV = False

archivo = ARCHIVO_CSV
if SUBIR_CSV:
    try:
        from google.colab import files
        subidos = files.upload()
        if subidos:
            archivo = next(iter(subidos))
    except ImportError:
        print("ℹ️ La subida de archivos solo funciona en Google Colab; se usa el CSV generado.")

try:
    datos_csv = ae.cargar_csv(archivo)
    datos_csv.describir()
    print()
    # Error de la ecuación de la configuración contra el CSV cargado
    ae.evaluar_modelo(ECUACION, datos_csv)
except ErrorExpresion as err:
    print(f"❌ {err}")

> Si cargas el CSV que se generó arriba, el error no es exactamente 0: y se guardó con 6 decimales, así que hay diferencias de redondeo de ~10⁻⁷.

## Paso 30 · Buscar ecuaciones con el generador y medir su error

Se juntan las dos partes del proyecto: el **generador** crea ecuaciones al azar con una **biblioteca** y una **altura máxima**, y las **métricas** dicen cuál se parece más a los datos.
Es la idea básica de la *regresión simbólica*. La programación genética la mejora haciendo "evolucionar" las mejores ecuaciones en lugar de generarlas al azar.

La ecuación original tiene altura 6 y se puede formar con esta biblioteca. Aun así, encontrarla por azar es muy poco probable; la tabla muestra las mejores de los candidatos generados.

In [ ]:
biblioteca_actividad = Biblioteca(
    operadores=['+', '*', '^'],
    funciones=['cos'],
    variables=['x'],
    valores=[6.87, 11, 7.23, 3],
)
generador_actividad = GeneradorExpresiones(biblioteca_actividad, altura_maxima=6, altura_minima=2,
                                           metodo='mixto', semilla=7)

print(f"La ecuación original cabe en la biblioteca: {not biblioteca_actividad.verificar(ecuacion.raiz)}"
      f" | altura de su árbol: {ecuacion.altura()}\n")
mejores = ae.busqueda_aleatoria(generador_actividad, datos, candidatos=500, top=5)

In [ ]:
mejor = mejores[0]['arbol']
ae.graficar_ajuste(datos, mejor, zoom=ZOOM)
mejor.dibujar()

# 🌳 Parte 3 · Árbol de expresión de la primera ecuación y generador

## Paso 31 · Reporte completo de sqrt(2x² + 5x − 3)

$\sqrt{2x^2 + 5x - 3}$ evaluada en $x = 2$: $\sqrt{8 + 10 - 3} = \sqrt{15} \approx 3.873$

In [ ]:
arbol = analizar("sqrt(2*(x^2)+5*x-3)", valores={'x': 2}, rango=(-8, 6), guardar_como="arbol_expresion.png")

## Paso 32 · Uso con la clase `ArbolExpresion`

In [ ]:
try:
    a = ArbolExpresion("sqrt(2*(x^2)+5*x-3)")
    print("Preorden :", ' '.join(a.preorden()))
    print("Inorden  :", ' '.join(a.inorden()))
    print("Postorden:", ' '.join(a.postorden()))
    print("Infija   :", a.infija_minima())
    print("Estadísticas:", a.estadisticas())
    for x in [0.5, 1, 2, 5]:
        print(f"f({x}) = {a.evaluar(x=x):.6f}")
    a.evaluar(x=0)          # error de dominio a propósito
except ErrorExpresion as err:
    print("⚠️", err)

## Paso 33 · Biblioteca, valores válidos y altura máxima

| Parámetro | Para qué sirve |
|---|---|
| `operadores` | `'+', '-', '*', '/', '^', 'neg'` |
| `funciones` | `'sqrt', 'sin', 'cos', 'tan', 'log', 'ln', 'exp', 'abs'` |
| `variables` | letras, por ejemplo `['x']` |
| `valores` | **lista de valores válidos** para las hojas (≥ 0) |
| `altura_maxima` / `altura_minima` | límites del tamaño del árbol (1 a 8) |
| `metodo` | `'grow'`, `'full'` o `'mixto'` |
| `semilla` | número fijo para repetir las mismas ecuaciones; `None` para que cambien |
| `puntos_validos` | la ecuación debe poder evaluarse en esos puntos |

In [ ]:
biblioteca = Biblioteca(operadores=['+', '-', '*', '/', '^'], funciones=['sqrt'],
                        variables=['x'], valores=[1, 2, 3, 5])
generador = GeneradorExpresiones(biblioteca, altura_maxima=4, altura_minima=3, metodo='mixto',
                                 semilla=2026, puntos_validos={'x': [1, 2, 3]})
generador.describir()

## Paso 34 · Generar una ecuación y analizarla

Ejecuta la celda varias veces para obtener ecuaciones distintas.

In [ ]:
ecuacion_generada = generador.analizar_nueva(valores={'x': 2}, mostrar_pasos=False, rango=(0, 6))

## Paso 35 · Generar un lote de ecuaciones

In [ ]:
lote = generador.generar_lote(10, evaluar_en={'x': [1, 2, 3]})

## Paso 36 · La biblioteca y la altura máxima también validan tus ecuaciones

In [ ]:
analizar("sqrt(2*(x^2)+5*x-3)", altura_maxima=4, graficar=False)
analizar("sqrt(2*(x^2)+5*x-3)", biblioteca=Biblioteca(operadores=['+', '*'], valores=[2, 3, 5]),
         graficar=False)
for prueba in [lambda: Biblioteca(valores=[-2]),
               lambda: Biblioteca(operadores=['%']),
               lambda: GeneradorExpresiones(biblioteca, altura_maxima=20),
               lambda: ae.generar_datos("x^2", 0, 1, 0.3),
               lambda: ae.generar_datos("x^2", 5, -5, 0.1)]:
    try:
        prueba()
    except ErrorExpresion as err:
        print("⚠️", err)

# 🧬 Parte 4 · Programación genética y algoritmo evolutivo

**Algoritmo (cuaderno, 20/10/2026):**

```
1. GENERAR la población → 100 ecuaciones aleatorias
2. EVALUAR             → error de cada ecuación contra los datos (RMSE)
3. MIENTRAS (no se cumpla el criterio de paro):
      → SELECCIÓN
      → CRUZA      (monopunto: intercambio de subárboles)
      → MUTACIÓN   (4 tipos, se elige UNO al azar):
            1. mutación de hojas
            2. mutación de nodos internos
            3. switching
            4. generación de una rama nueva
      → REPARACIÓN (si se excede el límite de altura)
```

Primero se demuestra **cada operador por separado** con los árboles del cuaderno.
En los dibujos, el **borde amarillo** marca lo que cambió y la **línea roja** marca el límite de altura.
Después se ejecuta el algoritmo completo sobre los datos de la Parte 2.

## Paso 37 · Biblioteca para las demostraciones

Valores válidos del apunte: 2, 3, 5, 7.2 y π; variable x.
`numerar_nodos()` muestra el número de cada nodo en **preorden**; sirve para elegir dónde aplicar cada operador.

In [ ]:
biblioteca_demo = Biblioteca(operadores=['+', '-', '*', '/'], funciones=['sqrt'],
                             variables=['x'], valores=[2, 3, 5, 7.2], constantes=['pi'])
biblioteca_demo.describir()
print()
ae.numerar_nodos("5/x - 3")

## Paso 38 · Mutación 1 · Mutación de hojas

*"Simplemente: aleatoriamente seleccionar un nodo hoja (y cambiarlo) por un valor válido."*
Ejemplo del cuaderno: en el árbol `5/x − 3`, la hoja **3** puede cambiar por **7.2**, **π** o **x**. Se dibuja un resultado por cada valor.

In [ ]:
ae.demostrar_mutacion("5/x - 3", 'hoja', biblioteca_demo, altura_maxima=4,
                      nodo=4, valores=['7.2', 'pi', 'x'])

## Paso 39 · Mutación 2 · Mutación de nodos internos

*"Operadores: sustituir el valor de un nodo interno por otro operador válido."*
El operador nuevo debe tener el **mismo número de hijos**: un binario (`+ − * /`) cambia por otro binario y una función (`sqrt`, `cos`…) por otra función. Así el árbol sigue siendo válido.

In [ ]:
ae.demostrar_mutacion("5/x - 3", 'interno', biblioteca_demo, altura_maxima=4,
                      nodo=1, valores=['*', '+'])

## Paso 40 · Mutación 3 · Switching (intercambio izquierda ↔ derecha)

*"Intercambiar valores de los lados (con los hijos del nodo): izquierda → derecha, izquierda ← derecha."*
Se intercambian los subárboles izquierdo y derecho de un nodo, con todo lo que cuelga de ellos.

> En el cuaderno el ejemplo usa `sqrt` con dos hijos. En este programa `sqrt` es una función de **un** argumento (un solo hijo), así que el switching se aplica a un **operador binario**; aquí, la resta. Al elegir al azar se prefieren `−` y `/`: en `+` y `*` el intercambio no cambia el valor (a + b = b + a).

In [ ]:
ae.demostrar_mutacion("x/5 - 2*x", 'switching', biblioteca_demo, altura_maxima=4, nodo=0)

## Paso 41 · Mutación 4 · Generación de una rama nueva

*"Generación de una rama nueva: agarro y genero algo nuevo cuidando la altura del árbol."*
Un subárbol se reemplaza por una rama aleatoria. Si el nodo está en el nivel *p* y la altura máxima es *H*, la rama nueva puede medir como máximo **H − p + 1**; así el árbol nunca rebasa el límite.
Como en el cuaderno: en `5 − 2 + 3/x` se elimina la **x** y en su lugar crece una rama nueva.

In [ ]:
ae.demostrar_mutacion("5 - 2 + 3/x", 'rama', biblioteca_demo, altura_maxima=5, nodo=6, semilla=7)

### "No mezclar los métodos de mutación"

En cada mutación se **sortea un solo tipo**. Si se aplicaran varios a la vez, la ecuación cambiaría demasiado y se perdería su sentido.
Esta celda aplica 8 mutaciones al azar al mismo árbol; cada renglón usa exactamente un tipo.

In [ ]:
import random
from arbol_expresion.arbol import infija_minima

generador_demo = GeneradorExpresiones(biblioteca_demo, altura_maxima=4, semilla=1, exigir_variable=False)
rng_demo = random.Random(2026)
original = ArbolExpresion("5/x - 3").raiz
print(f"Original: {infija_minima(original)}\n")
for i in range(1, 9):
    mutado, info = op.mutar_aleatorio(original, generador_demo, rng_demo, altura_maxima=4)
    print(f"{i}. {infija_minima(mutado):<24} ← {info['descripcion']}")
print(f"\nEl original no cambió: {infija_minima(original)}")

## Paso 42 · Cruza monopunto

*"Tengo 2 árboles y rompo cada árbol: utilizo monopunto. Agarras cualquier parte (de cualquier altura) del árbol, aleatoriamente, e intercambias las conexiones."*
Se elige un punto de cruza en cada padre y se intercambian los subárboles. Cada hijo conserva parte de cada padre.

In [ ]:
ae.numerar_nodos("sqrt(x/5) + 2*x")
print()
hijo1, hijo2 = ae.demostrar_cruza("5/x - 3", "sqrt(x/5) + 2*x", punto1=4, punto2=5)

## Paso 43 · Método de reparación

*"Método de reparación: (se aplica cuando el árbol) excede el límite del árbol que hemos establecido."*
La cruza puede producir un hijo **más alto** que la altura máxima. Aquí, con altura máxima 4, la `x` de `5 − 2 + 3/x` recibe el subárbol `pi*sqrt(5*x)` y el hijo queda de altura 6.

- **Reparar (poda)**, la opción por defecto: todo nodo del último nivel permitido que todavía tenga hijos se reemplaza por una **hoja válida**, como el π del apunte.
- **Rechazar**, con `control_altura='rechazar'`: el hijo no vale y se conserva el padre.

In [ ]:
hijo_alto, _ = ae.demostrar_cruza("5 - 2 + 3/x", "pi*sqrt(5*x)", punto1=6, punto2=0)
print()
reparado = ae.demostrar_reparacion(hijo_alto, biblioteca_demo, altura_maxima=4, semilla=1)

## Paso 44 · Configuración de la programación genética (👉 celda para cambiar)

Se usan los **datos de la Parte 2** (`datos`, 20,001 puntos de la ecuación de la actividad). Si cargaste el CSV del doctor, cambia `DATOS_GP = datos_csv`.

| Parámetro | Valor | Para qué sirve |
|---|---|---|
| `tam_poblacion` | 200 | ecuaciones por generación (en clase se vieron 100; con 200 se encuentra la solución con más frecuencia) |
| `generaciones` | 100 | máximo de generaciones del *mientras* |
| `altura_maxima` | 6 | límite del árbol (la ecuación de la actividad mide 6) |
| `seleccion` / `tam_torneo` | torneo / 3 | gana el mejor de 3 ecuaciones elegidas al azar |
| `prob_cruza` / `prob_mutacion` | 0.9 / 0.3 | probabilidad de cruzar y de mutar |
| `pesos_mutacion` | 1, 1, 1, 1 | probabilidad relativa de cada tipo de mutación |
| `elitismo` | 2 | las 2 mejores ecuaciones pasan sin cambios |
| `control_altura` | `'reparar'` | método de reparación, o `'rechazar'` |
| `objetivo` | 1e-9 | se detiene si el RMSE llega a ~0 (ecuación exacta) |
| `max_puntos` | 2000 | evalúa con 2,000 puntos repartidos (10 veces más rápido); las métricas finales usan los 20,001 |
| `semilla` | 3 | misma semilla = mismos resultados; `None` = distintos en cada ejecución |

La **biblioteca** contiene los operadores y valores que aparecen en la ecuación del doctor. Cuando lleguen ecuaciones nuevas, pon aquí sus operadores, funciones y valores.

In [ ]:
DATOS_GP = datos

biblioteca_gp = Biblioteca(
    operadores=['+', '*', '^'],
    funciones=['cos'],
    variables=['x'],
    valores=[6.87, 11, 7.23, 3],
)

pg = ProgramacionGenetica(
    DATOS_GP, biblioteca_gp,
    tam_poblacion=200,
    generaciones=100,
    altura_maxima=6,
    seleccion='torneo', tam_torneo=3,
    prob_cruza=0.9, prob_mutacion=0.3,
    pesos_mutacion={'hoja': 1, 'interno': 1, 'switching': 1, 'rama': 1},
    elitismo=2,
    control_altura='reparar',
    metrica='RMSE',
    objetivo=1e-9,
    max_puntos=2000,
    semilla=3,
    mostrar_cada=5,
)

## Paso 45 · Ejecutar la programación genética

Cada renglón de la tabla es una generación: el error del **mejor** individuo, la **mediana** del error de la población (se usa la mediana porque unos pocos individuos con errores enormes, como 10⁶⁰, harían inútil el promedio), cuántos individuos son **inválidos** (no definidos en algún punto), cuántas ecuaciones **distintas** hay y la mejor ecuación.

In [ ]:
resultado_pg = pg.ejecutar()

## Paso 46 · Convergencia, árbol y ajuste de la mejor ecuación

In [ ]:
resultado_pg.graficar_convergencia(guardar_como="convergencia_pg.png")
resultado_pg.arbol.dibujar(guardar_como="arbol_mejor_pg.png")
ae.graficar_ajuste(DATOS_GP, resultado_pg.arbol, zoom=ZOOM)

## Paso 47 · ¿Siempre encuentra la ecuación? (5 semillas)

La programación genética es **aleatoria**: con otra semilla recorre otro camino. Esta celda la ejecuta con 5 semillas, sin imprimir cada generación, y resume el resultado (≈ 15 s).
En las pruebas con 30 semillas encontró la ecuación exacta en **23 (77 %)**.

In [ ]:
print(f"{'Semilla':>7} │ {'Generaciones':>12} │ {'RMSE':>10} │ {'R²':>9} │ Ecuación")
print('─' * 100)
for s in range(1, 6):
    r = ProgramacionGenetica(DATOS_GP, biblioteca_gp, tam_poblacion=200, generaciones=100, altura_maxima=6,
                             objetivo=1e-9, max_puntos=2000, semilla=s, mostrar_cada=0).ejecutar(mostrar=False)
    rmse = r.metricas['RMSE']
    print(f"{s:>7} │ {r.generaciones:>12} │ {rmse if rmse > 1e-12 else 0:>10.4g} │ "
          f"{r.metricas['R2']:>9.6f} │ {r.ecuacion[:60]}")

Cuando no encuentra la ecuación exacta, normalmente se queda **cerca**: ya descubrió el término `cos(7.23*x^3)`, pero los coeficientes de afuera no son los correctos. Para ese caso sirve el algoritmo evolutivo de constantes.

## Paso 48 · Algoritmo evolutivo: ajuste de constantes

La programación genética solo puede usar los **valores de la biblioteca**. El algoritmo evolutivo de números reales toma una ecuación con forma fija y **ajusta sus constantes**:

| Paso | Programación genética | Algoritmo evolutivo de constantes |
|---|---|---|
| Individuo | árbol de expresión | vector de reales, por ejemplo [7, 10, 7.2] |
| Generar | ecuaciones aleatorias | vectores alrededor de los valores iniciales (±10 %) |
| Evaluar | RMSE contra los datos | RMSE contra los datos |
| Selección | torneo | torneo |
| Cruza | monopunto (subárboles) | BLX-α (valor entre los de los padres) |
| Mutación | hojas, internos, switching, rama | gaussiana (ruido normal pequeño) |

**Ejemplo 1:** partir de constantes aproximadas, `7 + 10*cos(7.2*x^3)`, y recuperar 6.87, 11 y 7.23.
Recuerda que en la Parte 2 cambiar 7.23 por 7.2 daba R² < 0: este algoritmo corrige ese error.

In [ ]:
ajuste = AjusteConstantes("7 + 10*cos(7.2*x^3)", DATOS_GP, tam_poblacion=60, generaciones=150,
                          rango=0.1, objetivo=1e-6, max_puntos=4000, semilla=1, mostrar_cada=15)
resultado_ajuste = ajuste.ejecutar()
resultado_ajuste.graficar_convergencia()

**Ejemplo 2 · Programación genética + algoritmo evolutivo (flujo completo).** Con la semilla 9, la programación genética se queda en R² ≈ 0.91: encontró la forma correcta, `cos(7.23·x³)·(algo) + 6.87`, pero el coeficiente de afuera da ≈ 14.4 en lugar de 11. Después, el algoritmo evolutivo ajusta las constantes de esa ecuación.

In [ ]:
pg_cerca = ProgramacionGenetica(DATOS_GP, biblioteca_gp, tam_poblacion=200, generaciones=100, altura_maxima=6,
                                objetivo=1e-9, max_puntos=2000, semilla=9, mostrar_cada=0)
resultado_cerca = pg_cerca.ejecutar(mostrar=False)
print(f"Programación genética:  ŷ = {resultado_cerca.ecuacion}")
print(f"   RMSE = {resultado_cerca.metricas['RMSE']:.6g} | R² = {resultado_cerca.metricas['R2']:.6f}\n")

if resultado_cerca.metricas['RMSE'] > 1e-6:
    resultado_final = AjusteConstantes(resultado_cerca.arbol, DATOS_GP, tam_poblacion=60, generaciones=150,
                                       rango=0.1, objetivo=1e-6, max_puntos=4000, semilla=1,
                                       mostrar_cada=30).ejecutar()
    ae.graficar_ajuste(DATOS_GP, resultado_final.arbol, zoom=ZOOM)
else:
    print("✅ La programación genética ya encontró la ecuación exacta; no hace falta ajustar.")

# ✅ Parte 5 · Pruebas y modo interactivo

## Paso 49 · Pruebas automáticas

In [ ]:
ae.ejecutar_pruebas()

## Paso 50 · Modo interactivo (opcional)

Escribe expresiones; si tienen variables se piden sus valores. Escribe **`salir`** para terminar.

In [ ]:
ae.modo_interactivo()